# SMGF-Net + MS-CA-ECA — Brain Tumor MRI Classification (Full Research Notebook)

**Final proposed architecture:** `SMGF-Net + MS-CA-ECA` (Spatial branch + Morphology branch + Gated Fusion, unchanged from SMGF-Net, with plain ECA replaced by **MS-CA-ECA** — Multi-Scale Coordinate-ECA Attention).

This notebook is restructured around that single architecture. NeuroM3F-Net (the old predecessor model), the old NeuroM3F-Net component ablation (§21, A→I), and the old preprocessing ablation on NeuroM3F-Net (§21b) have been **removed entirely**. The Spatial branch, Morphology branch, and Gated Fusion mechanism are reused **verbatim** from the SMGF-Net notebook; they are not redesigned here.

**Story:** Data cleaning → SMGF-Net building blocks → MS-CA-ECA → component ablation → training/validation analysis → internal test evaluation → external validation → XAI.

**Research question:** *Does the proposed lightweight MS-CA-ECA mechanism improve the SMGF-Net architecture, and are its individual components (multi-scale context, coordinate attention, ECA channel refinement) responsible for the observed performance improvement?*

Every numeric result in this notebook (accuracy, AUC, F1, ablation deltas, etc.) comes from cells that must actually be executed top-to-bottom — nothing is pre-filled. The one exception is the parameter-count numbers quoted in the markdown below, which were verified by actually instantiating each model configuration in isolation (see §8) before writing this notebook.

## §1 · Dependencies

In [ ]:
!pip install -q -U timm scikit-learn thop scikit-image

## §2 · Config

In [ ]:
import os, math, random, warnings, hashlib
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                              precision_score, recall_score, roc_auc_score,
                              confusion_matrix, classification_report,
                              matthews_corrcoef, cohen_kappa_score)
import matplotlib.pyplot as plt
import timm
import pandas as pd
from scipy.stats import spearmanr
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE    = 224
NUM_CLASSES = 4
CLASS_NAMES  = ["glioma", "meningioma", "notumor", "pituitary"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".jfif", ".tif", ".tiff", ".webp"}

OUT_DIR  = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE  = 32
NUM_WORKERS = 2

DROPOUT_RATE  = 0.3
USE_AMP       = True
EARLY_STOP_PATIENCE = 10   # epochs with no val AUC improvement before stopping

print("Device:", DEVICE)
print("Classes:", CLASS_NAMES)
print(f"DROPOUT_RATE={DROPOUT_RATE}  USE_AMP={USE_AMP}")

## §3 · Preprocessing (+ AMEF — Adaptive MRI Enhancement Framework)

Reused verbatim from the SMGF-Net notebook — not touched by this redesign.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def read_gray(path: str) -> np.ndarray:
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError(f"Cannot read: {path}")
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY if img.shape[-1] == 3
                           else cv2.COLOR_BGRA2GRAY)
    return img

def robust_normalize(img: np.ndarray) -> np.ndarray:
    img = img.astype(np.float32)
    p1, p99 = np.percentile(img, [1, 99])
    img = np.clip(img, p1, p99)
    img -= img.min(); img /= (img.max() + 1e-8)
    return img

def brain_mask(img01: np.ndarray) -> np.ndarray:
    """Otsu threshold + morphological cleanup + largest-connected-component
    to isolate brain tissue from black background/border."""
    img_u8 = (np.clip(img01, 0, 1) * 255).astype(np.uint8)
    _, mask = cv2.threshold(img_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    num, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num <= 1:
        return np.ones_like(img01, dtype=bool)
    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    brain = (labels == largest)
    if brain.sum() < 0.02 * img01.size:
        return np.ones_like(img01, dtype=bool)
    return brain


def laplacian_noise_estimate(gray_u8: np.ndarray) -> float:
    """Fast noise-sigma estimator (Immerkaer 1996) on an 8-bit grayscale image."""
    H, W = gray_u8.shape
    if H < 3 or W < 3:
        return 0.0
    kernel = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], dtype=np.float32)
    conv = cv2.filter2D(gray_u8.astype(np.float32), -1, kernel)
    sigma = np.sqrt(np.pi / 2) * np.mean(np.abs(conv)) / (6 * (W - 2) * (H - 2) + 1e-8) * (W * H)
    return float(np.clip(sigma, 0.0, 50.0))


def adaptive_gamma(img01: np.ndarray, mask: np.ndarray, target: float = 0.5,
                    max_shift: float = 0.15, deadzone: float = 0.08) -> float:
    """REDESIGNED: gamma that nudges masked-brain mean intensity toward `target`,
    but only when it's meaningfully off (|mu - target| > deadzone) and only
    partway there (capped at max_shift). The original version forced every
    image's mean all the way to exactly 0.5 regardless of starting point,
    which erases inter-image/inter-class intensity variance that
    robust_normalize already preserved -- variance that can carry real
    diagnostic signal (tumor subtypes differ in characteristic T1 intensity).
    This version corrects only genuinely under/over-exposed images and leaves
    normally-exposed ones alone."""
    vals = img01[mask]
    if vals.size == 0:
        return 1.0
    mu = float(np.clip(vals.mean(), 1e-3, 1 - 1e-3))
    if abs(mu - target) <= deadzone:
        return 1.0  # already reasonable -- don't touch it
    bounded_target = mu + np.clip(target - mu, -max_shift, max_shift)
    gamma = np.log(bounded_target) / np.log(mu)
    return float(np.clip(gamma, 0.7, 1.4))  # tighter bound than original [0.5, 2.0]


def masked_entropy(img01: np.ndarray, mask: np.ndarray, bins: int = 32) -> float:
    vals = img01[mask]
    if vals.size == 0:
        return 0.0
    hist, _ = np.histogram(vals, bins=bins, range=(0, 1), density=True)
    hist = hist / (hist.sum() + 1e-8)
    hist = hist[hist > 0]
    return float(-np.sum(hist * np.log2(hist)))


def amef_preprocess(img01: np.ndarray, base_clip: float = 2.0, kappa: float = 0.05,
                     entropy_skip_threshold: float = 3.6) -> np.ndarray:
    """Adaptive MRI Enhancement Framework (v2): noise-gated CLAHE clip limit +
    bounded/deadzoned adaptive gamma correction + entropy-based CLAHE skip.
    Parameter-free at inference time (all thresholds fixed constants).
    Only the gamma step changed vs. v1 -- see adaptive_gamma docstring."""
    mask = brain_mask(img01)
    gray_u8 = (np.clip(img01, 0, 1) * 255).astype(np.uint8)

    sigma_noise = laplacian_noise_estimate(gray_u8)
    gamma = adaptive_gamma(img01, mask)
    img_gamma = np.clip(img01 ** gamma, 0, 1) if gamma != 1.0 else img01

    ent = masked_entropy(img_gamma, mask)
    if ent < entropy_skip_threshold:
        clip_limit = base_clip / (1.0 + kappa * sigma_noise)
        clahe = cv2.createCLAHE(clipLimit=max(clip_limit, 0.5), tileGridSize=(8, 8))
        img_u8 = (img_gamma * 255).astype(np.uint8)
        img_out = clahe.apply(img_u8).astype(np.float32) / 255.0
    else:
        img_out = img_gamma
    return img_out


def to_3ch_tensor(img01: np.ndarray) -> torch.Tensor:
    rgb = np.stack([img01, img01, img01], axis=2)
    rgb = torch.tensor(rgb).permute(2, 0, 1).float()
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (rgb - mean) / std


def preprocess(path: str) -> torch.Tensor:
    """Returns a single [3,224,224] image tensor.
    Pipeline: read_gray -> resize -> robust_normalize -> tensor.
    AMEF (adaptive gamma + entropy-gated CLAHE) is INTENTIONALLY NOT applied
    here. Two independent single-seed ablation runs (original AMEF, then a
    redesigned bounded/deadzoned adaptive_gamma) both showed AMEF losing to
    plain robust-normalize-only preprocessing on every metric (accuracy, F1,
    AUC, ECE) -- see section 21b. `amef_preprocess` and its helpers are kept
    below (unused by this function) only so that ablation cell stays
    reproducible; they are not part of the model's actual preprocessing path."""
    gray = read_gray(path)
    gray = cv2.resize(gray, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    img01 = robust_normalize(gray)
    return to_3ch_tensor(img01)


# --- Diagnostic utility (call manually, not part of the pipeline) ---
def amef_diagnostic(sample_paths, n=200):
    """Reports how often gamma correction actually fires and by how much,
    plus CLAHE trigger rate. Run before/after any AMEF change to confirm
    the fix is doing what's intended, and to sanity-check against the old
    version's behavior."""
    gammas, mu_before, mu_after, clahe_fired = [], [], [], []
    for p in sample_paths[:n]:
        gray = cv2.resize(read_gray(p), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        img01 = robust_normalize(gray)
        mask = brain_mask(img01)
        mu_before.append(float(img01[mask].mean()))
        gamma = adaptive_gamma(img01, mask)
        gammas.append(gamma)
        img_gamma = np.clip(img01 ** gamma, 0, 1) if gamma != 1.0 else img01
        mu_after.append(float(img_gamma[mask].mean()))
        clahe_fired.append(masked_entropy(img_gamma, mask) < 3.6)

    gammas = np.array(gammas)
    print(f"gamma == 1.0 (untouched): {100*np.mean(gammas == 1.0):.1f}% of images")
    print(f"gamma (when applied):     mean={gammas[gammas!=1.0].mean() if (gammas!=1.0).any() else float('nan'):.3f}")
    print(f"mean-intensity before gamma: {np.mean(mu_before):.3f} ± {np.std(mu_before):.3f}")
    print(f"mean-intensity after  gamma: {np.mean(mu_after):.3f} ± {np.std(mu_after):.3f}")
    print(f"CLAHE fired on {100*np.mean(clahe_fired):.1f}% of images")

print("Preprocessing + AMEF (v2 - bounded/deadzoned gamma) ready.")

## §4 · Augmentation (training split only)

In [ ]:
import torchvision.transforms.functional as TF

class MRIAugment:
    def __init__(self, p_flip=0.5, max_rotate=10.0, p_rotate=0.5,
                 brightness=0.2, contrast=0.2, p_jitter=0.5,
                 translate=0.06, p_translate=0.3,
                 use_gamma=True, gamma_range=(0.85, 1.15), p_gamma=0.3):
        self.p_flip = p_flip
        self.max_rotate = max_rotate; self.p_rotate = p_rotate
        self.brightness = brightness; self.contrast = contrast; self.p_jitter = p_jitter
        self.translate = translate; self.p_translate = p_translate
        self.use_gamma = use_gamma; self.gamma_range = gamma_range; self.p_gamma = p_gamma

    def __call__(self, img: torch.Tensor) -> torch.Tensor:
        if random.random() < self.p_flip:
            img = TF.hflip(img)
        if random.random() < self.p_rotate:
            angle = random.uniform(-self.max_rotate, self.max_rotate)
            img = TF.rotate(img, angle)
        if random.random() < self.p_translate:
            tx = random.uniform(-self.translate, self.translate) * IMG_SIZE
            ty = random.uniform(-self.translate, self.translate) * IMG_SIZE
            img = TF.affine(img, angle=0, translate=[int(tx), int(ty)], scale=1.0, shear=0)
        if random.random() < self.p_jitter:
            b = 1.0 + random.uniform(-self.brightness, self.brightness)
            c = 1.0 + random.uniform(-self.contrast, self.contrast)
            img = torch.clamp(img * c + (b - 1.0), -3.0, 3.0)
        if self.use_gamma and random.random() < self.p_gamma:
            gamma = random.uniform(*self.gamma_range)
            img = torch.sign(img) * (torch.abs(img) ** gamma)
        return img

train_augment = MRIAugment()
print("MRIAugment ready.")

## §5 · Dataset Class

In [ ]:
class BrainTumorDataset(Dataset):
    def __init__(self, samples: List[Dict], is_train: bool = False,
                 augmenter: Optional[MRIAugment] = None):
        self.samples   = samples
        self.is_train  = is_train
        self.augmenter = augmenter

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = preprocess(s["path"])
        if self.is_train and self.augmenter is not None:
            img = self.augmenter(img)
        return {"image": img, "label": torch.tensor(s["label"], dtype=torch.long),
                "class_name": s["class_name"], "path": s["path"]}

print("BrainTumorDataset ready.")

## §6 · Shared Building Blocks (Spatial branch, Morphology branch, Gated Fusion)

Reused **verbatim** from the SMGF-Net notebook: `ECA`, `DWSepBlock`, `SpatialBackbone`, `MultiScaleSpatial` (the Spatial branch, Fs), `MorphologyBranch` (Fm), `GatedFusion2` (Gated Fusion), `SmallHead`. None of these are redesigned in this notebook — per the task brief, the only architectural change is the attention module (§7–§8).

`SimpleAvgFusion` is new but trivial and parameter-free: a plain 50/50 average of Fs and Fm, used only in ablation config **D** ("remove Gated Fusion") so the model still has *some* way to combine the two branches when Gated Fusion is switched off.

In [ ]:
class ECA(nn.Module):
    """Efficient Channel Attention (Wang et al., 2020). ~5-9 parameters."""
    def __init__(self, channels: int, gamma: int = 2, b: int = 1):
        super().__init__()
        t = int(abs((math.log2(channels) + b) / gamma))
        k = t if t % 2 else t + 1
        k = max(k, 3)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k, padding=(k - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.avg_pool(x)
        y = y.squeeze(-1).transpose(-1, -2)
        y = self.conv(y)
        y = self.sigmoid(y).transpose(-1, -2).unsqueeze(-1)
        return x * y.expand_as(x)


class DWSepBlock(nn.Module):
    """Depthwise-separable conv block: DW3x3 -> BN -> act -> PW1x1 -> BN -> act,
    with an optional stride-2 downsample and a residual connection when shapes match."""
    def __init__(self, c_in: int, c_out: int, stride: int = 1, act=nn.SiLU):
        super().__init__()
        self.dw = nn.Conv2d(c_in, c_in, 3, stride=stride, padding=1, groups=c_in, bias=False)
        self.bn1 = nn.BatchNorm2d(c_in)
        self.pw = nn.Conv2d(c_in, c_out, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        self.act = act(inplace=True)
        self.use_res = (stride == 1 and c_in == c_out)

    def forward(self, x):
        identity = x
        x = self.act(self.bn1(self.dw(x)))
        x = self.bn2(self.pw(x))
        if self.use_res:
            x = x + identity
        return self.act(x)


class SpatialBackbone(nn.Module):
    """Lightweight from-scratch depthwise-separable stem + 3 stages. Exposes
    F1/F2/F3 (stage outputs, strides /4, /8, /16) for multi-scale fusion --
    deliberately NOT a pretrained off-the-shelf backbone."""
    def __init__(self, stem_ch=32, c1=48, c2=88, c3=160, depth=(1, 2, 2)):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, stem_ch, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(stem_ch),
            nn.SiLU(inplace=True),
        )
        self.stage1 = self._make_stage(stem_ch, c1, depth[0])
        self.stage2 = self._make_stage(c1, c2, depth[1])
        self.stage3 = self._make_stage(c2, c3, depth[2])
        self.out_channels = [c1, c2, c3]

    @staticmethod
    def _make_stage(c_in, c_out, n_extra):
        layers = [DWSepBlock(c_in, c_out, stride=2)]
        for _ in range(n_extra):
            layers.append(DWSepBlock(c_out, c_out, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)          # /2
        f1 = self.stage1(x)       # /4
        f2 = self.stage2(f1)      # /8
        f3 = self.stage3(f2)      # /16
        return f1, f2, f3


class MultiScaleSpatial(nn.Module):
    """Spatial representation Fs: projects F1/F2/F3 to a common width, aligns
    to F3's resolution, fuses with learned global softmax weights, then a
    depthwise residual refine."""
    def __init__(self, in_channels_list: List[int], out_channels: int):
        super().__init__()
        self.proj = nn.ModuleList([
            nn.Conv2d(c, out_channels, 1, bias=False) for c in in_channels_list
        ])
        self.bn = nn.ModuleList([nn.BatchNorm2d(out_channels) for _ in in_channels_list])
        self.scale_weight = nn.Parameter(torch.ones(len(in_channels_list)))
        self.refine = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, padding=1, groups=out_channels, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, feats: List[torch.Tensor]) -> torch.Tensor:
        target_hw = feats[-1].shape[-2:]
        aligned = [F.adaptive_avg_pool2d(bn(p(f)), target_hw)
                   for p, bn, f in zip(self.proj, self.bn, feats)]
        alpha = torch.softmax(self.scale_weight, dim=0)
        fused = sum(a * x for a, x in zip(alpha, aligned))
        return fused + self.refine(fused)


class MorphologyBranch(nn.Module):
    """Morphology-aware representation Fm. A fixed, parameter-free
    morphological-gradient operator (dilation - erosion via max-pool) applied
    to F3, then a learnable depthwise+pointwise transform, trained end-to-end."""
    def __init__(self, c_in: int, out_channels: int):
        super().__init__()
        self.pool = nn.MaxPool2d(3, stride=1, padding=1)
        self.dw = nn.Conv2d(c_in, c_in, 3, padding=1, groups=c_in, bias=False)
        self.bn1 = nn.BatchNorm2d(c_in)
        self.pw = nn.Conv2d(c_in, out_channels, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        dilation = self.pool(x)
        erosion = -self.pool(-x)
        grad = dilation - erosion          # morphological gradient, parameter-free
        x = self.act(self.bn1(self.dw(grad)))
        x = self.act(self.bn2(self.pw(x)))
        return x


class GatedFusion2(nn.Module):
    """Two-branch per-channel gated fusion (Spatial + Morphology). A
    per-channel, per-sample gate learned from both branches' pooled
    statistics -- gives the network the capacity to weight branches
    differently per output channel and per input image."""
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        hidden = max(channels // reduction, 8)
        self.gate = nn.Sequential(
            nn.Linear(channels * 2, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels * 2),
        )

    def forward(self, fs: torch.Tensor, fm: torch.Tensor):
        b, c = fs.shape[0], fs.shape[1]
        pooled = torch.cat([self.pool(fs).flatten(1), self.pool(fm).flatten(1)], dim=1)
        raw = self.gate(pooled).view(b, 2, c)
        weights = torch.softmax(raw, dim=1)          # per-channel, per-sample, sums to 1 over the 2 branches
        fused = (weights[:, 0, :].view(b, c, 1, 1) * fs +
                 weights[:, 1, :].view(b, c, 1, 1) * fm)
        return fused, weights


class SimpleAvgFusion(nn.Module):
    """Parameter-free 50/50 averaging fusion. Used ONLY in ablation config D
    ('remove Gated Fusion') so the model still has some way to combine
    Fs/Fm -- not part of the proposed architecture itself."""
    def forward(self, fs: torch.Tensor, fm: torch.Tensor):
        fused = 0.5 * fs + 0.5 * fm
        b, c = fs.shape[0], fs.shape[1]
        weights = torch.full((b, 2, c), 0.5, device=fs.device)
        return fused, weights


class SmallHead(nn.Module):
    """GAP -> small projection -> BatchNorm1d -> GELU -> Dropout -> Linear(4)."""
    def __init__(self, in_dim: int, hidden_dim: int, num_classes: int, dropout: float):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.bn = nn.BatchNorm1d(hidden_dim)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.gap(x).flatten(1)
        emb = self.act(self.bn(self.fc1(x)))
        out = self.fc2(self.drop(emb))
        return out, emb

print("Shared building blocks ready (Spatial branch, Morphology branch, Gated Fusion — unchanged).")

## §7 · MS-CA-ECA — Multi-Scale Coordinate-ECA Attention

The proposed attention module, replacing plain `ECA` in SMGF-Net's `GatedFusion2 -> ECA -> SmallHead` pipeline.

```
        Fused Feature (Fs, Fm gated-fused)
                     │
          ┌──────────┴──────────┐
          │                     │
     Multi-scale            Coordinate
     context (dilated        Attention
     depthwise convs)        (H-direction +
          │                   W-direction pools)
          └──────────┬──────────┘
                     ↓
                   ECA
                     ↓
                Residual
                     ↓
              Refined Feature
```

- **`MultiScaleContext`** — captures multi-scale contextual information cheaply: a 1×1 channel-reduction, three *parallel depthwise* 3×3 convs at dilation 1/2/3 (receptive fields 3×3 / 5×5 / 7×7 — local + wider spatial context without any full/non-depthwise convolution), concatenated and expanded back with a 1×1.
- **`CoordinateAttention`** — captures direction-specific information: separate average-pools along the H and W axes (so height-direction and width-direction statistics aren't blended together the way a plain global-average-pool would), a shared 1×1 reduction, then two 1×1 projections producing an H-gate and a W-gate.
- **`ECA`** (reused from §6) — channel-importance refinement, ~5 parameters.
- These three stages are summed/applied in sequence and wrapped in a residual connection.

`use_multiscale` / `use_coord` / `use_eca` gate each stage independently (identity pass-through when off) — this is what lets the *same* `MSCAECA` class serve both the full module and every internal-ablation variant (G/H/I) in §16, with no duplicated code.

In [ ]:
class MultiScaleContext(nn.Module):
    """Cheap multi-scale contextual extraction: channel-reduce with 1x1, three
    parallel DEPTHWISE 3x3 convs at dilation 1/2/3 (receptive fields 3/5/7),
    concat, expand back with 1x1. No full (non-depthwise, non-1x1) convs, per
    the lightweight-operations requirement."""
    def __init__(self, channels: int, reduction: int = 18):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.reduce = nn.Conv2d(channels, hidden, 1, bias=False)
        self.bn_reduce = nn.BatchNorm2d(hidden)
        self.dw1 = nn.Conv2d(hidden, hidden, 3, padding=1, dilation=1, groups=hidden, bias=False)
        self.dw2 = nn.Conv2d(hidden, hidden, 3, padding=2, dilation=2, groups=hidden, bias=False)
        self.dw3 = nn.Conv2d(hidden, hidden, 3, padding=3, dilation=3, groups=hidden, bias=False)
        self.bn_cat = nn.BatchNorm2d(hidden * 3)
        self.expand = nn.Conv2d(hidden * 3, channels, 1, bias=False)
        self.bn_expand = nn.BatchNorm2d(channels)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        r = self.act(self.bn_reduce(self.reduce(x)))
        m = torch.cat([self.dw1(r), self.dw2(r), self.dw3(r)], dim=1)
        m = self.act(self.bn_cat(m))
        return self.bn_expand(self.expand(m))


class CoordinateAttention(nn.Module):
    """Coordinate Attention (Hou et al., 2021), lightweight form: separately
    pools along H and W to preserve direction-specific information, encodes
    both with a shared 1x1 reduction, then splits back into an H-gate and a
    W-gate (both 1x1 projections, no full convolutions)."""
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        self.reduce = nn.Conv2d(channels, hidden, 1, bias=False)
        self.bn = nn.BatchNorm2d(hidden)
        self.act = nn.SiLU(inplace=True)
        self.conv_h = nn.Conv2d(hidden, channels, 1, bias=False)
        self.conv_w = nn.Conv2d(hidden, channels, 1, bias=False)

    def forward(self, x):
        b, c, h, w = x.shape
        x_h = self.pool_h(x)                          # (b,c,h,1) -- height-direction info
        x_w = self.pool_w(x).permute(0, 1, 3, 2)       # (b,c,w,1) -- width-direction info
        y = torch.cat([x_h, x_w], dim=2)               # (b,c,h+w,1)
        y = self.act(self.bn(self.reduce(y)))
        y_h, y_w = torch.split(y, [h, w], dim=2)
        y_w = y_w.permute(0, 1, 3, 2)                  # back to (b,hidden,1,w)
        g_h = torch.sigmoid(self.conv_h(y_h))          # (b,c,h,1)
        g_w = torch.sigmoid(self.conv_w(y_w))          # (b,c,1,w)
        return x * g_h * g_w


class MSCAECA(nn.Module):
    """Multi-Scale Coordinate-ECA Attention (proposed). Fused feature ->
    parallel {MultiScaleContext, CoordinateAttention} -> ECA channel
    refinement -> residual. use_multiscale / use_coord / use_eca gate each
    stage independently for the internal-ablation study (F-I, section 16);
    a disabled stage is an identity pass-through, so this ONE class serves
    both the full module and every internal-ablation variant -- there is no
    separate ECA/SE/CBAM comparison code."""
    def __init__(self, channels: int, ms_reduction: int = 18, coord_reduction: int = 16,
                 use_multiscale: bool = True, use_coord: bool = True, use_eca: bool = True):
        super().__init__()
        self.use_multiscale = use_multiscale
        self.use_coord = use_coord
        self.use_eca = use_eca
        self.ms = MultiScaleContext(channels, reduction=ms_reduction) if use_multiscale else None
        self.coord = CoordinateAttention(channels, reduction=coord_reduction) if use_coord else None
        self.eca = ECA(channels) if use_eca else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        out = x
        if self.ms is not None:
            out = out + self.ms(out)
        if self.coord is not None:
            out = self.coord(out)
        if self.eca is not None:
            out = self.eca(out)
        return out + identity

print("MS-CA-ECA (Multi-Scale Coordinate-ECA Attention) ready.")

## §8 · SMGF-Net + MS-CA-ECA (final proposed model)

`SMGFNetMSCAECA` is the **only** final architecture used in this notebook. It reuses `SpatialBackbone`, `MultiScaleSpatial` (Spatial branch), `MorphologyBranch`, and `GatedFusion2` exactly as defined in §6 — none of those are redesigned here — and replaces plain `ECA` with `MSCAECA` from §7.

The class also accepts `use_spatial` / `use_morphology` / `use_gated_fusion` / `attention` (component-ablation switches, §15) and forwards `ms_kwargs` for `use_multiscale` / `use_coord` / `use_eca` (internal-attention-ablation switches, §16). **This is the same class for every experiment in this notebook** (the full model and every ablation variant) — there are no separate duplicated model definitions to keep in sync.

**Parameter budget.** The reference SMGF-Net (with plain ECA) trained at 211,560 parameters. Plain ECA is only ~5 parameters, so replacing it with anything more capable (MS-CA-ECA) and staying **under the same 211,560-parameter ceiling** required freeing a little headroom elsewhere. The Spatial branch, Morphology branch, and Gated Fusion *mechanisms* are unchanged; the only concession made was narrowing the shared representation width (`rep_width`) from 160 to **148** so those (otherwise-untouched) branches take up slightly less of the budget, leaving room for MS-CA-ECA. This is a capacity/width adjustment, not a redesign of any branch's mechanism — flagged here explicitly rather than left implicit.

In [ ]:
# Channel configuration -- rep_width narrowed from SMGF-Net's 160 to 148 to make
# room for MS-CA-ECA under the same 211,560-parameter ceiling (see markdown above).
# Spatial branch / Morphology branch / Gated Fusion configs are otherwise IDENTICAL
# to the reference SMGF-Net.
SMGF_MSCAECA_CONFIG = dict(stem_ch=32, c1=48, c2=88, c3=160, depth=(1, 2, 2),
                            rep_width=148, head_hidden=128)
MSCAECA_CONFIG = dict(ms_reduction=18, coord_reduction=16)
REFERENCE_SMGFNET_PARAMS = 211_560   # previous model (plain ECA), for the required parameter-diff report
PARAM_BUDGET = 211_560               # hard constraint: MS-CA-ECA SMGF-Net <= this


class SMGFNetMSCAECA(nn.Module):
    """SMGF-Net + MS-CA-ECA -- the final proposed architecture, and the ONLY
    model class used throughout this notebook (full model + every ablation
    variant, selected via the flags below).

    Input MRI -> SpatialBackbone (stem + 3 depthwise-separable stages,
    exposing F1/F2/F3) -> {MultiScaleSpatial(Fs), MorphologyBranch(Fm)} ->
    GatedFusion2 -> MS-CA-ECA -> SmallHead -> 4-class logits.

    use_spatial / use_morphology / use_gated_fusion / attention: component
    ablation (section 15, configs A-E). ms_kwargs (use_multiscale /
    use_coord / use_eca): MS-CA-ECA internal ablation (section 16, configs
    F-I). Spatial branch, Morphology branch and Gated Fusion are NOT
    redesigned for any of these ablations -- only removed/replaced wholesale,
    exactly as the task brief specifies.
    """
    def __init__(self, num_classes: int = NUM_CLASSES, dropout: float = DROPOUT_RATE,
                 stem_ch: int = 32, c1: int = 48, c2: int = 88, c3: int = 160,
                 depth=(1, 2, 2), rep_width: int = 148, head_hidden: int = 128,
                 use_spatial: bool = True, use_morphology: bool = True,
                 use_gated_fusion: bool = True, attention: str = "msca_eca",
                 ms_kwargs: Optional[Dict[str, Any]] = None):
        super().__init__()
        assert use_spatial or use_morphology, "at least one of Spatial/Morphology must be active"
        self.use_spatial = use_spatial
        self.use_morphology = use_morphology
        self.use_gated_fusion = use_gated_fusion
        self.attention_type = attention
        ms_kwargs = {**MSCAECA_CONFIG, **(ms_kwargs or {})}

        self.backbone = SpatialBackbone(stem_ch, c1, c2, c3, depth=depth)
        self.multiscale = MultiScaleSpatial(self.backbone.out_channels, rep_width) if use_spatial else None
        self.morphology = MorphologyBranch(c3, rep_width) if use_morphology else None

        if use_spatial and use_morphology:
            self.fusion = GatedFusion2(channels=rep_width) if use_gated_fusion else SimpleAvgFusion()
        else:
            self.fusion = None   # single-branch ablation (B/C): nothing to fuse

        if attention == "msca_eca":
            self.attn = MSCAECA(rep_width, **ms_kwargs)
        elif attention == "eca":
            self.attn = ECA(rep_width)
        elif attention == "none":
            self.attn = nn.Identity()
        else:
            raise ValueError(f"unknown attention type: {attention}")

        self.head = SmallHead(rep_width, head_hidden, num_classes, dropout)

    def forward(self, x: torch.Tensor, return_all: bool = False) -> Dict[str, torch.Tensor]:
        f1, f2, f3 = self.backbone(x)
        fs = self.multiscale([f1, f2, f3]) if self.use_spatial else None
        fm = self.morphology(f3) if self.use_morphology else None

        weights = None
        if self.use_spatial and self.use_morphology:
            fused, weights = self.fusion(fs, fm)
        elif self.use_spatial:
            fused = fs
        else:
            fused = fm

        attn = self.attn(fused)
        logits, emb = self.head(attn)
        out = {"logits": logits, "emb": emb}
        if return_all:
            out.update({"f1": f1, "f2": f2, "f3": f3, "spatial": fs, "morphology": fm,
                        "fusion_weights": weights, "fused": fused, "attn": attn})
        return out


def build_smgf_mscaeca(**kwargs) -> SMGFNetMSCAECA:
    """Builds SMGF-Net + MS-CA-ECA (or an ablation variant of it). Any of
    SMGFNetMSCAECA's constructor args can be overridden via kwargs, e.g.
    build_smgf_mscaeca(use_spatial=False) or
    build_smgf_mscaeca(ms_kwargs=dict(use_coord=False))."""
    cfg = {**SMGF_MSCAECA_CONFIG}
    ms_override = kwargs.pop("ms_kwargs", None)
    cfg.update(kwargs)
    if ms_override is not None:
        cfg["ms_kwargs"] = ms_override
    model = SMGFNetMSCAECA(**cfg).to(DEVICE)
    return model


def param_breakdown(model: "SMGFNetMSCAECA") -> Dict[str, int]:
    def count(m):
        return 0 if m is None else sum(p.numel() for p in m.parameters() if p.requires_grad)
    parts = {
        "backbone (stem + 3 DW-sep stages)": count(model.backbone),
        "spatial branch (F1/F2/F3 -> Fs)": count(model.multiscale),
        "morphology branch (Fm)": count(model.morphology),
        "gated fusion (Fs, Fm -> fused)": count(model.fusion),
        "MS-CA-ECA attention": count(model.attn),
        "classifier head": count(model.head),
    }
    parts["TOTAL"] = sum(v for k, v in parts.items())
    return parts


print("build_smgf_mscaeca() ready.")

In [ ]:
### Parameter budget check (hard constraint: <= 211,560, the previous model's count)

_probe = build_smgf_mscaeca()
total_params = sum(p.numel() for p in _probe.parameters())
trainable_params = sum(p.numel() for p in _probe.parameters() if p.requires_grad)
non_trainable_params = total_params - trainable_params

print(f"Total parameters:         {total_params:,}")
print(f"Trainable parameters:     {trainable_params:,}")
print(f"Non-trainable parameters: {non_trainable_params:,}")
print(f"Parameter difference vs. previous model (SMGF-Net, {REFERENCE_SMGFNET_PARAMS:,} params): "
      f"{trainable_params - REFERENCE_SMGFNET_PARAMS:+,}")
print(f"Budget OK (<= {PARAM_BUDGET:,}): {trainable_params <= PARAM_BUDGET}")
assert trainable_params <= PARAM_BUDGET, (
    f"SMGF-Net + MS-CA-ECA exceeds the {PARAM_BUDGET:,}-parameter budget: {trainable_params:,}")

_breakdown = param_breakdown(_probe)
print("\nSMGF-Net + MS-CA-ECA parameter breakdown by component:")
for k, v in _breakdown.items():
    print(f"  {k:38s} {v:>10,}")
print("\nMS-CA-ECA internal breakdown:")
print(f"  multi-scale context   {sum(p.numel() for p in _probe.attn.ms.parameters()):>10,}")
print(f"  coordinate attention  {sum(p.numel() for p in _probe.attn.coord.parameters()):>10,}")
print(f"  ECA refinement        {sum(p.numel() for p in _probe.attn.eca.parameters()):>10,}")

_probe_input = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
_probe_out = _probe(_probe_input, return_all=True)
print(f"\nlogits shape: {tuple(_probe_out['logits'].shape)}   "
      f"fusion_weights shape: {tuple(_probe_out['fusion_weights'].shape)}")

try:
    from thop import profile
    _macs, _ = profile(build_smgf_mscaeca(), inputs=(_probe_input[:1],), verbose=False)
    print(f"MACs (bs=1, 224x224): {_macs:,.0f}  (~{2 * _macs / 1e9:.4f} GFLOPs)")
except Exception as e:
    print(f"[thop profiling skipped: {e}]")

print(f"Model size (fp32): {total_params * 4 / (1024 ** 2):.3f} MB")

del _probe, _probe_input, _probe_out
if DEVICE == "cuda":
    torch.cuda.empty_cache()

print("\nSMGFNetMSCAECA ready -- this is the ONLY model class used throughout the rest of this notebook.")

## §9 · CAWL+ — Calibration-Aware Loss

Reused verbatim.

In [ ]:
def expected_calibration_error(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15):
    """Returns (ece, bin_accs, bin_confs, bin_counts)."""
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies = (predictions == labels).astype(np.float32)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_accs, bin_confs, bin_counts = [], [], []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        count = mask.sum()
        if count == 0:
            bin_accs.append(0); bin_confs.append(0); bin_counts.append(0)
            continue
        bin_acc = accuracies[mask].mean()
        bin_conf = confidences[mask].mean()
        ece += (count / len(labels)) * abs(bin_acc - bin_conf)
        bin_accs.append(bin_acc); bin_confs.append(bin_conf); bin_counts.append(count)
    return ece, bin_accs, bin_confs, bin_counts


def compute_class_weights(samples: List[Dict], num_classes: int = NUM_CLASSES) -> torch.Tensor:
    """Inverse-frequency class weights, normalized to mean 1.0."""
    counts = np.zeros(num_classes)
    for s in samples:
        counts[s["label"]] += 1
    counts = np.maximum(counts, 1)
    weights = counts.sum() / (num_classes * counts)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)


class CAWLPlus(nn.Module):
    def __init__(self, class_weights: torch.Tensor, num_classes: int = NUM_CLASSES,
                 emb_dim: int = 64, label_smoothing: float = 0.05,
                 target_confidence_cap: float = 0.85, lambda_max: float = 0.3,
                 anneal_epochs: int = 15, proto_momentum: float = 0.9,
                 proto_margin: float = 0.5, beta: float = 0.1,
                 proto_warmup_epochs: int = 8):
        super().__init__()
        self.register_buffer("class_weights", class_weights)
        self.label_smoothing = label_smoothing
        self.target_confidence_cap = target_confidence_cap
        self.lambda_max = lambda_max
        self.anneal_epochs = anneal_epochs
        self.current_epoch = 0

        self.num_classes = num_classes
        self.proto_momentum = proto_momentum
        self.proto_margin = proto_margin
        self.beta = beta
        self.proto_warmup_epochs = proto_warmup_epochs
        self.register_buffer("prototypes", torch.zeros(num_classes, emb_dim))
        self.register_buffer("proto_initialized", torch.zeros(num_classes, dtype=torch.bool))

    def _lambda_t(self):
        frac = min(self.current_epoch / max(self.anneal_epochs, 1), 1.0)
        return self.lambda_max * frac

    @torch.no_grad()
    def _update_prototypes(self, emb: torch.Tensor, targets: torch.Tensor):
        for c in range(self.num_classes):
            mask = targets == c
            if mask.sum() == 0:
                continue
            batch_mean = emb[mask].mean(dim=0)
            if not self.proto_initialized[c]:
                self.prototypes[c] = batch_mean
                self.proto_initialized[c] = True
            else:
                self.prototypes[c] = (self.proto_momentum * self.prototypes[c] +
                                       (1 - self.proto_momentum) * batch_mean)

    def _prototype_margin_loss(self, emb: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        if not self.proto_initialized.all():
            return torch.tensor(0.0, device=emb.device)
        protos = self.prototypes.to(emb.device)
        d = torch.cdist(emb, protos) ** 2
        d_pos = d.gather(1, targets.view(-1, 1)).squeeze(1)
        d_masked = d.clone()
        d_masked.scatter_(1, targets.view(-1, 1), float("inf"))
        d_neg = d_masked.min(dim=1).values
        return F.relu(d_pos - d_neg + self.proto_margin).mean()

    def forward(self, logits: torch.Tensor, targets: torch.Tensor,
                emb: Optional[torch.Tensor] = None) -> torch.Tensor:
        ce = F.cross_entropy(logits, targets, weight=self.class_weights,
                              label_smoothing=self.label_smoothing)
        probs = F.softmax(logits, dim=1)
        conf, _ = probs.max(dim=1)
        pred = logits.argmax(dim=1)
        correct = (pred == targets).float()
        overconf = F.relu(conf - self.target_confidence_cap) * correct
        conf_penalty = overconf.mean()
        loss = ce + self._lambda_t() * conf_penalty

        if emb is not None and self.training:
            self._update_prototypes(emb.detach(), targets)   # EMA warms up silently from epoch 1
        if emb is not None and self.current_epoch >= self.proto_warmup_epochs:
            loss = loss + self.beta * self._prototype_margin_loss(emb, targets)  # gated
        return loss

print("CAWLPlus ready (with proto_warmup_epochs gate).")

## §10 · Dataset Scanning & Leakage-Check Utilities

Reused verbatim.

In [ ]:
def _file_hash(path: str) -> str:
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def dedupe_splits(splits: Dict[str, List[Dict]],
                   priority: List[str] = ["test", "val", "train"]) -> Dict[str, List[Dict]]:
    hash_to_owner: Dict[str, Tuple[str, int]] = {}
    to_remove: Dict[str, set] = {name: set() for name in splits}
    n_cross = n_intra = 0

    for name in sorted(splits, key=lambda n: priority.index(n) if n in priority else 99):
        for idx, s in enumerate(splits[name]):
            h = _file_hash(s["path"])
            if h in hash_to_owner:
                owner_name, owner_idx = hash_to_owner[h]
                if owner_name == name:
                    n_intra += 1
                    if n_intra <= 5:
                        print(f"  [DUPLICATE within '{name}'] {splits[name][owner_idx]['path']}  ==  {s['path']}")
                else:
                    n_cross += 1
                    if n_cross <= 5:
                        print(f"  [LEAKAGE: kept in '{owner_name}', removed from '{name}'] {s['path']}")
                to_remove[name].add(idx)
            else:
                hash_to_owner[h] = (name, idx)

    cleaned = {name: [s for i, s in enumerate(samples) if i not in to_remove[name]]
               for name, samples in splits.items()}
    print(f"[LEAKAGE CHECK] {n_cross} cross-split leak(s) + {n_intra} intra-split "
          f"duplicate(s) auto-removed ({n_cross + n_intra} total).")
    for name in splits:
        removed = len(to_remove[name])
        if removed:
            print(f"  {name}: {len(splits[name])} -> {len(cleaned[name])} ({removed} removed)")
    return cleaned


def scan_dataset_generic(root: Path, folder_to_class: Dict[str, str], split_name: str) -> List[Dict[str, Any]]:
    samples = []
    for folder_name, cls in folder_to_class.items():
        cls_dir = root / folder_name
        if not cls_dir.exists():
            print(f"  [WARN] missing: {cls_dir}")
            continue
        n_before = len(samples)
        for p in cls_dir.rglob("*"):
            if p.is_file() and p.suffix.lower() in IMG_EXTS:
                samples.append({"path": str(p), "label": CLASS_TO_IDX[cls],
                                 "class_name": cls, "split": split_name})
        print(f"  {split_name}/{folder_name} -> {cls}: {len(samples) - n_before} images")
    print(f"{split_name}: {len(samples)} images across {NUM_CLASSES} classes")
    return samples

print("Scanner + leakage-check utilities ready.")

## §11 · Dataset Paths (BRISC2025)

In [ ]:
BRISC_ROOT = Path("/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/classification_task")
BRISC_TRAIN_ROOT = BRISC_ROOT / "train"
BRISC_TEST_ROOT  = BRISC_ROOT / "test"
print(f"BRISC2025 train exists: {BRISC_TRAIN_ROOT.exists()} -> {BRISC_TRAIN_ROOT}")
print(f"BRISC2025 test  exists: {BRISC_TEST_ROOT.exists()} -> {BRISC_TEST_ROOT}")
if not (BRISC_TRAIN_ROOT.exists() and BRISC_TEST_ROOT.exists()):
    raise RuntimeError("BRISC2025 not found. Make sure it's attached to this "
                        "Kaggle notebook (Add Input -> search 'brisc2025').")

BRISC_FOLDER_TO_CLASS = {
    "glioma": "glioma", "meningioma": "meningioma",
    "no_tumor": "notumor", "pituitary": "pituitary",
}

## §12 · LR Schedule, Training & Evaluation Utilities

Reused verbatim.

In [ ]:
TOTAL_EPOCHS  = 50
WEIGHT_DECAY  = 2e-4
GRAD_CLIP     = 1.0
PEAK_LR       = 1.8e-4   # was 4e-4 -- too high a peak was a major source of curve noise
MIN_LR        = 1e-6

WARMUP_EPOCHS_V2     = 15   # was 10 -- longer, gentler ramp instead of a hard 10-epoch climb
WARMUP_START_DIV_V2  = 50

FREEZE_BACKBONE_EPOCHS = 3

# Fine-grained early logging: for the first FINEGRAIN_EPOCHS epochs, evaluate
# train(clean)/val this many times DURING the epoch (not just once at the end).
# A frozen-backbone linear-probe head converges fast -- within a single epoch it can
# already climb from chance-level (~25% for 4 classes) most of the way to its plateau.
# Logging only once per epoch was hiding that climb and making it look like val
# "jumped" straight to a high number with no visible ramp. This makes the ramp visible.
FINEGRAIN_EPOCHS   = 2
FINEGRAIN_SUBSTEPS = 5

def lr_at_epoch_v2(epoch: int) -> float:
    warmup_start_lr = PEAK_LR / WARMUP_START_DIV_V2
    if epoch <= WARMUP_EPOCHS_V2:
        frac = epoch / WARMUP_EPOCHS_V2
        return warmup_start_lr + frac * (PEAK_LR - warmup_start_lr)
    remaining_epochs = TOTAL_EPOCHS - WARMUP_EPOCHS_V2
    progress = min((epoch - WARMUP_EPOCHS_V2) / max(remaining_epochs, 1), 1.0)
    return MIN_LR + 0.5 * (PEAK_LR - MIN_LR) * (1 + math.cos(math.pi * progress))


def train_one_epoch_v2(model, loader, optimizer, criterion: CAWLPlus,
                        scaler: Optional[GradScaler] = None):
    model.train()
    total_loss = correct = total = 0
    use_amp = scaler is not None
    for batch in loader:
        img   = batch["image"].to(DEVICE, non_blocking=True)
        label = batch["label"].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            with autocast():
                out  = model(img)
                loss = criterion(out["logits"], label, out["emb"])
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            out  = model(img)
            loss = criterion(out["logits"], label, out["emb"])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        pred = out["logits"].argmax(1)
        correct += (pred == label).sum().item()
        total   += label.size(0)
        total_loss += loss.item() * label.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_v2(model, loader, criterion: CAWLPlus, device: str = DEVICE) -> Dict[str, Any]:
    model.eval()
    all_preds, all_labels, all_probs, total_loss = [], [], [], 0.0
    for batch in loader:
        img   = batch["image"].to(device, non_blocking=True)
        label = batch["label"].to(device, non_blocking=True)
        out   = model(img)
        loss  = criterion(out["logits"], label, out["emb"])
        total_loss += loss.item() * label.size(0)
        probs = torch.softmax(out["logits"], dim=1).cpu().numpy()
        all_probs.extend(probs.tolist())
        all_preds.extend(out["logits"].argmax(1).cpu().tolist())
        all_labels.extend(label.cpu().tolist())

    n = len(all_labels)
    y_true = np.array(all_labels); y_pred = np.array(all_preds); y_prob = np.array(all_probs)
    try:
        auc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except Exception:
        auc = float("nan")
    cm = confusion_matrix(y_true, y_pred)
    ece, _, _, _ = expected_calibration_error(y_prob, y_true)

    return {
        "loss": total_loss / n,
        "acc": accuracy_score(y_true, y_pred),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "precision_mac": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_mac": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_mac": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "auc": auc,
        "mcc": matthews_corrcoef(y_true, y_pred),
        "kappa": cohen_kappa_score(y_true, y_pred),
        "ece": ece,
        "cm": cm,
        "report": classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0),
    }


def plot_confusion_matrix(cm: np.ndarray, title: str = "Confusion Matrix", save_path: Optional[str] = None):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right"); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.colorbar(im, ax=ax); plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

# Fine-grained ("chunked") epoch trainer -- identical math to train_one_epoch_v2,
# but pauses n_chunks-1 times during the epoch to log a clean train/val snapshot at
# that fractional-epoch point. Only used for the first FINEGRAIN_EPOCHS epochs.
def train_one_epoch_chunked_v2(model, loader, optimizer, criterion: CAWLPlus,
                                scaler: Optional[GradScaler], n_chunks: int, epoch: int,
                                train_eval_loader, val_loader, history: Dict[str, list]):
    model.train()
    use_amp = scaler is not None
    batches = list(loader)
    n_batches = len(batches)
    chunk_size = max(1, n_batches // n_chunks)
    total_loss = correct = total = 0

    for i, batch in enumerate(batches):
        img   = batch["image"].to(DEVICE, non_blocking=True)
        label = batch["label"].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            with autocast():
                out  = model(img)
                loss = criterion(out["logits"], label, out["emb"])
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            out  = model(img)
            loss = criterion(out["logits"], label, out["emb"])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        pred = out["logits"].argmax(1)
        correct += (pred == label).sum().item()
        total   += label.size(0)
        total_loss += loss.item() * label.size(0)

        is_chunk_boundary = ((i + 1) % chunk_size == 0) and (i + 1) < n_batches
        if is_chunk_boundary:
            frac_epoch = (epoch - 1) + (i + 1) / n_batches
            train_clean = evaluate_v2(model, train_eval_loader, criterion)
            val_metrics = evaluate_v2(model, val_loader, criterion)
            print(f"  epoch {frac_epoch:06.2f} (substep)  "
                  f"train_acc(clean)={train_clean['acc']:.4f}  "
                  f"val_acc={val_metrics['acc']:.4f}  val_auc={val_metrics['auc']:.4f}")
            history["epoch"].append(frac_epoch)
            history["train_loss"].append(train_clean["loss"])
            history["train_acc"].append(train_clean["acc"])
            history["train_loss_aug"].append(total_loss / total)
            history["train_acc_aug"].append(correct / total)
            history["val_loss"].append(val_metrics["loss"])
            history["val_acc"].append(val_metrics["acc"])
            history["val_auc"].append(val_metrics["auc"])
            history["val_ece"].append(val_metrics["ece"])
            history["lr"].append(optimizer.param_groups[0]["lr"])
            model.train()

    return total_loss / total, correct / total


print(f"Schedule ready: {TOTAL_EPOCHS} epochs, {WARMUP_EPOCHS_V2}-epoch gentle warmup "
      f"(start={PEAK_LR/WARMUP_START_DIV_V2:.1e}), peak_lr={PEAK_LR:.1e}, AMP={USE_AMP}, "
      f"early_stop_patience={EARLY_STOP_PATIENCE}, freeze_backbone_epochs={FREEZE_BACKBONE_EPOCHS}, "
      f"finegrain_epochs={FINEGRAIN_EPOCHS}x{FINEGRAIN_SUBSTEPS}")

## §13 · Mixup + EMA + flip-TTA experiment runner

Reused from the SMGF-Net notebook with one generalization: `run_experiment` now takes a `model_builder` (defaults to `build_smgf_mscaeca`) and `model_kwargs`, so the exact same function trains the full proposed model AND every ablation variant in §15/§16 — same dataset loading / splitting / leakage-check / preprocessing / augmentation / optimizer / LR-schedule / CAWL+ loss / Mixup / EMA / flip-TTA pipeline every time, which is what the fair-ablation-protocol requirement (section 5 of the task brief) needs.

In [ ]:
class ModelEMA:
    """Exponential moving average of the model's weights."""
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model: nn.Module):
        model.load_state_dict(self.shadow, strict=True)


def mixup_batch(img: torch.Tensor, label: torch.Tensor, alpha: float = 0.2):
    """Standard Mixup (Zhang et al., 2018)."""
    if alpha <= 0:
        return img, label, label, 1.0
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(img.size(0), device=img.device)
    mixed = lam * img + (1 - lam) * img[idx]
    return mixed, label, label[idx], lam


def mixup_cawl_loss(criterion: CAWLPlus, logits: torch.Tensor, emb: torch.Tensor,
                     y_a: torch.Tensor, y_b: torch.Tensor, lam: float) -> torch.Tensor:
    """Mixup-safe re-implementation of CAWLPlus.forward() (§9): computes CE +
    confidence-penalty for both mixed labels, but touches the prototype
    buffer only once per batch (dominant label), to avoid the in-place
    prototype-mutation issue of calling criterion(...) twice."""
    ce_a = F.cross_entropy(logits, y_a, weight=criterion.class_weights,
                            label_smoothing=criterion.label_smoothing)
    ce_b = F.cross_entropy(logits, y_b, weight=criterion.class_weights,
                            label_smoothing=criterion.label_smoothing)
    ce = lam * ce_a + (1 - lam) * ce_b

    probs = F.softmax(logits, dim=1)
    conf, _ = probs.max(dim=1)
    pred = logits.argmax(dim=1)
    correct = lam * (pred == y_a).float() + (1 - lam) * (pred == y_b).float()
    conf_penalty = (F.relu(conf - criterion.target_confidence_cap) * correct).mean()
    loss = ce + criterion._lambda_t() * conf_penalty

    y_dom = y_a if lam >= 0.5 else y_b
    if criterion.training:
        criterion._update_prototypes(emb.detach(), y_dom)
    if criterion.current_epoch >= criterion.proto_warmup_epochs:
        loss = loss + criterion.beta * criterion._prototype_margin_loss(emb, y_dom)
    return loss


def train_one_epoch_mixup(model, loader, optimizer, criterion: CAWLPlus,
                           scaler: Optional[GradScaler] = None,
                           ema: Optional[ModelEMA] = None,
                           mixup_alpha: float = 0.2):
    model.train()
    total_loss = correct = total = 0
    use_amp = scaler is not None
    for batch in loader:
        img   = batch["image"].to(DEVICE, non_blocking=True)
        label = batch["label"].to(DEVICE, non_blocking=True)
        img_m, y_a, y_b, lam = mixup_batch(img, label, alpha=mixup_alpha)
        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            with autocast():
                out  = model(img_m)
                loss = mixup_cawl_loss(criterion, out["logits"], out["emb"], y_a, y_b, lam)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            out  = model(img_m)
            loss = mixup_cawl_loss(criterion, out["logits"], out["emb"], y_a, y_b, lam)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        if ema is not None:
            ema.update(model)

        pred = out["logits"].argmax(1)
        correct += (lam * (pred == y_a).float() + (1 - lam) * (pred == y_b).float()).sum().item()
        total   += label.size(0)
        total_loss += loss.item() * label.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_v2_tta(model, loader, criterion: CAWLPlus, device: str = DEVICE) -> Dict[str, Any]:
    """Same metric suite as evaluate_v2 (§12), but averages softmax
    probabilities over the original image and its horizontal flip
    (test-time augmentation) before taking the argmax."""
    model.eval()
    all_preds, all_labels, all_probs, total_loss = [], [], [], 0.0
    for batch in loader:
        img   = batch["image"].to(device, non_blocking=True)
        label = batch["label"].to(device, non_blocking=True)
        out1  = model(img)
        out2  = model(torch.flip(img, dims=[3]))
        probs = (torch.softmax(out1["logits"], dim=1) + torch.softmax(out2["logits"], dim=1)) / 2
        loss  = criterion(out1["logits"], label, out1["emb"])
        total_loss += loss.item() * label.size(0)
        all_probs.extend(probs.cpu().numpy().tolist())
        all_preds.extend(probs.argmax(1).cpu().tolist())
        all_labels.extend(label.cpu().tolist())

    n = len(all_labels)
    y_true = np.array(all_labels); y_pred = np.array(all_preds); y_prob = np.array(all_probs)
    try:
        auc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except Exception:
        auc = float("nan")
    cm = confusion_matrix(y_true, y_pred)
    ece, _, _, _ = expected_calibration_error(y_prob, y_true)
    return {
        "loss": total_loss / n,
        "acc": accuracy_score(y_true, y_pred),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "precision_mac": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_mac": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_mac": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "auc": auc,
        "mcc": matthews_corrcoef(y_true, y_pred),
        "kappa": cohen_kappa_score(y_true, y_pred),
        "ece": ece,
        "cm": cm,
        "report": classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0),
    }


def run_experiment(dataset_name: str, train_root: Path, test_root: Path,
                    folder_to_class: Dict[str, str],
                    model_builder=None,
                    total_epochs: Optional[int] = None,
                    dropout: Optional[float] = None,
                    weight_decay: Optional[float] = None,
                    mixup_alpha: float = 0.2,
                    ema_decay: float = 0.999,
                    model_kwargs: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """Trains a model built by `model_builder(dropout=..., **model_kwargs)`
    (defaults to build_smgf_mscaeca -- the full SMGF-Net + MS-CA-ECA model)
    with the SAME dataset loading / splitting / leakage-check / preprocessing
    / augmentation / optimizer / LR-schedule / CAWL+ loss / Mixup / EMA /
    flip-TTA pipeline every time. Used for the full model (§14) AND every
    component/internal ablation variant (§15/§16) -- this identical function
    call is what keeps the ablation protocol fair (section 5 of the task
    brief): same splits, same preprocessing, same augmentation, same image
    size, same batch size, same optimizer, same LR schedule, same epochs,
    same loss, same early stopping, same seed."""
    model_builder = model_builder or build_smgf_mscaeca
    total_epochs = TOTAL_EPOCHS if total_epochs is None else total_epochs
    dropout = 0.4 if dropout is None else dropout
    weight_decay = WEIGHT_DECAY * 1.5 if weight_decay is None else weight_decay
    model_kwargs = model_kwargs or {}

    print("\n" + "#" * 65)
    print(f"# EXPERIMENT: {dataset_name}  "
          f"[dropout={dropout} weight_decay={weight_decay:.2e} mixup_alpha={mixup_alpha} "
          f"ema_decay={ema_decay} epochs={total_epochs} model_kwargs={model_kwargs}]")
    print("#" * 65)

    all_train = scan_dataset_generic(train_root, folder_to_class, "train")
    test_s    = scan_dataset_generic(test_root, folder_to_class, "test")
    if len(all_train) == 0 or len(test_s) == 0:
        raise RuntimeError(f"[{dataset_name}] No images found — check paths.")
    train_s, val_s = train_test_split(
        all_train, test_size=0.15, random_state=SEED,
        stratify=[s["label"] for s in all_train])

    splits = dedupe_splits({"train": train_s, "val": val_s, "test": test_s})
    train_s, val_s, test_s = splits["train"], splits["val"], splits["test"]

    train_ds = BrainTumorDataset(train_s, is_train=True, augmenter=train_augment)
    val_ds   = BrainTumorDataset(val_s,   is_train=False)
    test_ds  = BrainTumorDataset(test_s,  is_train=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                               num_workers=NUM_WORKERS, pin_memory=True)

    if len(train_s) > len(val_s):
        train_eval_s, _ = train_test_split(
            train_s, train_size=len(val_s), random_state=SEED,
            stratify=[s["label"] for s in train_s])
    else:
        train_eval_s = train_s
    train_eval_ds = BrainTumorDataset(train_eval_s, is_train=False)
    train_eval_loader = DataLoader(train_eval_ds, batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=NUM_WORKERS, pin_memory=True)

    model = model_builder(dropout=dropout, **model_kwargs)
    total_params = sum(p.numel() for p in model.parameters())
    ema = ModelEMA(model, decay=ema_decay)

    class_weights = compute_class_weights(train_s).to(DEVICE)
    criterion = CAWLPlus(class_weights, num_classes=NUM_CLASSES,
                          emb_dim=model.head.fc1.out_features).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=PEAK_LR, weight_decay=weight_decay)
    scaler = GradScaler() if USE_AMP else None

    history = {"epoch": [], "train_loss": [], "train_acc": [], "train_loss_aug": [], "train_acc_aug": [],
               "val_loss": [], "val_acc": [], "val_auc": [], "val_ece": [], "lr": []}

    _t0 = evaluate_v2(model, train_eval_loader, criterion)
    _v0 = evaluate_v2(model, val_loader, criterion)
    print(f"  epoch 000 (pre-training baseline)  "
          f"train_acc(clean)={_t0['acc']:.4f}  val_acc={_v0['acc']:.4f}  val_auc={_v0['auc']:.4f}")
    history["epoch"].append(0)
    history["train_loss"].append(_t0["loss"]); history["train_acc"].append(_t0["acc"])
    history["train_loss_aug"].append(_t0["loss"]); history["train_acc_aug"].append(_t0["acc"])
    history["val_loss"].append(_v0["loss"]); history["val_acc"].append(_v0["acc"])
    history["val_auc"].append(_v0["auc"]); history["val_ece"].append(_v0["ece"])
    history["lr"].append(0.0)

    best_val_auc = -1.0
    best_state = None
    epochs_no_improve = 0

    for epoch in range(1, total_epochs + 1):
        for g in optimizer.param_groups:
            g["lr"] = lr_at_epoch_v2(epoch)
        criterion.current_epoch = epoch

        train_loss, train_acc = train_one_epoch_mixup(
            model, train_loader, optimizer, criterion, scaler, ema=ema, mixup_alpha=mixup_alpha)
        train_clean = evaluate_v2(model, train_eval_loader, criterion)
        val_metrics = evaluate_v2(model, val_loader, criterion)

        print(f"  epoch {epoch:03d}  lr={lr_at_epoch_v2(epoch):.2e}  "
              f"train_acc(clean)={train_clean['acc']:.4f} train_acc(mixup)={train_acc:.4f}  "
              f"val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['acc']:.4f} "
              f"val_auc={val_metrics['auc']:.4f} val_ece={val_metrics['ece']:.4f}")

        history["epoch"].append(epoch)
        history["train_loss"].append(train_clean["loss"]); history["train_acc"].append(train_clean["acc"])
        history["train_loss_aug"].append(train_loss); history["train_acc_aug"].append(train_acc)
        history["val_loss"].append(val_metrics["loss"]); history["val_acc"].append(val_metrics["acc"])
        history["val_auc"].append(val_metrics["auc"]); history["val_ece"].append(val_metrics["ece"])
        history["lr"].append(lr_at_epoch_v2(epoch))

        if val_metrics["auc"] > best_val_auc:
            best_val_auc = val_metrics["auc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= EARLY_STOP_PATIENCE:
                print(f"  [EARLY STOP] no val AUC improvement for {EARLY_STOP_PATIENCE} epochs.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    raw_val = evaluate_v2(model, val_loader, criterion)

    ema_model = model_builder(dropout=dropout, **model_kwargs)
    ema.copy_to(ema_model)
    ema_val = evaluate_v2(ema_model, val_loader, criterion)

    if ema_val["auc"] >= raw_val["auc"]:
        print(f"  [FINAL WEIGHTS] EMA (val_auc={ema_val['auc']:.4f}) >= "
              f"best-checkpoint (val_auc={raw_val['auc']:.4f}) -> using EMA weights.")
        model = ema_model
    else:
        print(f"  [FINAL WEIGHTS] best-checkpoint (val_auc={raw_val['auc']:.4f}) > "
              f"EMA (val_auc={ema_val['auc']:.4f}) -> keeping best-checkpoint weights.")
        del ema_model

    test_metrics = evaluate_v2_tta(model, test_loader, criterion)
    print(f"[{dataset_name} [TEST, flip-TTA]] Accuracy={test_metrics['acc']:.4f}  "
          f"BalancedAcc={test_metrics['balanced_acc']:.4f}  F1(macro)={test_metrics['f1_mac']:.4f}  "
          f"AUC={test_metrics['auc']:.4f}  MCC={test_metrics['mcc']:.4f}  "
          f"CohenKappa={test_metrics['kappa']:.4f}  ECE={test_metrics['ece']:.4f}")
    print(test_metrics["report"])
    plot_confusion_matrix(test_metrics["cm"], title=f"{dataset_name} Confusion Matrix (flip-TTA)",
                           save_path=str(OUT_DIR / f"{dataset_name}_confusion_matrix.png"))

    torch.save(model.state_dict(), OUT_DIR / f"{dataset_name}.pth")

    return {
        "name": dataset_name, "model": model, "criterion": criterion,
        "total_params": total_params,
        "n_train": len(train_s), "n_val": len(val_s), "n_test": len(test_s),
        "train_samples": train_s, "val_samples": val_s, "test_samples": test_s,
        "test_metrics": test_metrics,
        "history": history,
        "config": {"dropout": dropout, "weight_decay": weight_decay, "mixup_alpha": mixup_alpha,
                   "ema_decay": ema_decay, "epochs": total_epochs, **SMGF_MSCAECA_CONFIG, **model_kwargs},
    }

print("run_experiment() ready (Mixup + EMA + flip-TTA, model-builder-generic).")

## §14 · Train the FULL proposed model — SMGF-Net + MS-CA-ECA — on BRISC2025

In [ ]:
full_result = run_experiment("A_full_smgfnet_mscaeca", BRISC_TRAIN_ROOT, BRISC_TEST_ROOT, BRISC_FOLDER_TO_CLASS,
                              model_builder=build_smgf_mscaeca)
full_model = full_result["model"]

_m = full_result["test_metrics"]
print("\n[SMGF-Net + MS-CA-ECA -- FULL MODEL summary]")
print(f"  Parameters   = {full_result['total_params']:,}  "
      f"(budget {PARAM_BUDGET:,}, diff vs. previous model = {full_result['total_params'] - REFERENCE_SMGFNET_PARAMS:+,})")
print(f"  Test Accuracy       = {_m['acc']:.4f}")
print(f"  Balanced Accuracy   = {_m['balanced_acc']:.4f}")
print(f"  Macro Precision     = {_m['precision_mac']:.4f}")
print(f"  Macro Recall        = {_m['recall_mac']:.4f}")
print(f"  Macro F1            = {_m['f1_mac']:.4f}")
print(f"  Weighted F1         = {_m['f1_weighted']:.4f}")
print(f"  MCC                 = {_m['mcc']:.4f}")
print(f"  ROC-AUC             = {_m['auc']:.4f}")

## §15 · Training Curves & Classification Report (Full Model)


In [ ]:

glioma_idx, meningioma_idx = CLASS_TO_IDX["glioma"], CLASS_TO_IDX["meningioma"]
m = full_result["test_metrics"]
print(m["report"])
plot_confusion_matrix(m["cm"], title="SMGF-Net + MS-CA-ECA \u2014 Confusion Matrix",
                       save_path=str(OUT_DIR / "confmat_full.png"))
cm = m["cm"]
g_to_m = cm[glioma_idx, meningioma_idx]
m_to_g = cm[meningioma_idx, glioma_idx]
g_total = cm[glioma_idx].sum(); m_total = cm[meningioma_idx].sum()
print(f"  glioma\u2192meningioma misclassifications: {g_to_m}/{g_total} "
      f"({100*g_to_m/max(g_total,1):.1f}%)")


In [ ]:
def plot_training_curves_v2(history: Dict[str, list], title_prefix: str,
                             save_path: Optional[str] = None):
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))

    ep = history.get("epoch", list(range(1, len(history["val_loss"]) + 1)))
    ax = axes[0, 0]
    ax.plot(ep, history["train_loss"], color="#e8664a", alpha=0.4, linestyle="--", label="Train")
    ax.plot(ep, history["val_loss"], color="#e8664a", linewidth=2, label="Val")
    ax.set_title("Loss"); ax.set_xlabel("Epoch"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.set_ylim(0, max(history["train_loss"] + history["val_loss"]) * 1.1)

    ax = axes[0, 1]
    ax.plot(ep, history["train_acc"], color="#4ac9c2", alpha=0.4, linestyle="--", label="Train")
    ax.plot(ep, history["val_acc"], color="#4ac9c2", linewidth=2, label="Val")
    ax.set_title("Accuracy"); ax.set_xlabel("Epoch"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.set_ylim(0, 1.02)

    ax = axes[1, 0]
    ax.plot(ep, history["val_auc"], color="#8b7fd6", linewidth=2, label="Val AUC")
    ax.set_title("Val AUC"); ax.set_xlabel("Epoch"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.set_ylim(0, 1.02)

    ax = axes[1, 1]
    ax.plot(ep, history["val_ece"], color="#e0a94a", linewidth=2, label="Val ECE")
    ax.set_title("Val ECE"); ax.set_xlabel("Epoch"); ax.legend(fontsize=8, loc="upper left"); ax.grid(alpha=0.3)
    ax2 = ax.twinx()
    ax2.plot(ep, history["lr"], color="#999999", linewidth=1, linestyle=":", label="LR")
    ax2.set_ylabel("LR"); ax2.legend(fontsize=8, loc="upper right")

    fig.suptitle(f"Training Curves — {title_prefix}")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def summarize_training(history: Dict[str, list], name: str) -> Dict[str, float]:
    """Best validation epoch/accuracy, final train/val accuracy, and the
    train-minus-val accuracy gap at the best epoch (overfitting indicator)."""
    val_acc = history["val_acc"]
    best_idx = int(np.argmax(val_acc))
    best_epoch = history["epoch"][best_idx]
    best_val_acc = val_acc[best_idx]
    final_train_acc = history["train_acc"][-1]
    final_val_acc = history["val_acc"][-1]
    overfit_gap = history["train_acc"][best_idx] - best_val_acc
    print(f"[{name}] best_val_epoch={best_epoch}  best_val_acc={best_val_acc:.4f}  "
          f"final_train_acc={final_train_acc:.4f}  final_val_acc={final_val_acc:.4f}  "
          f"train-val_gap_at_best_epoch={overfit_gap:+.4f}"
          f"{'  [possible overfitting]' if overfit_gap > 0.05 else ''}")
    return {"name": name, "best_epoch": best_epoch, "best_val_acc": best_val_acc,
            "final_train_acc": final_train_acc, "final_val_acc": final_val_acc,
            "overfit_gap": overfit_gap}

In [ ]:
plot_training_curves_v2(full_result["history"], "SMGF-Net + MS-CA-ECA",
                         save_path=str(OUT_DIR / "training_curves_full.png"))
summarize_training(full_result["history"], "SMGF-Net + MS-CA-ECA")

glioma_idx, meningioma_idx = CLASS_TO_IDX["glioma"], CLASS_TO_IDX["meningioma"]
m = full_result["test_metrics"]
print(m["report"])
plot_confusion_matrix(m["cm"], title="SMGF-Net + MS-CA-ECA — Confusion Matrix",
                       save_path=str(OUT_DIR / "confmat_full.png"))
cm = m["cm"]
g_to_m = cm[glioma_idx, meningioma_idx]
m_to_g = cm[meningioma_idx, glioma_idx]
g_total = cm[glioma_idx].sum(); m_total = cm[meningioma_idx].sum()
print(f"  glioma→meningioma misclassifications: {g_to_m}/{g_total} "
      f"({100*g_to_m/max(g_total,1):.1f}%)")

## §20 · External Validation (final proposed model only)

Evaluated ONLY for `SMGF-Net + MS-CA-ECA` (`full_result`) — not retrained, not fine-tuned, not used for architecture selection. Same preprocessing pipeline as internal evaluation. A cross-dataset file-hash check flags any byte-identical overlap with the BRISC2025 train/val/test splits before reporting a dataset as "external."

In [ ]:
def run_external_validation(model: nn.Module, criterion: CAWLPlus, dataset_name: str,
                             external_root: Path, folder_to_class: Dict[str, str],
                             known_splits: Optional[Dict[str, List[Dict]]] = None,
                             batch_size: int = BATCH_SIZE) -> Dict[str, Any]:
    """Evaluate an already-trained model on an external (out-of-distribution) dataset.
    known_splits: pass {"train": full_result["train_samples"], "val": full_result["val_samples"],
    "test": full_result["test_samples"]} to flag any file-hash overlap with the external set
    (cross-dataset leakage/contamination check)."""
    ext_samples = scan_dataset_generic(external_root, folder_to_class, dataset_name)
    if len(ext_samples) == 0:
        raise RuntimeError(f"[{dataset_name}] No images found under {external_root}")

    if known_splits is not None:
        known_hashes = set()
        for split_name, split_samples in known_splits.items():
            for s in split_samples:
                known_hashes.add(_file_hash(s["path"]))
        overlap = sum(1 for s in ext_samples if _file_hash(s["path"]) in known_hashes)
        if overlap:
            print(f"  [WARNING] {overlap}/{len(ext_samples)} external images are byte-identical "
                  f"to images already seen in train/val/test -- this dataset is CONTAMINATED, "
                  f"not independent external validation.")
        else:
            print(f"  [OK] no file-hash overlap between '{dataset_name}' and train/val/test.")

    ext_ds = BrainTumorDataset(ext_samples, is_train=False)
    ext_loader = DataLoader(ext_ds, batch_size=batch_size, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)

    ext_metrics = evaluate_v2(model, ext_loader, criterion)
    print(f"\n[{dataset_name} EXTERNAL VALIDATION]  n={len(ext_samples)}  "
          f"Accuracy={ext_metrics['acc']:.4f}  BalancedAcc={ext_metrics['balanced_acc']:.4f}  "
          f"MacroPrec={ext_metrics['precision_mac']:.4f}  MacroRec={ext_metrics['recall_mac']:.4f}  "
          f"F1(macro)={ext_metrics['f1_mac']:.4f}  F1(weighted)={ext_metrics['f1_weighted']:.4f}  "
          f"AUC={ext_metrics['auc']:.4f}  MCC={ext_metrics['mcc']:.4f}  ECE={ext_metrics['ece']:.4f}")
    print(ext_metrics["report"])
    plot_confusion_matrix(ext_metrics["cm"], title=f"{dataset_name} \u2014 External Validation Confusion Matrix",
                           save_path=str(OUT_DIR / f"{dataset_name}_external_confusion_matrix.png"))
    return ext_metrics

print("run_external_validation() ready.")

In [ ]:
# Dataset 1's single subfolder has a long, easily-truncated name (parenthetical class
# list) -- resolved dynamically so a copy/paste error can't silently point at the wrong path.
_ds1_root = Path("/kaggle/input/datasets/neyamul/bt-mendeley-3")
_ds1_subdirs = [d for d in _ds1_root.iterdir() if d.is_dir()]
assert len(_ds1_subdirs) == 1, f"Expected exactly one subdir under bt-mendeley-3, found: {_ds1_subdirs}"
DATASET1_ROOT = _ds1_subdirs[0] / "Test"

EXTERNAL_DATASETS = [
    {"name": "mendeley3", "root": DATASET1_ROOT,
     "folder_to_class": {"glioma_tumor": "glioma", "meningioma_tumor": "meningioma",
                          "no_tumor": "notumor", "pituitary_tumor": "pituitary"}},
    {"name": "mendeley5", "root": Path("/kaggle/input/datasets/neyamul/bt-mendeley-5/Brain Tumor data/Testing"),
     "folder_to_class": {"glioma": "glioma", "meningioma": "meningioma",
                          "notumor": "notumor", "pituitary": "pituitary"}},
    {"name": "mendeley_epic_cscr", "root": Path("/kaggle/input/datasets/neyamul/mendeley-bt-data/Epic and CSCR hospital Dataset/Test"),
     "folder_to_class": {"glioma": "glioma", "meningioma": "meningioma",
                          "notumor": "notumor", "pituitary": "pituitary"}},
    {"name": "mendeley2", "root": Path("/kaggle/input/datasets/neyamul/mendeley-2/test"),
     "folder_to_class": {"glioma": "glioma", "meningioma": "meningioma",
                          "no_tumor": "notumor", "pituitary": "pituitary"}},
]

_known_splits = {"train": full_result["train_samples"], "val": full_result["val_samples"],
                  "test": full_result["test_samples"]}

external_results = {}
external_counts = {}
for ds in EXTERNAL_DATASETS:
    print(f"\n{'='*70}\n[{ds['name']}]  root={ds['root']}\n{'='*70}")
    external_counts[ds["name"]] = len(scan_dataset_generic(ds["root"], ds["folder_to_class"], ds["name"]))
    external_results[ds["name"]] = run_external_validation(
        full_result["model"], full_result["criterion"], ds["name"],
        ds["root"], ds["folder_to_class"], known_splits=_known_splits)

In [ ]:
ext_summary_rows = []
for name, m in external_results.items():
    ext_summary_rows.append({
        "dataset": name, "n": external_counts[name],
        "accuracy": m["acc"], "balanced_acc": m["balanced_acc"],
        "macro_precision": m["precision_mac"], "macro_recall": m["recall_mac"],
        "f1_macro": m["f1_mac"], "f1_weighted": m["f1_weighted"],
        "auc": m["auc"], "mcc": m["mcc"], "ece": m["ece"],
    })
ext_summary_df = pd.DataFrame(ext_summary_rows)
_internal_test = full_result["test_metrics"]
print(f"\n[EXTERNAL VALIDATION SUMMARY]  Internal Test Performance (BRISC2025 held-out): "
      f"Accuracy={_internal_test['acc']:.4f}  Macro-F1={_internal_test['f1_mac']:.4f}  "
      f"AUC={_internal_test['auc']:.4f}  MCC={_internal_test['mcc']:.4f}")
print("\nExternal Validation Performance (per dataset):")
print(ext_summary_df.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
ext_summary_df.to_csv(OUT_DIR / "external_validation_summary.csv", index=False)

## §21 · XAI (final proposed model only)

Grad-CAM++, Score-CAM, LIME — all three already used elsewhere in the SMGF-Net pipeline, reused here as-is rather than reimplemented, targeted at `full_model.attn` (the MS-CA-ECA block) instead of the old `model.eca`. XAI here is applied **only** to `SMGF-Net + MS-CA-ECA`; there is no per-ablation-variant XAI comparison.

In [ ]:
def denormalize_to_rgb01(img_tensor: torch.Tensor) -> np.ndarray:
    """[3,H,W] normalized tensor -> [H,W,3] float array in [0,1] for display/overlay."""
    mean = torch.tensor(IMAGENET_MEAN, device=img_tensor.device).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD,  device=img_tensor.device).view(3, 1, 1)
    img = (img_tensor.detach().cpu() * std.cpu() + mean.cpu()).clamp(0, 1)
    return img.permute(1, 2, 0).numpy()


def overlay_heatmap(rgb01: np.ndarray, cam01: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    """cam01: 2D array in [0,1], same H,W as rgb01. Returns [H,W,3] uint8 overlay."""
    heat = cv2.applyColorMap((cam01 * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    base = rgb01.astype(np.float32)
    blended = (1 - alpha) * base + alpha * heat
    return (np.clip(blended, 0, 1) * 255).astype(np.uint8)


def load_input_tensor(path: str) -> torch.Tensor:
    """Single image -> [1,3,224,224] tensor on DEVICE, ready for model(...)."""
    return preprocess(path).unsqueeze(0).to(DEVICE)


class GradCAMPlusPlus:
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.activations = None
        self.gradients = None
        self._fwd = target_layer.register_forward_hook(self._save_act)
        self._bwd = target_layer.register_full_backward_hook(self._save_grad)

    def _save_act(self, module, inp, out):
        self.activations = out

    def _save_grad(self, module, grad_in, grad_out):
        self.gradients = grad_out[0]

    def remove(self):
        self._fwd.remove(); self._bwd.remove()

    def generate(self, input_tensor: torch.Tensor, class_idx: Optional[int] = None):
        self.model.eval()
        self.model.zero_grad(set_to_none=True)
        out = self.model(input_tensor)
        logits = out["logits"]
        if class_idx is None:
            class_idx = int(logits.argmax(dim=1).item())
        score = logits[:, class_idx].sum()
        score.backward()

        acts, grads = self.activations, self.gradients
        eps = 1e-8
        g2, g3 = grads ** 2, grads ** 3
        sum_acts = acts.sum(dim=(2, 3), keepdim=True)
        alpha_denom = 2 * g2 + sum_acts * g3
        alpha_denom = torch.where(alpha_denom.abs() > eps, alpha_denom,
                                   torch.full_like(alpha_denom, eps))
        alpha = g2 / alpha_denom
        weights = (alpha * F.relu(grads)).sum(dim=(2, 3), keepdim=True)

        cam = F.relu((weights * acts).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=input_tensor.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + eps)
        return cam, class_idx


class ScoreCAM:
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.activations = None
        self._fwd = target_layer.register_forward_hook(self._save_act)

    def _save_act(self, module, inp, out):
        self.activations = out

    def remove(self):
        self._fwd.remove()

    @torch.no_grad()
    def generate(self, input_tensor: torch.Tensor, class_idx: Optional[int] = None,
                 batch_size: int = 16):
        self.model.eval()
        out0 = self.model(input_tensor)
        if class_idx is None:
            class_idx = int(out0["logits"].argmax(dim=1).item())

        acts = self.activations[0]
        C = acts.shape[0]
        img_hw = input_tensor.shape[-2:]
        acts_up = F.interpolate(acts.unsqueeze(0), size=img_hw,
                                 mode="bilinear", align_corners=False).squeeze(0)

        eps = 1e-8
        c_min = acts_up.amin(dim=(1, 2), keepdim=True)
        c_max = acts_up.amax(dim=(1, 2), keepdim=True)
        masks = (acts_up - c_min) / (c_max - c_min + eps)

        base = input_tensor.expand(1, -1, -1, -1)
        scores = torch.zeros(C, device=input_tensor.device)
        for start in range(0, C, batch_size):
            chunk_masks = masks[start:start + batch_size].unsqueeze(1)
            masked_inputs = base * chunk_masks
            out = self.model(masked_inputs)
            probs = torch.softmax(out["logits"], dim=1)[:, class_idx]
            scores[start:start + chunk_masks.shape[0]] = probs

        weights = torch.softmax(scores, dim=0)
        cam = F.relu((weights.view(C, 1, 1) * acts_up).sum(dim=0))
        cam = cam.cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + eps)
        return cam, class_idx


from skimage.segmentation import slic
from sklearn.linear_model import Ridge

class LimeImageExplainer:
    def __init__(self, model: nn.Module, num_samples: int = 200, num_segments: int = 50,
                 kernel_width: float = 0.25, batch_size: int = 32):
        self.model = model
        self.num_samples = num_samples
        self.num_segments = num_segments
        self.kernel_width = kernel_width
        self.batch_size = batch_size

    @torch.no_grad()
    def explain(self, input_tensor: torch.Tensor, class_idx: Optional[int] = None,
                num_top_segments: int = 8, seed: int = SEED):
        self.model.eval()
        device = input_tensor.device
        mean = torch.tensor(IMAGENET_MEAN, device=device).view(3, 1, 1)
        std  = torch.tensor(IMAGENET_STD,  device=device).view(3, 1, 1)

        img_denorm = (input_tensor[0] * std + mean).clamp(0, 1)
        img_np = img_denorm.permute(1, 2, 0).cpu().numpy()
        segments = slic(img_np, n_segments=self.num_segments, compactness=10, start_label=0)
        n_segs = int(segments.max()) + 1
        segments_t = torch.from_numpy(segments).to(device)

        if class_idx is None:
            out0 = self.model(input_tensor)
            class_idx = int(out0["logits"].argmax(dim=1).item())

        baseline = img_denorm.mean(dim=(1, 2), keepdim=True).expand_as(img_denorm)

        rng = np.random.default_rng(seed)
        Z = rng.integers(0, 2, size=(self.num_samples, n_segs)).astype(np.float32)
        Z[0, :] = 1.0

        preds = np.zeros(self.num_samples, dtype=np.float32)
        for start in range(0, self.num_samples, self.batch_size):
            chunk = Z[start:start + self.batch_size]
            batch_imgs = []
            for row in chunk:
                on_segments = torch.tensor(np.where(row == 1)[0], device=device)
                mask = torch.isin(segments_t, on_segments).float().unsqueeze(0)
                perturbed = img_denorm * mask + baseline * (1 - mask)
                batch_imgs.append((perturbed - mean) / std)
            batch_imgs = torch.stack(batch_imgs, dim=0)
            out = self.model(batch_imgs)
            probs = torch.softmax(out["logits"], dim=1)[:, class_idx]
            preds[start:start + chunk.shape[0]] = probs.cpu().numpy()

        distances = np.sqrt(((1 - Z) ** 2).sum(axis=1)) / np.sqrt(n_segs)
        weights = np.exp(-(distances ** 2) / (self.kernel_width ** 2))

        surrogate = Ridge(alpha=1.0)
        surrogate.fit(Z, preds, sample_weight=weights)
        importances = surrogate.coef_

        top_segments = np.argsort(np.abs(importances))[::-1][:num_top_segments]
        return {"segments": segments, "importances": importances,
                "top_segments": top_segments, "class_idx": class_idx}

    @staticmethod
    def visualize(rgb01: np.ndarray, explanation: Dict[str, Any], positive_only: bool = True) -> np.ndarray:
        segments, importances = explanation["segments"], explanation["importances"]
        top = explanation["top_segments"]
        mask = np.zeros(segments.shape, dtype=bool)
        overlay = (rgb01.copy() * 255).astype(np.uint8)
        for seg_id in top:
            if positive_only and importances[seg_id] <= 0:
                continue
            seg_mask = segments == seg_id
            mask |= seg_mask
            boundary = cv2.dilate(seg_mask.astype(np.uint8), np.ones((3, 3), np.uint8)) - seg_mask.astype(np.uint8)
            overlay[boundary.astype(bool)] = [255, 255, 0]
        dim = overlay.copy()
        dim[~mask] = (dim[~mask] * 0.35).astype(np.uint8)
        return dim


def explain_sample(model: nn.Module, image_path: str, true_label: Optional[str] = None,
                    save_path: Optional[str] = None):
    """Grad-CAM++ / Score-CAM / LIME, all targeted at model.attn (MS-CA-ECA) --
    the attribution/localization terminology below deliberately avoids any
    claim of clinical correctness."""
    input_tensor = load_input_tensor(image_path)
    rgb01 = denormalize_to_rgb01(input_tensor[0])

    gcpp = GradCAMPlusPlus(model, model.attn)
    cam_gcpp, pred_idx = gcpp.generate(input_tensor)
    gcpp.remove()

    scam = ScoreCAM(model, model.attn)
    cam_score, _ = scam.generate(input_tensor, class_idx=pred_idx)
    scam.remove()

    lime_exp = LimeImageExplainer(model).explain(input_tensor, class_idx=pred_idx)

    pred_name = CLASS_NAMES[pred_idx]
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    axes[0].imshow(rgb01); axes[0].set_title(f"Input\n(true: {true_label or '?'})")
    axes[1].imshow(overlay_heatmap(rgb01, cam_gcpp)); axes[1].set_title(f"Grad-CAM++\n(pred: {pred_name})")
    axes[2].imshow(overlay_heatmap(rgb01, cam_score)); axes[2].set_title("Score-CAM")
    axes[3].imshow(LimeImageExplainer.visualize(rgb01, lime_exp)); axes[3].set_title("LIME")
    for ax in axes: ax.axis("off")
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return {"pred_idx": pred_idx, "pred_name": pred_name,
            "gradcampp": cam_gcpp, "scorecam": cam_score, "lime": lime_exp}

print("XAI utilities ready (Grad-CAM++, Score-CAM, LIME), targeted at MS-CA-ECA.")

### Representative cases, including difficult glioma↔meningioma pairs and high-confidence failures

For each class, a representative correctly-classified case, plus glioma→meningioma and meningioma→glioma misclassifications specifically, plus the highest-confidence incorrect predictions overall.

In [ ]:
@torch.no_grad()
def collect_test_predictions(model: nn.Module, samples: List[Dict], batch_size: int = BATCH_SIZE) -> pd.DataFrame:
    """Runs inference over `samples` and returns one row per image with the true/pred
    class, full softmax vector, and top-1 confidence."""
    ds = BrainTumorDataset(samples, is_train=False)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    model.eval()
    rows = []
    ptr = 0
    for batch in loader:
        img = batch["image"].to(DEVICE, non_blocking=True)
        out = model(img)
        probs = torch.softmax(out["logits"], dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)
        for i in range(len(preds)):
            s = samples[ptr]
            rows.append({
                "path": s["path"], "true_class": s["class_name"], "true_idx": s["label"],
                "pred_idx": int(preds[i]), "pred_class": CLASS_NAMES[int(preds[i])],
                "confidence": float(probs[i, preds[i]]),
                **{f"prob_{c}": float(probs[i, j]) for j, c in enumerate(CLASS_NAMES)},
            })
            ptr += 1
    df = pd.DataFrame(rows)
    df["correct"] = df["true_idx"] == df["pred_idx"]
    return df


test_pred_df = collect_test_predictions(full_model, full_result["test_samples"])
print(f"[test predictions] n={len(test_pred_df)}  accuracy={test_pred_df['correct'].mean():.4f}")

xai_cases = []
for cls in CLASS_NAMES:
    correct_rows = test_pred_df[(test_pred_df["true_class"] == cls) & test_pred_df["correct"]]
    if len(correct_rows):
        row = correct_rows.sort_values("confidence", ascending=False).iloc[0]
        xai_cases.append(("representative_" + cls, row))

g_to_m = test_pred_df[(test_pred_df["true_class"] == "glioma") & (test_pred_df["pred_class"] == "meningioma")]
m_to_g = test_pred_df[(test_pred_df["true_class"] == "meningioma") & (test_pred_df["pred_class"] == "glioma")]
if len(g_to_m):
    xai_cases.append(("glioma_to_meningioma", g_to_m.sort_values("confidence", ascending=False).iloc[0]))
if len(m_to_g):
    xai_cases.append(("meningioma_to_glioma", m_to_g.sort_values("confidence", ascending=False).iloc[0]))

high_conf_wrong = test_pred_df[~test_pred_df["correct"]].sort_values("confidence", ascending=False).head(2)
for i, (_, row) in enumerate(high_conf_wrong.iterrows()):
    xai_cases.append((f"high_confidence_incorrect_{i}", row))

print(f"\n[XAI case list] {len(xai_cases)} cases: {[name for name, _ in xai_cases]}")

xai_explanations = {}
for name, row in xai_cases:
    print(f"\n--- {name}: true={row['true_class']}  pred={row['pred_class']}  conf={row['confidence']:.3f} ---")
    xai_explanations[name] = explain_sample(
        full_model, row["path"], true_label=row["true_class"],
        save_path=str(OUT_DIR / f"xai_{name}.png"))

## §22 · XAI Quantitative Analysis (final proposed model only)

Deletion AUC, Insertion AUC, cross-method agreement (IoU + Spearman ρ between Grad-CAM++/Score-CAM/LIME), and a cascading-randomization sanity check — the metric definitions are unchanged from the SMGF-Net notebook, applied here only to `SMGF-Net + MS-CA-ECA`, with `model.eca` replaced by `model.attn` (MS-CA-ECA) as the target layer throughout. Terminology stays at attribution / localization / faithfulness / stability — no clinical-correctness claims.

In [ ]:
def _gaussian_blur_baseline(img_denorm: torch.Tensor, ksize: int = 31) -> torch.Tensor:
    """Blurred version of the image, used as the 'nothing here' baseline for
    deletion/insertion -- less of an artificial edge artifact than a flat/zero fill."""
    img_np = (img_denorm.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    blurred = cv2.GaussianBlur(img_np, (ksize, ksize), 0)
    return torch.from_numpy(blurred).float().permute(2, 0, 1) / 255.0


@torch.no_grad()
def deletion_insertion_pixelwise(model: nn.Module, input_tensor: torch.Tensor, cam01: np.ndarray,
                                  class_idx: int, n_steps: int = 20) -> Dict[str, Any]:
    """For a continuous per-pixel heatmap (Grad-CAM++ or Score-CAM). Ranks pixels by
    cam value, then reveals/removes them n_steps at a time. Returns probability
    curves and their AUC (trapezoidal, x-axis = fraction of pixels processed)."""
    device = input_tensor.device
    mean = torch.tensor(IMAGENET_MEAN, device=device).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD,  device=device).view(3, 1, 1)
    img_denorm = (input_tensor[0] * std + mean).clamp(0, 1)
    baseline = _gaussian_blur_baseline(img_denorm).to(device)

    H, W = cam01.shape
    order = np.argsort(cam01.flatten())[::-1]
    n_pixels = H * W
    step_size = max(1, n_pixels // n_steps)
    steps = [min(i * step_size, n_pixels) for i in range(n_steps + 1)]

    del_imgs = []
    flat_mask = np.zeros(n_pixels, dtype=bool)
    for k in steps:
        flat_mask[:] = False
        flat_mask[order[:k]] = True
        m = torch.from_numpy(flat_mask.reshape(H, W)).to(device).float().unsqueeze(0)
        perturbed = baseline * m + img_denorm * (1 - m)
        del_imgs.append((perturbed - mean) / std)
    del_batch = torch.stack(del_imgs, dim=0)
    del_out = model(del_batch)
    del_probs = torch.softmax(del_out["logits"], dim=1)[:, class_idx].cpu().numpy()

    ins_imgs = []
    for k in steps:
        flat_mask[:] = False
        flat_mask[order[:k]] = True
        m = torch.from_numpy(flat_mask.reshape(H, W)).to(device).float().unsqueeze(0)
        perturbed = img_denorm * m + baseline * (1 - m)
        ins_imgs.append((perturbed - mean) / std)
    ins_batch = torch.stack(ins_imgs, dim=0)
    ins_out = model(ins_batch)
    ins_probs = torch.softmax(ins_out["logits"], dim=1)[:, class_idx].cpu().numpy()

    x = np.array(steps) / n_pixels
    del_auc = float(np.trapz(del_probs, x))
    ins_auc = float(np.trapz(ins_probs, x))
    return {"x": x, "deletion_probs": del_probs, "insertion_probs": ins_probs,
            "deletion_auc": del_auc, "insertion_auc": ins_auc}


@torch.no_grad()
def deletion_insertion_segments(model: nn.Module, input_tensor: torch.Tensor,
                                 explanation: Dict[str, Any], class_idx: int) -> Dict[str, Any]:
    """Same protocol for LIME's segment-based explanation -- ranks superpixels by
    |importance| and reveals/removes them one at a time (x-axis = fraction of area)."""
    device = input_tensor.device
    mean = torch.tensor(IMAGENET_MEAN, device=device).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD,  device=device).view(3, 1, 1)
    img_denorm = (input_tensor[0] * std + mean).clamp(0, 1)
    baseline = _gaussian_blur_baseline(img_denorm).to(device)

    segments, importances = explanation["segments"], explanation["importances"]
    ranked = np.argsort(np.abs(importances))[::-1]
    total_area = segments.size

    del_imgs, ins_imgs, areas = [], [], []
    for n_segs_used in range(0, len(ranked) + 1):
        used = ranked[:n_segs_used]
        seg_mask = np.isin(segments, used)
        areas.append(seg_mask.sum() / total_area)
        m = torch.from_numpy(seg_mask).to(device).float().unsqueeze(0)
        del_perturbed = baseline * m + img_denorm * (1 - m)
        ins_perturbed = img_denorm * m + baseline * (1 - m)
        del_imgs.append((del_perturbed - mean) / std)
        ins_imgs.append((ins_perturbed - mean) / std)

    del_out = model(torch.stack(del_imgs, dim=0))
    ins_out = model(torch.stack(ins_imgs, dim=0))
    del_probs = torch.softmax(del_out["logits"], dim=1)[:, class_idx].cpu().numpy()
    ins_probs = torch.softmax(ins_out["logits"], dim=1)[:, class_idx].cpu().numpy()

    x = np.array(areas)
    del_auc = float(np.trapz(del_probs, x))
    ins_auc = float(np.trapz(ins_probs, x))
    return {"x": x, "deletion_probs": del_probs, "insertion_probs": ins_probs,
            "deletion_auc": del_auc, "insertion_auc": ins_auc}


def run_xai_batch_validation(model: nn.Module, samples: List[Dict], n_per_class: int = 5,
                              seed: int = SEED, save_grid_path: Optional[str] = None) -> Dict[str, Any]:
    """Picks n_per_class samples per class, runs Grad-CAM++/Score-CAM/LIME on each
    (targeted at model.attn), scores faithfulness (deletion/insertion AUC), and saves
    a comparison grid. Returns per-method aggregate stats (mean +/- std)."""
    rng = random.Random(seed)
    by_class: Dict[str, List[Dict]] = {c: [] for c in CLASS_NAMES}
    for s in samples:
        by_class[s["class_name"]].append(s)
    chosen = []
    for c, lst in by_class.items():
        chosen.extend(rng.sample(lst, min(n_per_class, len(lst))))

    rows = []
    grid_imgs = []

    for s in chosen:
        input_tensor = load_input_tensor(s["path"])
        rgb01 = denormalize_to_rgb01(input_tensor[0])

        gcpp = GradCAMPlusPlus(model, model.attn)
        cam_gcpp, pred_idx = gcpp.generate(input_tensor)
        gcpp.remove()
        di_gcpp = deletion_insertion_pixelwise(model, input_tensor, cam_gcpp, pred_idx)

        scam = ScoreCAM(model, model.attn)
        cam_score, _ = scam.generate(input_tensor, class_idx=pred_idx)
        scam.remove()
        di_score = deletion_insertion_pixelwise(model, input_tensor, cam_score, pred_idx)

        lime_exp = LimeImageExplainer(model).explain(input_tensor, class_idx=pred_idx)
        di_lime = deletion_insertion_segments(model, input_tensor, lime_exp, pred_idx)

        correct = (pred_idx == CLASS_TO_IDX[s["class_name"]])
        rows.append({
            "path": s["path"], "true_class": s["class_name"], "pred_class": CLASS_NAMES[pred_idx],
            "correct": correct,
            "gradcampp_del_auc": di_gcpp["deletion_auc"], "gradcampp_ins_auc": di_gcpp["insertion_auc"],
            "scorecam_del_auc": di_score["deletion_auc"], "scorecam_ins_auc": di_score["insertion_auc"],
            "lime_del_auc": di_lime["deletion_auc"], "lime_ins_auc": di_lime["insertion_auc"],
        })
        grid_imgs.append((rgb01, overlay_heatmap(rgb01, cam_gcpp), overlay_heatmap(rgb01, cam_score),
                           LimeImageExplainer.visualize(rgb01, lime_exp),
                           f"{s['class_name']}\n(pred: {CLASS_NAMES[pred_idx]})"))

    df = pd.DataFrame(rows)
    summary = {}
    for method in ["gradcampp", "scorecam", "lime"]:
        summary[method] = {
            "deletion_auc_mean": df[f"{method}_del_auc"].mean(), "deletion_auc_std": df[f"{method}_del_auc"].std(),
            "insertion_auc_mean": df[f"{method}_ins_auc"].mean(), "insertion_auc_std": df[f"{method}_ins_auc"].std(),
        }
    print("\n[XAI FAITHFULNESS SUMMARY]  (lower deletion-AUC = better, higher insertion-AUC = better)")
    for method, m in summary.items():
        print(f"  {method:10s}  deletion={m['deletion_auc_mean']:.4f}\u00b1{m['deletion_auc_std']:.4f}  "
              f"insertion={m['insertion_auc_mean']:.4f}\u00b1{m['insertion_auc_std']:.4f}")

    n = len(grid_imgs)
    fig, axes = plt.subplots(n, 4, figsize=(14, 3.2 * n))
    if n == 1:
        axes = axes.reshape(1, -1)
    for row, (rgb01, gcpp_img, score_img, lime_img, title) in enumerate(grid_imgs):
        axes[row, 0].imshow(rgb01); axes[row, 0].set_ylabel(title, fontsize=9, rotation=0,
                                                              labelpad=45, va="center")
        axes[row, 1].imshow(gcpp_img); axes[row, 2].imshow(score_img); axes[row, 3].imshow(lime_img)
        for col in range(4): axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
    for col, name in enumerate(["Input", "Grad-CAM++", "Score-CAM", "LIME"]):
        axes[0, col].set_title(name)
    plt.tight_layout()
    if save_grid_path: plt.savefig(save_grid_path, dpi=150, bbox_inches="tight")
    plt.show()

    return {"per_sample": df, "summary": summary}


xai_batch_results = run_xai_batch_validation(
    full_result["model"], full_result["test_samples"], n_per_class=3,
    save_grid_path=str(OUT_DIR / "xai_batch_grid.png"))
xai_batch_results["per_sample"].to_csv(OUT_DIR / "xai_faithfulness_per_sample.csv", index=False)

In [ ]:
def compute_method_agreement(cam_a: np.ndarray, cam_b: np.ndarray, top_percent: float = 0.15) -> Dict[str, float]:
    """Agreement between two continuous heatmaps of the same image: IoU of their
    top-k% most-important regions, plus Spearman rank correlation over all pixels."""
    assert cam_a.shape == cam_b.shape
    n = cam_a.size
    k = max(1, int(n * top_percent))

    thresh_a = np.partition(cam_a.flatten(), -k)[-k]
    thresh_b = np.partition(cam_b.flatten(), -k)[-k]
    mask_a = cam_a >= thresh_a
    mask_b = cam_b >= thresh_b
    intersection = np.logical_and(mask_a, mask_b).sum()
    union = np.logical_or(mask_a, mask_b).sum()
    iou = float(intersection / union) if union > 0 else 0.0

    rho, _ = spearmanr(cam_a.flatten(), cam_b.flatten())
    return {"iou_top_k": iou, "spearman_rho": float(rho)}


def lime_segments_to_dense_map(explanation: Dict[str, Any]) -> np.ndarray:
    """Converts LIME's per-segment importances into a dense [H,W] map so it can be
    compared against Grad-CAM++/Score-CAM with the same IoU/correlation function."""
    segments, importances = explanation["segments"], explanation["importances"]
    dense = np.zeros(segments.shape, dtype=np.float32)
    for seg_id, imp in enumerate(importances):
        dense[segments == seg_id] = imp
    dense = np.clip(dense, 0, None)
    if dense.max() > 0:
        dense = dense / dense.max()
    return dense


def run_agreement_analysis(model: nn.Module, samples: List[Dict], n_samples: int = 20,
                            seed: int = SEED) -> pd.DataFrame:
    rng = random.Random(seed)
    chosen = rng.sample(samples, min(n_samples, len(samples)))
    rows = []
    for s in chosen:
        input_tensor = load_input_tensor(s["path"])

        gcpp = GradCAMPlusPlus(model, model.attn)
        cam_gcpp, pred_idx = gcpp.generate(input_tensor)
        gcpp.remove()

        scam = ScoreCAM(model, model.attn)
        cam_score, _ = scam.generate(input_tensor, class_idx=pred_idx)
        scam.remove()

        lime_exp = LimeImageExplainer(model).explain(input_tensor, class_idx=pred_idx)
        cam_lime = lime_segments_to_dense_map(lime_exp)

        gs = compute_method_agreement(cam_gcpp, cam_score)
        gl = compute_method_agreement(cam_gcpp, cam_lime)
        sl = compute_method_agreement(cam_score, cam_lime)
        rows.append({
            "path": s["path"], "true_class": s["class_name"], "pred_class": CLASS_NAMES[pred_idx],
            "gradcampp_vs_scorecam_iou": gs["iou_top_k"], "gradcampp_vs_scorecam_rho": gs["spearman_rho"],
            "gradcampp_vs_lime_iou": gl["iou_top_k"], "gradcampp_vs_lime_rho": gl["spearman_rho"],
            "scorecam_vs_lime_iou": sl["iou_top_k"], "scorecam_vs_lime_rho": sl["spearman_rho"],
        })
    df = pd.DataFrame(rows)
    print("\n[CROSS-METHOD AGREEMENT]  (mean over {} samples)".format(len(df)))
    for pair in ["gradcampp_vs_scorecam", "gradcampp_vs_lime", "scorecam_vs_lime"]:
        print(f"  {pair:24s}  IoU={df[f'{pair}_iou'].mean():.3f}  "
              f"Spearman \u03c1={df[f'{pair}_rho'].mean():.3f}")
    return df


# output-side -> input-side order, using SMGFNetMSCAECA's own submodule names
CASCADE_BLOCKS = ["head", "attn", "fusion", "morphology", "multiscale", "backbone"]

def _randomize_block_(module: nn.Module, generator: Optional[torch.Generator] = None):
    """Reinitializes a block's weights with random noise matched to each parameter's
    existing scale, for a genuine (not just magnitude-changing) randomization."""
    with torch.no_grad():
        for p in module.parameters():
            std = p.data.std().clamp(min=1e-3)
            noise = torch.randn(p.shape, generator=generator, device="cpu").to(p.device) \
                if generator is not None else torch.randn(p.shape, device=p.device)
            p.data.copy_(noise * std)


def run_randomization_sanity_check(model: nn.Module, sample_path: str) -> Dict[str, Any]:
    """Cascading randomization test (Adebayo et al., 2018): progressively randomize
    the model's weights output-to-input, recomputing Grad-CAM++ each time. A real,
    weight-dependent explanation should degrade as more of the trained model is
    destroyed; if agreement stays high, Grad-CAM++ is acting like an edge detector
    on this input rather than explaining the trained model."""
    import copy
    input_tensor = load_input_tensor(sample_path)

    model_copy = copy.deepcopy(model)
    gcpp = GradCAMPlusPlus(model_copy, model_copy.attn)
    cam_original, class_idx = gcpp.generate(input_tensor)
    gcpp.remove()

    generator = torch.Generator(device="cpu").manual_seed(SEED)
    results = [{"stage": "original (trained weights)", "spearman_rho": 1.0, "iou_top_k": 1.0}]
    print("[SANITY CHECK] cascading randomization vs. Grad-CAM++ agreement:")
    for block_name in CASCADE_BLOCKS:
        _randomize_block_(getattr(model_copy, block_name), generator)
        gcpp = GradCAMPlusPlus(model_copy, model_copy.attn)
        cam_rand, _ = gcpp.generate(input_tensor, class_idx=class_idx)
        gcpp.remove()
        agreement = compute_method_agreement(cam_original, cam_rand)
        results.append({"stage": f"+ {block_name} randomized", **agreement})
        print(f"  after randomizing '{block_name}':  IoU={agreement['iou_top_k']:.3f}  "
              f"Spearman \u03c1={agreement['spearman_rho']:.3f}")

    del model_copy
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return {"class_idx": class_idx, "cascade": results}


def plot_randomization_cascade(model: nn.Module, sample_path: str, save_path: Optional[str] = None):
    """Visual companion to run_randomization_sanity_check: Grad-CAM++ heatmap at
    each cascade stage, side by side."""
    import copy
    input_tensor = load_input_tensor(sample_path)
    rgb01 = denormalize_to_rgb01(input_tensor[0])

    model_copy = copy.deepcopy(model)
    stage_names = ["original"] + CASCADE_BLOCKS
    cams = []
    generator = torch.Generator(device="cpu").manual_seed(SEED)
    gcpp = GradCAMPlusPlus(model_copy, model_copy.attn)
    cam, class_idx = gcpp.generate(input_tensor)
    gcpp.remove()
    cams.append(cam)
    for block_name in CASCADE_BLOCKS:
        _randomize_block_(getattr(model_copy, block_name), generator)
        gcpp = GradCAMPlusPlus(model_copy, model_copy.attn)
        cam, _ = gcpp.generate(input_tensor, class_idx=class_idx)
        gcpp.remove()
        cams.append(cam)

    fig, axes = plt.subplots(1, len(cams), figsize=(4 * len(cams), 4))
    for ax, cam, name in zip(axes, cams, stage_names):
        ax.imshow(overlay_heatmap(rgb01, cam)); ax.set_title(name, fontsize=9); ax.axis("off")
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    del model_copy
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


agreement_df = run_agreement_analysis(full_result["model"], full_result["test_samples"], n_samples=15)
agreement_df.to_csv(OUT_DIR / "xai_agreement.csv", index=False)

sanity = run_randomization_sanity_check(full_result["model"], full_result["test_samples"][0]["path"])
plot_randomization_cascade(full_result["model"], full_result["test_samples"][0]["path"],
                            save_path=str(OUT_DIR / "gradcampp_randomization_cascade.png"))

## §16 · Comprehensive Error Analysis
False positives / false negatives per class, high-confidence failures, and low-confidence correct predictions, with image galleries and CSV exports.

In [ ]:
@torch.no_grad()
def collect_test_predictions(model: nn.Module, samples: List[Dict], batch_size: int = BATCH_SIZE) -> pd.DataFrame:
    """Runs inference over `samples` and returns one row per image with the true/pred
    class, full softmax vector, and top-1 confidence -- the raw material for every
    error-analysis view below."""
    ds = BrainTumorDataset(samples, is_train=False)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    model.eval()
    rows = []
    ptr = 0
    for batch in loader:
        img = batch["image"].to(DEVICE, non_blocking=True)
        out = model(img)
        probs = torch.softmax(out["logits"], dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)
        for i in range(len(preds)):
            s = samples[ptr]
            rows.append({
                "path": s["path"], "true_class": s["class_name"], "true_idx": s["label"],
                "pred_idx": int(preds[i]), "pred_class": CLASS_NAMES[int(preds[i])],
                "confidence": float(probs[i, preds[i]]),
                **{f"prob_{c}": float(probs[i, j]) for j, c in enumerate(CLASS_NAMES)},
            })
            ptr += 1
    df = pd.DataFrame(rows)
    df["correct"] = df["true_idx"] == df["pred_idx"]
    return df


def error_analysis_report(pred_df: pd.DataFrame, top_k: int = 8,
                           high_conf_thresh: float = 0.9, low_conf_thresh: float = 0.6) -> Dict[str, pd.DataFrame]:
    """Slices predictions into the four views that matter for a qualitative error
    analysis: per-class FP/FN counts, high-confidence failures (the model was
    confidently wrong -- most concerning), and low-confidence correct predictions
    (the model was right but unsure -- calibration-relevant)."""
    wrong = pred_df[~pred_df["correct"]].copy()
    correct = pred_df[pred_df["correct"]].copy()

    fp_fn_rows = []
    for cls in CLASS_NAMES:
        fp = wrong[wrong["pred_class"] == cls]
        fn = wrong[wrong["true_class"] == cls]
        fp_fn_rows.append({"class": cls, "false_positives": len(fp), "false_negatives": len(fn)})
    fp_fn_df = pd.DataFrame(fp_fn_rows)

    high_conf_failures = wrong[wrong["confidence"] >= high_conf_thresh].sort_values(
        "confidence", ascending=False).head(top_k)
    low_conf_correct = correct[correct["confidence"] <= low_conf_thresh].sort_values(
        "confidence", ascending=True).head(top_k)

    print("[ERROR ANALYSIS SUMMARY]")
    print(f"  Total={len(pred_df)}  Correct={len(correct)}  Wrong={len(wrong)}  "
          f"Error rate={len(wrong)/len(pred_df):.4f}")
    print(f"  High-confidence failures (conf>={high_conf_thresh}): {len(wrong[wrong['confidence']>=high_conf_thresh])}")
    print(f"  Low-confidence correct predictions (conf<={low_conf_thresh}): "
          f"{len(correct[correct['confidence']<=low_conf_thresh])}")
    print("\nPer-class FP/FN:")
    print(fp_fn_df.to_string(index=False))

    return {"fp_fn": fp_fn_df, "high_conf_failures": high_conf_failures,
            "low_conf_correct": low_conf_correct, "all_wrong": wrong}


def plot_error_gallery(rows_df: pd.DataFrame, title: str, save_path: Optional[str] = None, max_cols: int = 4):
    """Image grid for a slice of predictions (e.g. high-confidence failures), each
    tile captioned with true/pred/confidence so failure modes are visible at a glance."""
    n = len(rows_df)
    if n == 0:
        print(f"[{title}] no samples to show.")
        return
    ncols = min(max_cols, n)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.2 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, (_, row) in zip(axes, rows_df.iterrows()):
        img_tensor = preprocess(row["path"])
        rgb01 = denormalize_to_rgb01(img_tensor)
        ax.imshow(rgb01)
        ax.set_title(f"true={row['true_class']}\npred={row['pred_class']} ({row['confidence']:.2f})",
                     fontsize=8, color=("green" if row["true_class"] == row["pred_class"] else "crimson"))
        ax.axis("off")
    for ax in axes[len(rows_df):]:
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

print("Error-analysis utilities ready.")


In [ ]:
test_pred_df = collect_test_predictions(full_result["model"], full_result["test_samples"])
test_pred_df.to_csv(OUT_DIR / "test_predictions_full.csv", index=False)

error_slices = error_analysis_report(test_pred_df, top_k=8,
                                      high_conf_thresh=0.9, low_conf_thresh=0.6)
error_slices["fp_fn"].to_csv(OUT_DIR / "error_analysis_fp_fn.csv", index=False)
error_slices["high_conf_failures"].to_csv(OUT_DIR / "error_analysis_high_conf_failures.csv", index=False)
error_slices["low_conf_correct"].to_csv(OUT_DIR / "error_analysis_low_conf_correct.csv", index=False)

plot_error_gallery(error_slices["high_conf_failures"], "High-Confidence Failures (model confidently wrong)",
                    save_path=str(OUT_DIR / "error_gallery_high_conf_failures.png"))
plot_error_gallery(error_slices["low_conf_correct"], "Low-Confidence Correct Predictions",
                    save_path=str(OUT_DIR / "error_gallery_low_conf_correct.png"))


## §17 · Robustness Evaluation
Accuracy under brightness, contrast, Gaussian noise, blur, rotation, JPEG compression, and occlusion corruptions across severity levels.

In [ ]:
def apply_perturbation(img01: np.ndarray, kind: str, severity: float) -> np.ndarray:
    """All perturbations operate on the robust-normalized [0,1] grayscale image,
    i.e. *before* AMEF -- so AMEF still runs on the corrupted input, exactly as it
    would on a real degraded scan at inference time."""
    img = img01.copy()
    if kind == "brightness":
        img = np.clip(img + severity, 0, 1)
    elif kind == "contrast":
        img = np.clip((img - 0.5) * severity + 0.5, 0, 1)
    elif kind == "gaussian_noise":
        img = np.clip(img + np.random.normal(0, severity, img.shape).astype(np.float32), 0, 1)
    elif kind == "blur":
        k = max(1, int(round(severity)) | 1)   # force odd kernel size
        img = cv2.GaussianBlur(img, (k, k), 0)
    elif kind == "rotation":
        h, w = img.shape
        M = cv2.getRotationMatrix2D((w / 2, h / 2), severity, 1.0)
        img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    elif kind == "compression":
        img_u8 = (img * 255).astype(np.uint8)
        ok, enc = cv2.imencode(".jpg", img_u8, [cv2.IMWRITE_JPEG_QUALITY, int(severity)])
        img = cv2.imdecode(enc, cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
    elif kind == "occlusion":
        h, w = img.shape
        patch = int(severity)
        rng = np.random.RandomState(SEED)
        y0 = rng.randint(0, max(1, h - patch + 1)); x0 = rng.randint(0, max(1, w - patch + 1))
        img[y0:y0 + patch, x0:x0 + patch] = 0.0
    else:
        raise ValueError(f"Unknown perturbation kind: {kind}")
    return img


ROBUSTNESS_SEVERITIES = {
    "clean":         [0.0],
    "brightness":    [0.10, 0.20, 0.30, 0.40],
    "contrast":      [0.8, 0.6, 0.4, 0.2],          # multiplier; <1 = lower contrast
    "gaussian_noise":[0.02, 0.05, 0.08, 0.12],
    "blur":          [3, 5, 9, 13],                  # kernel size
    "rotation":      [5, 10, 15, 20],                # degrees
    "compression":   [50, 30, 15, 5],                # JPEG quality (lower = worse)
    "occlusion":     [20, 40, 60, 80],                # square patch side, px (of 224)
}


def preprocess_perturbed(path: str, kind: str, severity: float) -> torch.Tensor:
    gray = read_gray(path)
    gray = cv2.resize(gray, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    img01 = robust_normalize(gray)
    if kind != "clean":
        img01 = apply_perturbation(img01, kind, severity)
    img01 = amef_preprocess(img01)
    return to_3ch_tensor(img01)


class PerturbedDataset(Dataset):
    def __init__(self, samples: List[Dict], kind: str, severity: float):
        self.samples = samples; self.kind = kind; self.severity = severity

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = preprocess_perturbed(s["path"], self.kind, self.severity)
        return {"image": img, "label": torch.tensor(s["label"], dtype=torch.long),
                "class_name": s["class_name"], "path": s["path"]}


@torch.no_grad()
def evaluate_under_perturbation(model: nn.Module, criterion: CAWLPlus, samples: List[Dict],
                                 kind: str, severity: float, batch_size: int = BATCH_SIZE) -> Dict[str, Any]:
    ds = PerturbedDataset(samples, kind, severity)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    return evaluate_v2(model, loader, criterion)


def run_robustness_evaluation(model: nn.Module, criterion: CAWLPlus, samples: List[Dict],
                               severities: Dict[str, List[float]] = ROBUSTNESS_SEVERITIES,
                               n_subsample: Optional[int] = 400, seed: int = SEED) -> pd.DataFrame:
    """Sweeps each corruption type across its severity levels and records accuracy /
    F1 / AUC at each point. Uses a fixed stratified subsample by default so the full
    sweep (7 corruptions x 4 severities) stays fast; set n_subsample=None for the
    full test set."""
    if n_subsample is not None and n_subsample < len(samples):
        rng = random.Random(seed)
        by_class: Dict[str, List[Dict]] = {}
        for s in samples:
            by_class.setdefault(s["class_name"], []).append(s)
        per_class_n = max(1, n_subsample // len(by_class))
        eval_samples = []
        for cls, group in by_class.items():
            eval_samples.extend(rng.sample(group, min(per_class_n, len(group))))
    else:
        eval_samples = samples

    rows = []
    clean_metrics = evaluate_under_perturbation(model, criterion, eval_samples, "clean", 0.0)
    rows.append({"corruption": "clean", "severity_level": 0, "severity_value": 0.0,
                 "acc": clean_metrics["acc"], "f1_macro": clean_metrics["f1_mac"], "auc": clean_metrics["auc"]})
    print(f"[ROBUSTNESS] clean  acc={clean_metrics['acc']:.4f}")

    for kind, sev_list in severities.items():
        if kind == "clean":
            continue
        for level, sev in enumerate(sev_list, start=1):
            m = evaluate_under_perturbation(model, criterion, eval_samples, kind, sev)
            rows.append({"corruption": kind, "severity_level": level, "severity_value": sev,
                         "acc": m["acc"], "f1_macro": m["f1_mac"], "auc": m["auc"]})
            print(f"[ROBUSTNESS] {kind:16s} level={level} (val={sev})  acc={m['acc']:.4f}  "
                  f"f1={m['f1_mac']:.4f}  drop_vs_clean={clean_metrics['acc'] - m['acc']:.4f}")
    return pd.DataFrame(rows)


def plot_robustness_curves(robustness_df: pd.DataFrame, save_path: Optional[str] = None):
    kinds = [k for k in robustness_df["corruption"].unique() if k != "clean"]
    clean_acc = robustness_df[robustness_df["corruption"] == "clean"]["acc"].iloc[0]
    ncols = 4
    nrows = math.ceil(len(kinds) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.2 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, kind in zip(axes, kinds):
        sub = robustness_df[robustness_df["corruption"] == kind].sort_values("severity_level")
        ax.plot([0] + sub["severity_level"].tolist(), [clean_acc] + sub["acc"].tolist(),
                marker="o", color="#4ac9c2")
        ax.axhline(clean_acc, color="gray", linestyle="--", alpha=0.5, linewidth=1)
        ax.set_title(kind); ax.set_xlabel("severity level"); ax.set_ylabel("accuracy")
        ax.set_ylim(0, 1.02); ax.grid(alpha=0.3)
    for ax in axes[len(kinds):]:
        ax.axis("off")
    fig.suptitle("Robustness Evaluation — Accuracy vs. Corruption Severity")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

print("Robustness-evaluation utilities ready.")


In [ ]:
robustness_df = run_robustness_evaluation(full_result["model"], full_result["criterion"],
                                           full_result["test_samples"], n_subsample=400)
robustness_df.to_csv(OUT_DIR / "robustness_evaluation.csv", index=False)
plot_robustness_curves(robustness_df, save_path=str(OUT_DIR / "robustness_curves.png"))


## §18 · Statistical Validation
Bootstrap 95% confidence intervals on the trained model's test predictions, a McNemar-test utility for comparing two models' predictions, and an optional full multi-seed retraining utility for across-run CIs.

In [ ]:
def bootstrap_ci_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray,
                          n_boot: int = 1000, ci: float = 0.95, seed: int = SEED) -> pd.DataFrame:
    """Bootstrap resampling (with replacement) over the test set to get a 95% CI for
    each headline metric -- this is the standard way to attach uncertainty to a
    single trained model's test-set numbers without needing multiple training runs."""
    rng = np.random.RandomState(seed)
    n = len(y_true)
    metric_fns = {
        "accuracy": lambda yt, yp, ypr: accuracy_score(yt, yp),
        "balanced_accuracy": lambda yt, yp, ypr: balanced_accuracy_score(yt, yp),
        "f1_macro": lambda yt, yp, ypr: f1_score(yt, yp, average="macro", zero_division=0),
        "mcc": lambda yt, yp, ypr: matthews_corrcoef(yt, yp),
        "auc_macro_ovr": lambda yt, yp, ypr: (roc_auc_score(yt, ypr, multi_class="ovr", average="macro")
                                               if len(np.unique(yt)) > 1 else float("nan")),
    }
    samples: Dict[str, list] = {k: [] for k in metric_fns}
    for _ in range(n_boot):
        idx = rng.randint(0, n, size=n)
        yt, yp, ypr = y_true[idx], y_pred[idx], y_prob[idx]
        for name, fn in metric_fns.items():
            try:
                samples[name].append(fn(yt, yp, ypr))
            except Exception:
                pass

    alpha = (1 - ci) / 2
    rows = []
    for name, fn in metric_fns.items():
        point = fn(y_true, y_pred, y_prob)
        vals = np.array(samples[name])
        lo, hi = np.percentile(vals, [100 * alpha, 100 * (1 - alpha)])
        rows.append({"metric": name, "point_estimate": point, "ci_low": lo, "ci_high": hi,
                     "std": vals.std()})
    df = pd.DataFrame(rows)
    print(f"[BOOTSTRAP {int(ci*100)}% CI]  n_boot={n_boot}  n_test={n}")
    for _, r in df.iterrows():
        print(f"  {r['metric']:20s} {r['point_estimate']:.4f}  "
              f"[{r['ci_low']:.4f}, {r['ci_high']:.4f}]  (std={r['std']:.4f})")
    return df


def mcnemar_test(y_true: np.ndarray, pred_a: np.ndarray, pred_b: np.ndarray) -> Dict[str, float]:
    """McNemar's test on the paired disagreements between two models' predictions on
    the same test set (correct/incorrect only, ignores which wrong class). Use to
    check whether an accuracy difference between two configurations (e.g. an
    ablation variant vs. the full model) is statistically significant rather than
    noise."""
    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)
    b = int(np.sum(correct_a & ~correct_b))   # a right, b wrong
    c = int(np.sum(~correct_a & correct_b))   # a wrong, b right
    n = b + c
    if n == 0:
        return {"b": b, "c": c, "statistic": 0.0, "p_value": 1.0}
    if n < 25:
        # exact binomial version for small discordant counts
        from scipy.stats import binomtest
        p = binomtest(min(b, c), n, 0.5).pvalue
        stat = float("nan")
    else:
        stat = (abs(b - c) - 1) ** 2 / n   # continuity-corrected
        from scipy.stats import chi2
        p = 1 - chi2.cdf(stat, df=1)
    return {"b": b, "c": c, "statistic": stat, "p_value": float(p)}


def run_multiseed_experiment(seeds: List[int], dataset_name: str, train_root: Path, test_root: Path,
                              folder_to_class: Dict[str, str], total_epochs: int) -> pd.DataFrame:
    """Full retraining across several random seeds (data split + init + augmentation
    all reseeded) to report mean +/- std and a 95% CI across independent runs, rather
    than a single-run point estimate. Expensive -- intended to be run once for the
    final reported numbers, with a small total_epochs override for a quick check."""
    global SEED
    rows = []
    orig_seed = SEED
    for sd in seeds:
        SEED = sd
        random.seed(sd); np.random.seed(sd); torch.manual_seed(sd); torch.cuda.manual_seed_all(sd)
        print(f"\n[MULTI-SEED] seed={sd}")
        result = run_experiment(f"{dataset_name}_seed{sd}", train_root, test_root, folder_to_class,
                                    total_epochs=total_epochs)
        m = result["test_metrics"]
        rows.append({"seed": sd, "accuracy": m["acc"], "balanced_accuracy": m["balanced_acc"],
                     "f1_macro": m["f1_mac"], "auc": m["auc"], "mcc": m["mcc"], "ece": m["ece"]})
        del result
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    SEED = orig_seed
    random.seed(orig_seed); np.random.seed(orig_seed); torch.manual_seed(orig_seed); torch.cuda.manual_seed_all(orig_seed)

    df = pd.DataFrame(rows)
    summary = df.drop(columns="seed").agg(["mean", "std"])
    ci95 = 1.96 * summary.loc["std"] / math.sqrt(len(df))
    print(f"\n[MULTI-SEED SUMMARY over {len(seeds)} seeds]")
    for col in summary.columns:
        print(f"  {col:20s} {summary.loc['mean', col]:.4f} +/- {summary.loc['std', col]:.4f}  "
              f"(95% CI +/- {ci95[col]:.4f})")
    return df

print("Statistical-validation utilities ready.")


In [ ]:
# Bootstrap 95% CIs from the single trained model's test-set predictions (fast, no retraining)
y_true_arr = test_pred_df["true_idx"].to_numpy()
y_pred_arr = test_pred_df["pred_idx"].to_numpy()
y_prob_arr = test_pred_df[[f"prob_{c}" for c in CLASS_NAMES]].to_numpy()

bootstrap_ci_df = bootstrap_ci_metrics(y_true_arr, y_pred_arr, y_prob_arr, n_boot=1000)
bootstrap_ci_df.to_csv(OUT_DIR / "bootstrap_ci_metrics.csv", index=False)

# Full multi-seed retraining for an across-run 95% CI (expensive: each seed
# is a full training pass at total_epochs). Reuses run_multiseed_experiment
# (defined above) and run_experiment under the hood, so it inherits
# the StaticFusion architecture as-is -- nothing else needs to change.
MULTISEED_SEEDS = [13, 42, 77]
multiseed_df = run_multiseed_experiment(
    seeds=MULTISEED_SEEDS, dataset_name="smgfnet_brisc2025",
    train_root=BRISC_TRAIN_ROOT, test_root=BRISC_TEST_ROOT,
    folder_to_class=BRISC_FOLDER_TO_CLASS, total_epochs=TOTAL_EPOCHS)
multiseed_df.to_csv(OUT_DIR / "multiseed_results.csv", index=False)
multiseed_df


## §19 · Efficiency Analysis
FLOPs (thop), parameter count, on-disk model size, per-image latency/FPS, and peak GPU memory.

In [ ]:
import time
from thop import profile as thop_profile

def compute_efficiency_metrics(model: nn.Module, input_size: Tuple[int, int, int, int] = (1, 3, IMG_SIZE, IMG_SIZE),
                                n_warmup: int = 10, n_runs: int = 100, device: str = DEVICE) -> Dict[str, Any]:
    """FLOPs (thop), parameter count, on-disk model size, per-image latency/FPS
    (single-image batch, the deployment-relevant number), and peak memory footprint."""
    model = model.to(device).eval()
    dummy = torch.randn(input_size, device=device)

    # --- FLOPs / params (thop hooks nn.Module forward; the model's dict output is fine) ---
    flops, thop_params = thop_profile(model, inputs=(dummy,), verbose=False)

    total_params = sum(p.numel() for p in model.parameters())
    param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_bytes = sum(b.numel() * b.element_size() for b in model.buffers())
    model_size_mb = (param_bytes + buffer_bytes) / (1024 ** 2)

    # --- latency / FPS ---
    with torch.no_grad():
        for _ in range(n_warmup):
            model(dummy)
        if device == "cuda":
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        for _ in range(n_runs):
            model(dummy)
        if device == "cuda":
            torch.cuda.synchronize()
        t1 = time.perf_counter()

    latency_ms = (t1 - t0) / n_runs * 1000
    fps = 1000.0 / latency_ms
    peak_mem_mb = (torch.cuda.max_memory_allocated() / (1024 ** 2)) if device == "cuda" else float("nan")

    metrics = {
        "params": total_params,
        "params_millions": total_params / 1e6,
        "flops": flops,
        "gflops": flops / 1e9,
        "model_size_mb": model_size_mb,
        "latency_ms_per_image": latency_ms,
        "fps": fps,
        "peak_gpu_mem_mb": peak_mem_mb,
        "device": device,
    }
    print(f"[EFFICIENCY]  Params={total_params:,} ({total_params/1e6:.3f}M)  "
          f"GFLOPs={flops/1e9:.4f}  ModelSize={model_size_mb:.3f}MB  "
          f"Latency={latency_ms:.3f}ms/img  FPS={fps:.1f}  "
          f"PeakGPUMem={peak_mem_mb:.1f}MB" if device == "cuda" else
          f"[EFFICIENCY]  Params={total_params:,} ({total_params/1e6:.3f}M)  "
          f"GFLOPs={flops/1e9:.4f}  ModelSize={model_size_mb:.3f}MB  "
          f"Latency={latency_ms:.3f}ms/img  FPS={fps:.1f}  (CPU — no GPU mem stat)")
    return metrics


def compare_efficiency(models_dict: Dict[str, nn.Module], save_path: Optional[str] = None) -> pd.DataFrame:
    """models_dict: {display_name: model}. Runs compute_efficiency_metrics on each and
    plots a params-vs-accuracy-style efficiency bar comparison (params, GFLOPs, latency)."""
    rows = []
    for name, m in models_dict.items():
        metrics = compute_efficiency_metrics(m)
        rows.append({"model": name, **metrics})
    df = pd.DataFrame(rows)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].bar(df["model"], df["params_millions"], color="#4ac9c2"); axes[0].set_title("Params (M)")
    axes[1].bar(df["model"], df["gflops"], color="#e8664a"); axes[1].set_title("GFLOPs")
    axes[2].bar(df["model"], df["latency_ms_per_image"], color="#8b7fd6"); axes[2].set_title("Latency (ms/img)")
    for ax in axes:
        ax.tick_params(axis="x", rotation=30); ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return df

print("Efficiency-analysis utilities ready.")


In [ ]:
efficiency_metrics = compute_efficiency_metrics(full_result["model"])
efficiency_df = pd.DataFrame([{"model": "SMGF-Net", **efficiency_metrics}])
efficiency_df.to_csv(OUT_DIR / "efficiency_metrics.csv", index=False)
efficiency_df


## §20 · Reliability Diagram & Calibration
Reliability diagrams, ECE, and Brier score for the main test set, plus a cross-dataset calibration comparison against the external Mendeley validation sets.

In [ ]:
def brier_score_multiclass(probs: np.ndarray, labels: np.ndarray, num_classes: int = NUM_CLASSES) -> float:
    """Mean squared error between the predicted probability vector and the one-hot
    true label, averaged over samples -- a proper scoring rule that (unlike
    accuracy) rewards well-calibrated probabilities, not just correct top-1 picks."""
    onehot = np.eye(num_classes)[labels]
    return float(np.mean(np.sum((probs - onehot) ** 2, axis=1)))


def plot_reliability_diagram(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15,
                              title: str = "Reliability Diagram", save_path: Optional[str] = None):
    ece, bin_accs, bin_confs, bin_counts = expected_calibration_error(probs, labels, n_bins=n_bins)
    brier = brier_score_multiclass(probs, labels)
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    ax = axes[0]
    ax.bar(bin_centers, bin_accs, width=1 / n_bins, edgecolor="black", color="#4ac9c2",
           alpha=0.8, label="Accuracy")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy")
    ax.set_title(f"{title}\nECE={ece:.4f}  Brier={brier:.4f}")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = axes[1]
    ax.bar(bin_centers, bin_counts, width=1 / n_bins, color="#e0a94a", edgecolor="black")
    ax.set_xlabel("Confidence"); ax.set_ylabel("Count")
    ax.set_title("Confidence Histogram")
    ax.grid(alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return {"ece": ece, "brier": brier}


def compare_calibration(results_dict: Dict[str, Tuple[np.ndarray, np.ndarray]], n_bins: int = 15,
                         save_path: Optional[str] = None) -> pd.DataFrame:
    """results_dict: {name: (probs, labels)} -- e.g. {'BRISC2025 test': (...), 'mendeley3': (...)}.
    Compares ECE/Brier across datasets on one figure so calibration drift under
    distribution shift is visible directly."""
    rows = []
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
    for name, (probs, labels) in results_dict.items():
        ece, bin_accs, bin_confs, bin_counts = expected_calibration_error(probs, labels, n_bins=n_bins)
        brier = brier_score_multiclass(probs, labels)
        valid = [c > 0 for c in bin_counts]
        xs = [c for c, v in zip(bin_confs, valid) if v]
        ys = [a for a, v in zip(bin_accs, valid) if v]
        ax.plot(xs, ys, marker="o", markersize=3, label=f"{name} (ECE={ece:.3f})")
        rows.append({"dataset": name, "ece": ece, "brier": brier, "n": len(labels)})
    ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy"); ax.legend(fontsize=8)
    ax.set_title("Calibration Comparison Across Datasets"); ax.grid(alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return pd.DataFrame(rows)

print("Calibration / reliability-diagram utilities ready.")


In [ ]:
calib_metrics = plot_reliability_diagram(
    y_prob_arr, y_true_arr, n_bins=15, title="SMGF-Net — BRISC2025 Test",
    save_path=str(OUT_DIR / "reliability_diagram_brisc2025.png"))

# Compare calibration across the external Mendeley datasets too, if those were run (§13a/§13b)
if "external_results" in globals():
    calib_results = {"BRISC2025 test": (y_prob_arr, y_true_arr)}
    for name in external_results:
        # re-derive probs/labels for each external set for a fair, identical-binning comparison
        ext_ds_info = [d for d in EXTERNAL_DATASETS if d["name"] == name][0]
        ext_samples = scan_dataset_generic(ext_ds_info["root"], ext_ds_info["folder_to_class"], name)
        ext_pred_df = collect_test_predictions(full_result["model"], ext_samples)
        calib_results[name] = (ext_pred_df[[f"prob_{c}" for c in CLASS_NAMES]].to_numpy(),
                                ext_pred_df["true_idx"].to_numpy())
    calib_compare_df = compare_calibration(calib_results, save_path=str(OUT_DIR / "calibration_comparison.png"))
    calib_compare_df.to_csv(OUT_DIR / "calibration_comparison.csv", index=False)


## §23 · Feature Visualization (t-SNE / UMAP)
Projects the projection-head embeddings to 2D via t-SNE (and UMAP if `umap-learn` is installed), colored by class with misclassifications marked.

In [ ]:
from sklearn.manifold import TSNE

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("[feature-viz] 'umap-learn' not installed -- run `!pip install -q umap-learn` "
          "to also get the UMAP panel; t-SNE will still work.")


@torch.no_grad()
def extract_embeddings(model: nn.Module, samples: List[Dict], batch_size: int = BATCH_SIZE) -> Dict[str, np.ndarray]:
    """Pulls the projection-head embedding (pre-logit) for every sample -- the same
    representation the CAWL+ prototype-margin term operates on."""
    ds = BrainTumorDataset(samples, is_train=False)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    model.eval()
    embs, labels, preds, paths = [], [], [], []
    ptr = 0
    for batch in loader:
        img = batch["image"].to(DEVICE, non_blocking=True)
        out = model(img)
        if out["emb"] is None:
            raise ValueError("This model has no embedding output (emb=None) -- feature "
                              "visualization needs a model with a projection head.")
        embs.append(out["emb"].cpu().numpy())
        pred = out["logits"].argmax(1).cpu().numpy()
        for i in range(len(pred)):
            s = samples[ptr]; labels.append(s["label"]); paths.append(s["path"]); ptr += 1
        preds.extend(pred.tolist())
    return {"embeddings": np.concatenate(embs, axis=0), "labels": np.array(labels),
            "preds": np.array(preds), "paths": paths}


def plot_feature_embeddings(emb_data: Dict[str, np.ndarray], seed: int = SEED,
                             save_path: Optional[str] = None):
    embeddings, labels, preds = emb_data["embeddings"], emb_data["labels"], emb_data["preds"]
    correct = labels == preds
    colors = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))

    n_panels = 2 if HAS_UMAP else 1
    fig, axes = plt.subplots(1, n_panels, figsize=(6.5 * n_panels, 5.5))
    if n_panels == 1:
        axes = [axes]

    tsne = TSNE(n_components=2, random_state=seed, init="pca", perplexity=min(30, len(embeddings) - 1))
    tsne_xy = tsne.fit_transform(embeddings)
    ax = axes[0]
    for c in range(NUM_CLASSES):
        mask = labels == c
        ax.scatter(tsne_xy[mask & correct, 0], tsne_xy[mask & correct, 1], color=colors[c],
                   s=14, alpha=0.7, label=f"{CLASS_NAMES[c]} (correct)")
        ax.scatter(tsne_xy[mask & ~correct, 0], tsne_xy[mask & ~correct, 1], facecolors="none",
                   edgecolors=colors[c], s=30, linewidths=1.2, marker="x")
    ax.set_title("t-SNE of Projection-Head Embeddings"); ax.legend(fontsize=6, loc="best")

    if HAS_UMAP:
        reducer = umap.UMAP(n_components=2, random_state=seed)
        umap_xy = reducer.fit_transform(embeddings)
        ax = axes[1]
        for c in range(NUM_CLASSES):
            mask = labels == c
            ax.scatter(umap_xy[mask & correct, 0], umap_xy[mask & correct, 1], color=colors[c],
                       s=14, alpha=0.7, label=f"{CLASS_NAMES[c]} (correct)")
            ax.scatter(umap_xy[mask & ~correct, 0], umap_xy[mask & ~correct, 1], facecolors="none",
                       edgecolors=colors[c], s=30, linewidths=1.2, marker="x")
        ax.set_title("UMAP of Projection-Head Embeddings"); ax.legend(fontsize=6, loc="best")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

print("Feature-visualization utilities ready. (x markers = misclassified points)")


In [ ]:
test_emb_data = extract_embeddings(full_result["model"], full_result["test_samples"])
plot_feature_embeddings(test_emb_data, save_path=str(OUT_DIR / "feature_embeddings_tsne_umap.png"))


## §24 · Threshold & Probability Analysis
Per-class confidence-distribution histograms (correct vs. incorrect), a selective-prediction coverage-vs-accuracy curve, and per-class precision/recall vs. decision threshold.

In [ ]:
def plot_probability_distributions(pred_df: pd.DataFrame, save_path: Optional[str] = None):
    """Confidence histograms split by correct/incorrect, per class -- shows whether
    errors cluster at low confidence (well-behaved) or the model is confidently
    wrong for a particular class (a calibration/robustness red flag)."""
    fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(4 * NUM_CLASSES, 4), sharey=True)
    for i, cls in enumerate(CLASS_NAMES):
        sub = pred_df[pred_df["true_class"] == cls]
        ax = axes[i]
        ax.hist(sub[sub["correct"]]["confidence"], bins=20, range=(0, 1), alpha=0.7,
                color="#4ac9c2", label="correct")
        ax.hist(sub[~sub["correct"]]["confidence"], bins=20, range=(0, 1), alpha=0.7,
                color="#e8664a", label="incorrect")
        ax.set_title(cls); ax.set_xlabel("confidence"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
    axes[0].set_ylabel("count")
    fig.suptitle("Prediction Confidence Distributions by True Class")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def plot_coverage_accuracy_curve(pred_df: pd.DataFrame, save_path: Optional[str] = None) -> pd.DataFrame:
    """Selective-prediction curve: sweep a confidence threshold, keep only predictions
    at or above it ('coverage'), and report accuracy on that retained subset. Useful
    for a deployment setting where low-confidence cases get routed to a human."""
    thresholds = np.linspace(0.25, 0.99, 30)
    rows = []
    for t in thresholds:
        kept = pred_df[pred_df["confidence"] >= t]
        coverage = len(kept) / len(pred_df)
        acc = kept["correct"].mean() if len(kept) > 0 else np.nan
        rows.append({"threshold": t, "coverage": coverage, "accuracy": acc})
    df = pd.DataFrame(rows)

    fig, ax1 = plt.subplots(figsize=(7, 5))
    ax1.plot(df["threshold"], df["accuracy"], color="#4ac9c2", marker="o", markersize=3, label="Accuracy (retained)")
    ax1.set_xlabel("Confidence threshold"); ax1.set_ylabel("Accuracy", color="#4ac9c2")
    ax2 = ax1.twinx()
    ax2.plot(df["threshold"], df["coverage"], color="#e8664a", marker="s", markersize=3, label="Coverage")
    ax2.set_ylabel("Coverage (fraction retained)", color="#e8664a")
    ax1.set_title("Selective Prediction: Accuracy vs. Coverage")
    ax1.grid(alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    return df


def plot_per_class_pr_vs_threshold(pred_df: pd.DataFrame, save_path: Optional[str] = None):
    """One-vs-rest precision/recall as a function of decision threshold, per class --
    complements the single-operating-point numbers in the classification report."""
    fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(4 * NUM_CLASSES, 4), sharey=True)
    thresholds = np.linspace(0.05, 0.95, 25)
    for i, cls in enumerate(CLASS_NAMES):
        y_true_bin = (pred_df["true_class"] == cls).to_numpy()
        y_score = pred_df[f"prob_{cls}"].to_numpy()
        precisions, recalls = [], []
        for t in thresholds:
            y_pred_bin = y_score >= t
            tp = np.sum(y_true_bin & y_pred_bin)
            fp = np.sum(~y_true_bin & y_pred_bin)
            fn = np.sum(y_true_bin & ~y_pred_bin)
            precisions.append(tp / (tp + fp) if (tp + fp) > 0 else np.nan)
            recalls.append(tp / (tp + fn) if (tp + fn) > 0 else np.nan)
        ax = axes[i]
        ax.plot(thresholds, precisions, label="Precision", color="#4ac9c2")
        ax.plot(thresholds, recalls, label="Recall", color="#e8664a")
        ax.set_title(cls); ax.set_xlabel("threshold"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
        ax.set_ylim(0, 1.02)
    fig.suptitle("Per-Class Precision/Recall vs. Decision Threshold (one-vs-rest)")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

print("Threshold / probability-analysis utilities ready.")


In [ ]:
plot_probability_distributions(test_pred_df, save_path=str(OUT_DIR / "probability_distributions.png"))
coverage_acc_df = plot_coverage_accuracy_curve(test_pred_df, save_path=str(OUT_DIR / "coverage_accuracy_curve.png"))
coverage_acc_df.to_csv(OUT_DIR / "coverage_accuracy_curve.csv", index=False)
plot_per_class_pr_vs_threshold(test_pred_df, save_path=str(OUT_DIR / "per_class_pr_vs_threshold.png"))


# Multi-Scale Uncertainty Estimation and Explainability-Guided Reliability

### 1. Motivation

Everything above this point (§1–§25) is unchanged: the trained SMGF-Net checkpoint (`full_result["model"]`), its test-set predictions (`test_pred_df`), calibration, robustness, and XAI results are used exactly as already computed. Nothing below retrains the model, alters its architecture, or overwrites any existing result — this section is purely additive and reads from `full_result`, `test_pred_df`, `external_results`, `EXTERNAL_DATASETS`, and the XAI/robustness helper functions already defined earlier in the notebook.

**Why this section exists.** Softmax confidence alone conflates several very different things: a model can be confidently wrong because it has genuinely learned a spurious shortcut, or it can be unconfident on a case that is actually easy. This section asks whether the model, its learned representation, and its explanations agree with each other — and uses that agreement (or disagreement) as a second, independent signal for reliability, on top of confidence.

This section is organized to answer three research questions:

- **RQ1** — Can SMGF-Net quantify uncertainty at the input and prediction levels?
- **RQ2** — Does that uncertainty correlate with prediction errors and with distribution shift (external validation)?
- **RQ3** — Can explanation consistency improve identification of potentially unreliable high-confidence predictions, beyond confidence/uncertainty alone?

**Final proposed uncertainty framework.** The proposed uncertainty framework combines MC-Dropout-based prediction uncertainty with input-level perturbation sensitivity and explainability consistency to identify potentially unreliable predictions:

```
MC-Dropout prediction uncertainty
        +
Input perturbation sensitivity
        +
Explainability consistency
        ↓
Reliability assessment
        ↓
Potentially unreliable predictions
```

Input-level perturbation sensitivity is reported as a descriptive robustness/sensitivity signal, not as pure epistemic uncertainty; MC-Dropout / mutual information is used as the primary model-/prediction-level (epistemic) uncertainty measure throughout. Feature-level representation stability, explored earlier in this analysis, is **not** part of this final framework (see the note in §2/§7 below). Nothing here is a claim of clinical safety or clinical reliability — high uncertainty does not guarantee an incorrect prediction, and low uncertainty does not guarantee a correct one. Wording throughout is deliberately cautious: "potentially unreliable," "higher-risk predictions," "uncertainty-associated errors."

Each subsection below is additive to the existing pipeline and writes its own CSV/PNG outputs into `OUT_DIR`, alongside the files already produced by §1–§25.


In [ ]:
# --- Part 17 safeguard: snapshot the existing results BEFORE adding anything new,
# so the "Summary" subsection at the end can verify nothing above was disturbed.
# This only reads already-computed objects; it does not recompute or alter them.
_baseline_test_metrics = full_result["test_metrics"]
BASELINE_RESULTS_SNAPSHOT = {
    "test_accuracy": _baseline_test_metrics["acc"],
    "test_balanced_accuracy": _baseline_test_metrics["balanced_acc"],
    "test_f1_macro": _baseline_test_metrics["f1_mac"],
    "test_auc": _baseline_test_metrics["auc"],
    "test_mcc": _baseline_test_metrics["mcc"],
    "test_cohen_kappa": _baseline_test_metrics["kappa"],
    "test_ece": _baseline_test_metrics["ece"],
    "test_confusion_matrix": _baseline_test_metrics["cm"].copy(),
    "n_test_predictions_logged": len(test_pred_df),
    "robustness_clean_acc": (robustness_df.loc[robustness_df["corruption"] == "clean", "acc"].iloc[0]
                              if "robustness_df" in globals() else None),
    "calibration_ece": (calib_metrics["ece"] if "calib_metrics" in globals() else None),
    "external_summary": (ext_summary_df.copy() if "ext_summary_df" in globals() else None),
}
print("[SNAPSHOT] Existing (pre-uncertainty-section) results recorded for later verification:")
for k, v in BASELINE_RESULTS_SNAPSHOT.items():
    if isinstance(v, (int, float)):
        print(f"  {k:28s} = {v:.4f}")
    elif isinstance(v, np.ndarray):
        print(f"  {k:28s} = array{v.shape}")
    elif isinstance(v, pd.DataFrame):
        print(f"  {k:28s} = DataFrame({v.shape[0]} rows)")
    else:
        print(f"  {k:28s} = {v}")


### 2. Input-Level Perturbation Sensitivity

Computed below via a clean pass + one pass per mild perturbation (the same forward passes the earlier feature-level exploration reused; see note at the end of this cell).

**Perturbations used.** Rather than inventing a new perturbation set, this reuses the *mildest* (severity-level-1) settings already defined in `ROBUSTNESS_SEVERITIES` (§17) for brightness, contrast, Gaussian noise, blur, and rotation — the same five kinds the task brief recommends, at the gentlest severity already validated as "mild" in the existing robustness sweep. `apply_perturbation` / `preprocess_perturbed` (§17) are reused unchanged; no perturbation logic is duplicated here.

- **Input-Level Sensitivity / Perturbation Uncertainty** (not "aleatoric uncertainty" — that label would overclaim what a handful of synthetic corruptions can establish): prediction disagreement, perturbation-induced entropy, and input-probability variance across the clean + 5 mildly perturbed forward passes.

*(Feature-Level Representation Stability — cosine similarity and variance of the model's embedding (`out["emb"]`) between the clean and each perturbed pass — was computed in earlier iterations of this analysis and remains in the cached outputs below, but is **not part of the final proposed uncertainty framework**. See the `# FEATURE-LEVEL UNCERTAINTY REMOVED FROM FINAL METHODOLOGY` markers in the code cells that follow.)*


In [ ]:
def _entropy_np(probs: np.ndarray, axis: int = -1, eps: float = 1e-12) -> np.ndarray:
    """Shannon entropy (nats) of a probability array along `axis`. Small, generic,
    reused by every uncertainty measure below instead of redefining it per-measure."""
    p = np.clip(probs, eps, 1.0)
    return -np.sum(p * np.log(p), axis=axis)


class _PathTaggedDataset(Dataset):
    """Thin wrapper so an arbitrary path->tensor preprocessing function (`preprocess`,
    or `preprocess_perturbed` bound to a fixed kind/severity) can be run through a
    standard batched DataLoader, exactly like BrainTumorDataset/PerturbedDataset
    already do above -- this just avoids writing one bespoke Dataset subclass per
    perturbation kind."""
    def __init__(self, samples: List[Dict], preprocess_fn):
        self.samples = samples
        self.preprocess_fn = preprocess_fn

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        return {"image": self.preprocess_fn(s["path"]), "path": s["path"]}


@torch.no_grad()
def _run_inference_pass(model: nn.Module, samples: List[Dict], preprocess_fn,
                         batch_size: int = BATCH_SIZE, num_workers: int = NUM_WORKERS
                         ) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """One deterministic (dropout OFF) forward pass of `samples` through `model`,
    using `preprocess_fn(path) -> [3,H,W] tensor`. Returns (probs[N,C], emb[N,D],
    paths) in the fixed order `samples` was given -- reused for the clean pass and
    every perturbed pass below, and again for external-validation uncertainty."""
    ds = _PathTaggedDataset(samples, preprocess_fn)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                         num_workers=num_workers, pin_memory=True)
    model.eval()
    probs_chunks, emb_chunks, paths = [], [], []
    for batch in loader:
        img = batch["image"].to(DEVICE, non_blocking=True)
        out = model(img)
        probs_chunks.append(torch.softmax(out["logits"], dim=1).cpu().numpy())
        emb_chunks.append(out["emb"].cpu().numpy())
        paths.extend(batch["path"])
    return np.concatenate(probs_chunks, axis=0), np.concatenate(emb_chunks, axis=0), paths


# Mild perturbations = severity-LEVEL-1 (mildest) entries already defined in
# ROBUSTNESS_SEVERITIES (§17) -- reused as-is rather than re-tuned here.
MC_UNCERTAINTY_PERTURBATIONS = [
    ("brightness",     ROBUSTNESS_SEVERITIES["brightness"][0]),
    ("contrast",       ROBUSTNESS_SEVERITIES["contrast"][0]),
    ("gaussian_noise", ROBUSTNESS_SEVERITIES["gaussian_noise"][0]),
    ("blur",           ROBUSTNESS_SEVERITIES["blur"][0]),
    ("rotation",       ROBUSTNESS_SEVERITIES["rotation"][0]),
]
print("[INPUT/FEATURE SENSITIVITY] mild perturbation set (reused from ROBUSTNESS_SEVERITIES level-1):")
for kind, sev in MC_UNCERTAINTY_PERTURBATIONS:
    print(f"  {kind:16s} severity={sev}")


def compute_input_feature_uncertainty(model: nn.Module, samples: List[Dict],
                                       perturb_specs: List[Tuple[str, float]] = MC_UNCERTAINTY_PERTURBATIONS,
                                       batch_size: int = BATCH_SIZE) -> pd.DataFrame:
    """Runs 1 clean + len(perturb_specs) mildly-perturbed deterministic passes over
    `samples` (each pass batched over the full set, not per-image), then computes,
    per sample:
      Input-level:   input_perturbation_entropy, prediction_disagreement,
                     input_probability_variance
    (Feature-level representation-stability measures were computed in earlier runs
    but have been removed from the final uncertainty methodology; see note below.)
    """
    t0 = time.perf_counter()
    print(f"[INPUT/FEATURE SENSITIVITY] clean + {len(perturb_specs)} mild-perturbation "
          f"passes over {len(samples)} samples")

    clean_probs, clean_emb, paths = _run_inference_pass(model, samples, preprocess, batch_size)
    clean_pred = clean_probs.argmax(axis=1)

    perturbed_probs_list, perturbed_emb_list = [], []
    for kind, sev in perturb_specs:
        pf = (lambda p, kind=kind, sev=sev: preprocess_perturbed(p, kind, sev))
        probs_p, emb_p, paths_p = _run_inference_pass(model, samples, pf, batch_size)
        assert paths_p == paths, "sample ordering drifted between clean and perturbed pass"
        perturbed_probs_list.append(probs_p)
        perturbed_emb_list.append(emb_p)
        print(f"  [{kind:16s} severity={sev}]  done")

    perturbed_probs = np.stack(perturbed_probs_list, axis=0)   # [K, N, C]
    perturbed_emb   = np.stack(perturbed_emb_list,   axis=0)   # [K, N, D]

    # --- Input-level sensitivity ---
    perturbed_preds = perturbed_probs.argmax(axis=2)                              # [K, N]
    prediction_disagreement = (perturbed_preds != clean_pred[None, :]).mean(axis=0)  # [N]
    mean_perturbed_probs = perturbed_probs.mean(axis=0)                           # [N, C]
    input_perturbation_entropy = _entropy_np(mean_perturbed_probs, axis=1)        # [N]
    input_probability_variance = perturbed_probs.var(axis=0).mean(axis=1)         # [N]

# FEATURE-LEVEL UNCERTAINTY REMOVED FROM FINAL METHODOLOGY
# Retained only to preserve previously generated notebook outputs.
    #
    # (Previously this block also computed feature-level representation-stability
    # measures -- feature_mean/min_cosine_similarity, feature_variance, and
    # feature_instability -- from `perturbed_emb`/`clean_emb`. That computation is
    # intentionally disabled below so it is no longer part of the methodology going
    # forward; `perturbed_emb`/`clean_emb` are still returned-from `_run_inference_pass`
    # upstream but are simply unused here now.
    #
    # clean_emb_n = clean_emb / (np.linalg.norm(clean_emb, axis=1, keepdims=True) + 1e-8)
    # K = perturbed_emb.shape[0]
    # cos_sims = np.zeros((K, len(samples)), dtype=np.float64)
    # for k in range(K):
    #     pe_n = perturbed_emb[k] / (np.linalg.norm(perturbed_emb[k], axis=1, keepdims=True) + 1e-8)
    #     cos_sims[k] = np.sum(clean_emb_n * pe_n, axis=1)
    # feature_mean_cosine_similarity = cos_sims.mean(axis=0)
    # feature_min_cosine_similarity = cos_sims.min(axis=0)
    # feature_instability = 1.0 - feature_mean_cosine_similarity
    # feature_variance = perturbed_emb.var(axis=0).mean(axis=1)                     # [N]

    df = pd.DataFrame({
        "path": paths,
        "input_perturbation_entropy": input_perturbation_entropy,
        "prediction_disagreement": prediction_disagreement,
        "input_probability_variance": input_probability_variance,
    })
    elapsed = time.perf_counter() - t0
    df.attrs["elapsed_seconds"] = elapsed
    df.attrs["n_forward_passes"] = 1 + len(perturb_specs)
    print(f"[INPUT/FEATURE SENSITIVITY] done in {elapsed:.2f}s "
          f"({elapsed / len(samples) * 1000:.3f} ms/sample amortized, "
          f"{1 + len(perturb_specs)} passes total)")
    return df

print("Input-level perturbation-sensitivity utilities ready. (Feature-level stability utilities historical only -- not part of final methodology.)")


In [ ]:
input_feature_df = compute_input_feature_uncertainty(
    full_result["model"], full_result["test_samples"],
    perturb_specs=MC_UNCERTAINTY_PERTURBATIONS, batch_size=BATCH_SIZE)
input_feature_df.to_csv(OUT_DIR / "input_feature_uncertainty_test.csv", index=False)
input_feature_df.describe()


### 4. Prediction-Level MC Dropout Uncertainty

SMGF-Net's only stochastic-regularization layer is the `nn.Dropout` inside `ProjectionHead` (`model.head.drop`, rate = `DROPOUT_RATE` = 0.3) — the timm `features_only=True` MobileNetV3-Small backbone has no dropout of its own, and `AMFF`/`RLDA` are deterministic. `enable_mc_dropout` below switches on *only* `nn.Dropout*` modules (found generically via `model.modules()`, so this doesn't hardcode `head.drop` and stays correct if the architecture ever changes) while `model.eval()` keeps every `BatchNorm2d`/`BatchNorm1d` layer — and everything else — in evaluation mode, exactly as required. No architecture change, no retraining.


In [ ]:
_DROPOUT_TYPES = (nn.Dropout, nn.Dropout2d, nn.Dropout3d, nn.AlphaDropout)


def enable_mc_dropout(model: nn.Module) -> List[nn.Module]:
    """Call model.eval() first, then this: flips ONLY nn.Dropout*-type submodules
    back to train() (so they sample a fresh mask on every forward call) while every
    other module -- BatchNorm included -- stays in eval mode. Returns the list of
    dropout modules found, so callers can warn if there aren't any."""
    found = []
    for m in model.modules():
        if isinstance(m, _DROPOUT_TYPES):
            m.train()
            found.append(m)
    return found


@torch.no_grad()
def mc_dropout_predict(model: nn.Module, samples: List[Dict], T: int = 30,
                        batch_size: int = BATCH_SIZE, seed: int = SEED,
                        num_workers: int = NUM_WORKERS) -> pd.DataFrame:
    """Monte Carlo Dropout: T stochastic forward passes over the full `samples` set
    (batched -- T total sweeps of the loader, not T per individual image), with
    BatchNorm and all non-dropout layers frozen in eval mode throughout. `T` is
    configurable via the argument, default 30 per the task brief.

    For every sample, returns: mean predictive probability, predictive entropy
    (total uncertainty), expected entropy (average per-pass entropy), mutual
    information = predictive entropy - expected entropy (the primary epistemic/
    model-uncertainty estimate used throughout this section), and predictive
    probability variance (mean, across classes, of the per-class variance over the
    T passes)."""
    ds = BrainTumorDataset(samples, is_train=False)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                         num_workers=num_workers, pin_memory=True)

    model.eval()                                  # BatchNorm + everything -> eval
    dropout_modules = enable_mc_dropout(model)     # dropout only -> train
    n_dropout = len(dropout_modules)
    if n_dropout == 0:
        print("[MC DROPOUT][WARNING] No nn.Dropout* layers found in the model -- "
              "every stochastic pass will be numerically identical, so mutual "
              "information / predictive variance will be ~0 for all samples. "
              "Reporting the measures anyway for completeness, but they carry no "
              "epistemic-uncertainty signal in that case.")

    torch.manual_seed(seed)   # reproducible sequence of dropout masks across passes
    all_probs, paths_master = [], None
    t0 = time.perf_counter()
    for t in range(T):
        probs_t, paths_t = [], []
        for batch in loader:
            img = batch["image"].to(DEVICE, non_blocking=True)
            out = model(img)
            probs_t.append(torch.softmax(out["logits"], dim=1).cpu().numpy())
            paths_t.extend(batch["path"])
        all_probs.append(np.concatenate(probs_t, axis=0))
        if paths_master is None:
            paths_master = paths_t
        if (t + 1) % max(1, T // 5) == 0 or (t + 1) == T:
            print(f"  [MC Dropout] stochastic pass {t + 1}/{T} complete")
    elapsed = time.perf_counter() - t0

    model.eval()   # restore full eval mode (dropout back off) once MC sampling is done
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    probs_arr = np.stack(all_probs, axis=0)               # [T, N, C]
    mean_probs = probs_arr.mean(axis=0)                   # [N, C]
    mc_pred_idx = mean_probs.argmax(axis=1)
    predictive_entropy = _entropy_np(mean_probs, axis=1)              # [N]
    per_pass_entropy = _entropy_np(probs_arr, axis=2)                 # [T, N]
    expected_entropy = per_pass_entropy.mean(axis=0)                  # [N]
    mutual_information = predictive_entropy - expected_entropy        # [N]
    predictive_variance = probs_arr.var(axis=0).mean(axis=1)          # [N]

    rows = []
    for i, p in enumerate(paths_master):
        row = {
            "path": p, "mc_pred_idx": int(mc_pred_idx[i]),
            "predictive_entropy": float(predictive_entropy[i]),
            "expected_entropy": float(expected_entropy[i]),
            "mutual_information": float(mutual_information[i]),
            "predictive_variance": float(predictive_variance[i]),
        }
        for c, cname in enumerate(CLASS_NAMES):
            row[f"mc_mean_prob_{cname}"] = float(mean_probs[i, c])
        rows.append(row)
    df = pd.DataFrame(rows)
    df.attrs["mc_dropout_time_seconds"] = elapsed
    df.attrs["T"] = T
    df.attrs["n_dropout_layers"] = n_dropout
    print(f"[MC DROPOUT] T={T}  n_dropout_layers={n_dropout}  n_samples={len(df)}  "
          f"total_time={elapsed:.2f}s  ({elapsed / (len(df) * T) * 1000:.4f} ms/forward-pass, "
          f"{elapsed / len(df) * 1000:.3f} ms/sample amortized over all {T} passes)")
    return df

print("MC-Dropout utilities ready.")


In [ ]:
MC_T = 30   # configurable number of stochastic forward passes (per task brief default)

mc_dropout_df = mc_dropout_predict(full_result["model"], full_result["test_samples"],
                                    T=MC_T, batch_size=BATCH_SIZE, seed=SEED)
mc_dropout_df.to_csv(OUT_DIR / "mc_dropout_uncertainty_test.csv", index=False)
mc_dropout_df.describe()


### 5. Multi-Scale Uncertainty Analysis

Merges `test_pred_df` (§16, the existing per-sample prediction table) with `input_feature_df` and `mc_dropout_df` on `path` into one table, `multiscale_df`, saved as `multiscale_uncertainty.csv`. Every component below is kept individually interpretable; a single combined score is also reported, but only as an exploratory, clearly-documented, equal-weight mean of robust (median/IQR) z-scored components — never presented as a replacement for the individual measures, which are what the rest of this section actually analyzes.


In [ ]:
multiscale_df = (test_pred_df
                  .merge(input_feature_df, on="path", how="inner")
                  .merge(mc_dropout_df, on="path", how="inner"))
assert len(multiscale_df) == len(test_pred_df), (
    f"merge dropped rows: {len(multiscale_df)} vs {len(test_pred_df)} -- "
    "check for path mismatches between the three source tables.")


def _robust_zscore(x: np.ndarray) -> np.ndarray:
    """Median/IQR normalization (robust to the skewed, non-negative distributions
    typical of entropy/variance-type uncertainty measures) -- documented explicitly
    here rather than silently baked into a combined score."""
    x = np.asarray(x, dtype=np.float64)
    med = np.median(x)
    iqr = np.percentile(x, 75) - np.percentile(x, 25)
    if iqr < 1e-8:
        iqr = x.std() + 1e-8
    return (x - med) / iqr


# FEATURE-LEVEL UNCERTAINTY REMOVED FROM FINAL METHODOLOGY
# Retained only to preserve previously generated notebook outputs.
_MULTISCALE_COMPONENTS = [
    "predictive_entropy", "mutual_information", "predictive_variance",           # prediction-level
    "input_perturbation_entropy", "prediction_disagreement", "input_probability_variance",  # input-level
    # "feature_instability",   # feature-level -- removed from final methodology
]
for c in _MULTISCALE_COMPONENTS:
    multiscale_df[f"{c}_z"] = _robust_zscore(multiscale_df[c].to_numpy())

# Exploratory combined score ONLY -- equal-weight mean of the robust z-scores above.
# Reported alongside, never in place of, the individual components (Part 2 requirement).
multiscale_df["multiscale_uncertainty_score"] = multiscale_df[
    [f"{c}_z" for c in _MULTISCALE_COMPONENTS]].mean(axis=1)

_MULTISCALE_COLUMNS = [
    "path", "true_class", "pred_class", "correct", "confidence",
    "predictive_entropy", "expected_entropy", "mutual_information", "predictive_variance",
    "input_perturbation_entropy", "prediction_disagreement", "input_probability_variance",
    # "feature_variance", "feature_mean_cosine_similarity", "feature_min_cosine_similarity",
    # "feature_instability",   # feature-level columns -- removed from final methodology
    "multiscale_uncertainty_score",
] + [f"{c}_z" for c in _MULTISCALE_COMPONENTS]

multiscale_df = multiscale_df[_MULTISCALE_COLUMNS + [c for c in multiscale_df.columns
                                                       if c not in _MULTISCALE_COLUMNS]]
multiscale_df.to_csv(OUT_DIR / "multiscale_uncertainty.csv", index=False)
print(f"[MULTI-SCALE UNCERTAINTY] saved multiscale_uncertainty.csv  "
      f"({len(multiscale_df)} rows, {multiscale_df.shape[1]} columns)")
multiscale_df[["path", "correct", "confidence", "predictive_entropy", "mutual_information",
               "input_perturbation_entropy",
               "multiscale_uncertainty_score"]].head()


In [ ]:
# Figure 1 -- Multi-Scale Uncertainty Overview: Input -> Feature -> Prediction,
# split by correct/incorrect. Uses the robust z-scored components computed above
# so the three levels (measured in unrelated units/scales) are visually comparable;
# this is a qualitative overview figure, not a claim that the levels are literally
# on the same scale.
# FEATURE-LEVEL UNCERTAINTY REMOVED FROM FINAL METHODOLOGY
# Retained only to preserve previously generated notebook outputs.
_level_groups = {
    "Input-level":      ["prediction_disagreement_z", "input_perturbation_entropy_z",
                          "input_probability_variance_z"],
    # "Feature-level":  ["feature_instability_z"],   # removed from final methodology
    "Prediction-level":  ["predictive_entropy_z", "mutual_information_z", "predictive_variance_z"],
}
_level_names = list(_level_groups.keys())
_correct_means = [multiscale_df.loc[multiscale_df["correct"], cols].mean(axis=1).mean()
                   for cols in _level_groups.values()]
_incorrect_means = [multiscale_df.loc[~multiscale_df["correct"], cols].mean(axis=1).mean()
                     for cols in _level_groups.values()]

fig, ax = plt.subplots(figsize=(7, 5))
x = np.arange(len(_level_names))
width = 0.35
ax.bar(x - width / 2, _correct_means, width, color="#4ac9c2", label="Correct predictions")
ax.bar(x + width / 2, _incorrect_means, width, color="#e8664a", label="Incorrect predictions")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_xticks(x); ax.set_xticklabels(_level_names)
ax.set_ylabel("Mean robust z-score (higher = more uncertain)")
ax.set_title("Figure 1 — Multi-Scale Uncertainty Overview\n(Input → Prediction levels)")
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(OUT_DIR / "fig1_multiscale_uncertainty_overview.png", dpi=150, bbox_inches="tight")
plt.show()


### 6. Explainability Consistency

Reuses the existing `GradCAMPlusPlus`, `ScoreCAM`, `LimeImageExplainer`, `compute_method_agreement`, and `lime_segments_to_dense_map` (§14) unchanged — no duplicate CAM/LIME implementations. Full Grad-CAM++ + Score-CAM + LIME + deletion/insertion on every test image would be very expensive (LIME alone fits a surrogate model per image), so — following the same pattern `run_xai_batch_validation`/`run_agreement_analysis` already use in §14 — this runs on a fixed, stratified, per-class subsample (`EXPLAIN_N_PER_CLASS` per class, seeded) rather than the full test set. The multi-scale uncertainty table above (§5) still covers every test sample; only the explanation-based columns are subsample-limited, and this is used consistently for every explanation-dependent analysis later in this section.


In [ ]:
def select_stratified_subset(samples: List[Dict], n_per_class: int, seed: int = SEED) -> List[Dict]:
    """Same per-class random sampling pattern already used by run_xai_batch_validation
    / run_agreement_analysis (§14) -- factored out here so it's reused for the test
    subset, the validation-threshold subset (§9), and the external-validation
    subset (§11) instead of re-implemented three times."""
    rng = random.Random(seed)
    by_class: Dict[str, List[Dict]] = {c: [] for c in CLASS_NAMES}
    for s in samples:
        by_class[s["class_name"]].append(s)
    chosen = []
    for c, lst in by_class.items():
        chosen.extend(rng.sample(lst, min(n_per_class, len(lst))))
    return chosen


def run_explainability_reliability_batch(model: nn.Module, samples: List[Dict],
                                          top_percent: float = 0.15,
                                          compute_faithfulness: bool = True) -> pd.DataFrame:
    """Per sample: Grad-CAM++, Score-CAM, LIME (§14 classes, reused as-is), pairwise
    agreement via the existing compute_method_agreement (IoU-top-k + Spearman rho,
    §14), a combined Explanation Consistency Score (transparent mean of the three
    pairwise IoU-top-k values -- documented, not hidden; individual pair metrics are
    always reported alongside it), and -- when compute_faithfulness=True -- deletion/
    insertion AUC per method via the existing deletion_insertion_pixelwise /
    deletion_insertion_segments (§14b), reused unchanged as faithfulness indicators
    (never reinterpreted as uncertainty, per the task brief)."""
    rows = []
    for s in samples:
        input_tensor = load_input_tensor(s["path"])

        gcpp = GradCAMPlusPlus(model, model.attn)
        cam_gcpp, pred_idx = gcpp.generate(input_tensor)
        gcpp.remove()

        scam = ScoreCAM(model, model.attn)
        cam_score, _ = scam.generate(input_tensor, class_idx=pred_idx)
        scam.remove()

        lime_exp = LimeImageExplainer(model).explain(input_tensor, class_idx=pred_idx)
        cam_lime = lime_segments_to_dense_map(lime_exp)

        gs = compute_method_agreement(cam_gcpp, cam_score, top_percent)
        gl = compute_method_agreement(cam_gcpp, cam_lime, top_percent)
        sl = compute_method_agreement(cam_score, cam_lime, top_percent)
        explanation_consistency = float(np.mean([gs["iou_top_k"], gl["iou_top_k"], sl["iou_top_k"]]))

        row = {
            "path": s["path"], "true_class": s["class_name"], "pred_class": CLASS_NAMES[pred_idx],
            "gradcampp_vs_scorecam_iou": gs["iou_top_k"], "gradcampp_vs_scorecam_rho": gs["spearman_rho"],
            "gradcampp_vs_lime_iou": gl["iou_top_k"], "gradcampp_vs_lime_rho": gl["spearman_rho"],
            "scorecam_vs_lime_iou": sl["iou_top_k"], "scorecam_vs_lime_rho": sl["spearman_rho"],
            "explanation_consistency": explanation_consistency,
        }
        if compute_faithfulness:
            di_gcpp = deletion_insertion_pixelwise(model, input_tensor, cam_gcpp, pred_idx)
            di_score = deletion_insertion_pixelwise(model, input_tensor, cam_score, pred_idx)
            di_lime = deletion_insertion_segments(model, input_tensor, lime_exp, pred_idx)
            row.update({
                "gradcampp_deletion_auc": di_gcpp["deletion_auc"], "gradcampp_insertion_auc": di_gcpp["insertion_auc"],
                "scorecam_deletion_auc": di_score["deletion_auc"], "scorecam_insertion_auc": di_score["insertion_auc"],
                "lime_deletion_auc": di_lime["deletion_auc"], "lime_insertion_auc": di_lime["insertion_auc"],
            })
        rows.append(row)
    return pd.DataFrame(rows)

print("Explainability-reliability batch utilities ready.")


In [ ]:
EXPLAIN_N_PER_CLASS = 8   # per-class subsample size for the explanation-based analyses

explain_subset_samples = select_stratified_subset(full_result["test_samples"], EXPLAIN_N_PER_CLASS)
print(f"[EXPLAINABILITY CONSISTENCY] running Grad-CAM++/Score-CAM/LIME + faithfulness on "
      f"{len(explain_subset_samples)} stratified test samples ({EXPLAIN_N_PER_CLASS}/class)")

t0 = time.perf_counter()
explain_reliability_df = run_explainability_reliability_batch(
    full_result["model"], explain_subset_samples, compute_faithfulness=True)
_explain_elapsed = time.perf_counter() - t0
explain_reliability_df.attrs["elapsed_seconds"] = _explain_elapsed
explain_reliability_df.to_csv(OUT_DIR / "explainability_reliability_test_subset.csv", index=False)

print(f"[EXPLAINABILITY CONSISTENCY] done in {_explain_elapsed:.2f}s "
      f"({_explain_elapsed / len(explain_subset_samples):.2f} s/sample)")
print(f"  mean explanation_consistency = {explain_reliability_df['explanation_consistency'].mean():.4f}")
explain_reliability_df.describe()


### 7. Explainability-Guided Reliability

Merges the multi-scale uncertainty table (§5, full test set) with the explanation-reliability table (§6, stratified subsample) on `path`, restricted to the subsample where explanations were actually computed. This merged table (`combined_reliability_df`, saved as `combined_reliability.csv`) is what §8–§10 use for anything that needs both an uncertainty measure and an explanation measure on the same sample.


In [ ]:
combined_reliability_df = multiscale_df.merge(
    explain_reliability_df.drop(columns=["true_class", "pred_class"]), on="path", how="inner")
assert len(combined_reliability_df) == len(explain_reliability_df), (
    "merge lost rows between multiscale_df and explain_reliability_df -- path mismatch")

combined_reliability_df.to_csv(OUT_DIR / "combined_reliability.csv", index=False)
print(f"[EXPLAINABILITY-GUIDED RELIABILITY] combined_reliability.csv saved "
      f"({len(combined_reliability_df)} rows)")
# FEATURE-LEVEL UNCERTAINTY REMOVED FROM FINAL METHODOLOGY
# Retained only to preserve previously generated notebook outputs.
combined_reliability_df[[
    "path", "true_class", "pred_class", "correct", "confidence",
    "predictive_entropy", "expected_entropy", "mutual_information", "predictive_variance",
    "input_perturbation_entropy",  # "feature_instability" removed from final methodology
    "gradcampp_vs_scorecam_iou", "gradcampp_vs_lime_iou", "scorecam_vs_lime_iou",
    "explanation_consistency", "gradcampp_deletion_auc", "gradcampp_insertion_auc",
]].head()


### 8. Uncertainty–Explanation Relationship

Three things happen in this subsection:

1. **Correct vs. incorrect statistical comparison** on the full-test-set uncertainty measures (`multiscale_df`) and the subsample explanation-consistency measure (`combined_reliability_df`) — distributions are inspected first, a non-parametric test is used given the bounded, right-skewed shape typical of entropy/variance-type measures, effect sizes are reported, and a Benjamini–Hochberg correction is applied across the multiple comparisons.
2. **Correlation analysis** (Spearman, for the same distributional reason) among confidence, the uncertainty measures, and explanation consistency — reported as a heatmap. Correlation is not causation, and is not interpreted as such below.
3. **Calibration integration** (folds in the task brief's Part 11): reuses the existing `compare_calibration` (§20) unchanged to compare reliability diagrams for the below-median vs. above-median mutual-information halves of the test set. This split uses only the *uncertainty value* (mutual information), never the test labels, so it introduces no test-label leakage — it is a median split on an unsupervised score, not a threshold tuned against correctness.

Figures 2, 3, 4, and 5 (per the task brief) are produced at the end of this subsection.


In [ ]:
from scipy.stats import mannwhitneyu, shapiro

# --- Inspect distributions first (Part 9 requirement: "do not perform statistical
# tests blindly"), rather than assuming normality. ---
print("[DISTRIBUTION CHECK] Shapiro-Wilk normality test (subsampled to <=5000 for speed), "
      "per uncertainty measure, on the full test set:")
# FEATURE-LEVEL UNCERTAINTY REMOVED FROM FINAL METHODOLOGY
# Retained only to preserve previously generated notebook outputs.
_dist_check_cols = ["predictive_entropy", "mutual_information", "predictive_variance",
                     "input_perturbation_entropy"]  # "feature_instability" removed from final methodology
_rng_shapiro = np.random.RandomState(SEED)
for col in _dist_check_cols:
    vals = multiscale_df[col].to_numpy()
    sample = vals if len(vals) <= 5000 else _rng_shapiro.choice(vals, 5000, replace=False)
    try:
        stat, p = shapiro(sample)
        print(f"  {col:28s}  W={stat:.4f}  p={p:.2e}  "
              f"{'-> NOT normal (reject H0)' if p < 0.05 else '-> cannot reject normality'}")
    except Exception as e:
        print(f"  {col:28s}  shapiro failed: {e}")
print("Given the bounded, right-skewed shapes typical of entropy/variance measures "
      "(confirmed above), a non-parametric test (Mann-Whitney U) is used below rather "
      "than a t-test.")


def _rank_biserial_effect_size(a: np.ndarray, b: np.ndarray, u_stat: float) -> float:
    """Rank-biserial correlation from the Mann-Whitney U statistic: r = 1 - 2U/(n1*n2).
    Bounded in [-1, 1]; sign indicates which group tends higher."""
    n1, n2 = len(a), len(b)
    return 1.0 - (2.0 * u_stat) / (n1 * n2)


def compare_correct_incorrect(df: pd.DataFrame, cols: List[str],
                               group_col: str = "correct") -> pd.DataFrame:
    """Mann-Whitney U test (correct vs. incorrect) per column, with mean/median/std/
    IQR for both groups, a rank-biserial effect size, and a Benjamini-Hochberg
    multiple-comparison correction applied across all columns tested together."""
    rows = []
    for col in cols:
        a = df.loc[df[group_col] == True, col].dropna().to_numpy()   # noqa: E712
        b = df.loc[df[group_col] == False, col].dropna().to_numpy()  # noqa: E712
        if len(a) < 2 or len(b) < 2:
            print(f"  [SKIP] '{col}': insufficient samples in one group (n_correct={len(a)}, n_incorrect={len(b)})")
            continue
        u_stat, p = mannwhitneyu(a, b, alternative="two-sided")
        eff = _rank_biserial_effect_size(a, b, u_stat)
        rows.append({
            "metric": col, "n_correct": len(a), "n_incorrect": len(b),
            "mean_correct": a.mean(), "mean_incorrect": b.mean(),
            "median_correct": np.median(a), "median_incorrect": np.median(b),
            "std_correct": a.std(), "std_incorrect": b.std(),
            "iqr_correct": np.percentile(a, 75) - np.percentile(a, 25),
            "iqr_incorrect": np.percentile(b, 75) - np.percentile(b, 25),
            "U_statistic": u_stat, "p_value": p, "rank_biserial_effect_size": eff,
        })
    res = pd.DataFrame(rows)
    if len(res):
        # Benjamini-Hochberg FDR correction across the tests performed together.
        m = len(res)
        order = np.argsort(res["p_value"].to_numpy())
        sorted_p = res["p_value"].to_numpy()[order]
        bh = np.empty(m)
        running_min = 1.0
        for rank in range(m - 1, -1, -1):
            val = sorted_p[rank] * m / (rank + 1)
            running_min = min(running_min, val)
            bh[order[rank]] = running_min
        res["p_value_bh_corrected"] = np.clip(bh, 0, 1)
        res["significant_bh_0.05"] = res["p_value_bh_corrected"] < 0.05
    return res

print("Correct-vs-incorrect statistical comparison utilities ready.")


In [ ]:
# Uncertainty measures: full test set (multiscale_df) -- maximum statistical power.
# FEATURE-LEVEL UNCERTAINTY REMOVED FROM FINAL METHODOLOGY
# Retained only to preserve previously generated notebook outputs.
_uncertainty_stat_cols = ["predictive_entropy", "mutual_information", "predictive_variance",
                           "input_perturbation_entropy", "prediction_disagreement"]  # "feature_instability" removed
uncertainty_stats_df = compare_correct_incorrect(multiscale_df, _uncertainty_stat_cols)
uncertainty_stats_df.to_csv(OUT_DIR / "uncertainty_correct_vs_incorrect_stats.csv", index=False)
print("\n[CORRECT vs INCORRECT — uncertainty measures, full test set, Mann-Whitney U + BH-corrected]")
print(uncertainty_stats_df.to_string(index=False))

# Explanation consistency: subsample only (combined_reliability_df).
explanation_stats_df = compare_correct_incorrect(combined_reliability_df, ["explanation_consistency"])
explanation_stats_df.to_csv(OUT_DIR / "explanation_correct_vs_incorrect_stats.csv", index=False)
print("\n[CORRECT vs INCORRECT — explanation consistency, test subsample]")
print(explanation_stats_df.to_string(index=False))


In [ ]:
# --- Correlation analysis (Part 10). Spearman (rank) correlation given the
# non-normal distributions confirmed above; "correct" is included as a 0/1 indicator
# (point-biserial via Spearman is a standard, if approximate, way to relate a binary
# outcome to continuous scores). Correlation does NOT imply causation -- interpreted
# only descriptively below. ---
# FEATURE-LEVEL UNCERTAINTY REMOVED FROM FINAL METHODOLOGY
# Retained only to preserve previously generated notebook outputs.
_corr_cols = ["confidence", "predictive_entropy", "mutual_information",
              "input_perturbation_entropy", "explanation_consistency", "correct"]  # "feature_instability" removed
_corr_df = combined_reliability_df[_corr_cols].copy()
_corr_df["correct"] = _corr_df["correct"].astype(int)
spearman_corr = _corr_df.corr(method="spearman")
spearman_corr.to_csv(OUT_DIR / "uncertainty_explanation_correlation.csv")

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(spearman_corr.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(_corr_cols))); ax.set_yticks(range(len(_corr_cols)))
ax.set_xticklabels(_corr_cols, rotation=45, ha="right"); ax.set_yticklabels(_corr_cols)
for i in range(len(_corr_cols)):
    for j in range(len(_corr_cols)):
        ax.text(j, i, f"{spearman_corr.iloc[i, j]:.2f}", ha="center", va="center",
                fontsize=8, color="white" if abs(spearman_corr.iloc[i, j]) > 0.5 else "black")
plt.colorbar(im, ax=ax, label="Spearman ρ")
ax.set_title("Uncertainty–Explanation–Confidence Correlation (test subsample)\n"
             "(correlation, not causation)")
plt.tight_layout()
plt.savefig(OUT_DIR / "uncertainty_explanation_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Key correlations (Spearman ρ):")
print(f"  confidence vs predictive_entropy       = {spearman_corr.loc['confidence','predictive_entropy']:.3f}")
print(f"  confidence vs mutual_information        = {spearman_corr.loc['confidence','mutual_information']:.3f}")
print(f"  mutual_information vs explanation_consistency = {spearman_corr.loc['mutual_information','explanation_consistency']:.3f}")
# print(f"  feature_instability vs explanation_consistency = ...")  # removed from final methodology
print(f"  correct vs mutual_information            = {spearman_corr.loc['correct','mutual_information']:.3f}")


In [ ]:
# --- Calibration integration (Part 11): reuses compare_calibration (§20) unchanged.
# Split is a MEDIAN split on mutual_information itself (an unsupervised score) --
# no test labels are used to pick the split point, so this introduces no test-set
# leakage; labels are only used afterward, to evaluate calibration within each half. ---
_median_mi = float(multiscale_df["mutual_information"].median())
_low_unc_mask = multiscale_df["mutual_information"] <= _median_mi
_high_unc_mask = ~_low_unc_mask

_prob_cols = [f"prob_{c}" for c in CLASS_NAMES]
_low_probs = multiscale_df.loc[_low_unc_mask, _prob_cols].to_numpy()
_low_labels = multiscale_df.loc[_low_unc_mask, "true_idx"].to_numpy()
_high_probs = multiscale_df.loc[_high_unc_mask, _prob_cols].to_numpy()
_high_labels = multiscale_df.loc[_high_unc_mask, "true_idx"].to_numpy()

print(f"[CALIBRATION x UNCERTAINTY] median-split on mutual_information = {_median_mi:.4f}  "
      f"(unsupervised split, no label leakage)")
print(f"  low-uncertainty half:  n={_low_unc_mask.sum()}  accuracy={multiscale_df.loc[_low_unc_mask,'correct'].mean():.4f}")
print(f"  high-uncertainty half: n={_high_unc_mask.sum()}  accuracy={multiscale_df.loc[_high_unc_mask,'correct'].mean():.4f}")

calibration_by_uncertainty_df = compare_calibration(
    {"Low uncertainty (<= median MI)": (_low_probs, _low_labels),
     "High uncertainty (> median MI)": (_high_probs, _high_labels)},
    save_path=str(OUT_DIR / "calibration_by_uncertainty_group.png"))
calibration_by_uncertainty_df.to_csv(OUT_DIR / "calibration_by_uncertainty_group.csv", index=False)
print(calibration_by_uncertainty_df.to_string(index=False))


In [ ]:
# Figure 2 -- Correct vs. Incorrect Uncertainty
# FEATURE-LEVEL UNCERTAINTY REMOVED FROM FINAL METHODOLOGY
# Retained only to preserve previously generated notebook outputs.
_fig2_metrics = ["predictive_entropy", "mutual_information", "prediction_disagreement"]  # "feature_instability" removed
fig, axes = plt.subplots(1, len(_fig2_metrics), figsize=(4.5 * len(_fig2_metrics), 4.5))
for ax, metric in zip(axes, _fig2_metrics):
    data = [multiscale_df.loc[multiscale_df["correct"], metric].to_numpy(),
            multiscale_df.loc[~multiscale_df["correct"], metric].to_numpy()]
    bp = ax.boxplot(data, labels=["Correct", "Incorrect"], patch_artist=True, showfliers=False)
    for patch, color in zip(bp["boxes"], ["#4ac9c2", "#e8664a"]):
        patch.set_facecolor(color); patch.set_alpha(0.8)
    ax.set_title(metric.replace("_", " ").title()); ax.grid(alpha=0.3, axis="y")
fig.suptitle("Figure 2 — Uncertainty: Correct vs. Incorrect Predictions")
plt.tight_layout()
plt.savefig(OUT_DIR / "fig2_correct_vs_incorrect_uncertainty.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Figure 3 -- Uncertainty by Tumor Class (class names read from CLASS_NAMES, not hardcoded)
# FEATURE-LEVEL UNCERTAINTY REMOVED FROM FINAL METHODOLOGY
# Retained only to preserve previously generated notebook outputs.
_fig3_metrics = ["predictive_entropy", "mutual_information"]  # "feature_instability" removed from final methodology
fig, axes = plt.subplots(1, len(_fig3_metrics), figsize=(5.5 * len(_fig3_metrics), 4.5))
for ax, metric in zip(axes, _fig3_metrics):
    data = [multiscale_df.loc[multiscale_df["true_class"] == c, metric].to_numpy() for c in CLASS_NAMES]
    bp = ax.boxplot(data, labels=CLASS_NAMES, patch_artist=True, showfliers=False)
    for patch in bp["boxes"]:
        patch.set_facecolor("#8b7fd6"); patch.set_alpha(0.8)
    ax.set_title(metric.replace("_", " ").title())
    ax.tick_params(axis="x", rotation=30); ax.grid(alpha=0.3, axis="y")
fig.suptitle("Figure 3 — Uncertainty by Tumor Class")
plt.tight_layout()
plt.savefig(OUT_DIR / "fig3_uncertainty_by_class.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Figure 4 -- Uncertainty-Explanation Reliability Map
# x = epistemic uncertainty (mutual information), y = explanation consistency.
# Quadrants (not drawn as hard boundaries, just annotated): low-MI + high-consistency
# = reliable; high-MI + high-consistency = uncertain-but-interpretable; low-MI +
# low-consistency = potentially overconfident; high-MI + low-consistency = unreliable.
# Explanation consistency here reflects agreement between XAI methods, NOT a
# guarantee of clinical correctness -- it is one supporting signal, not proof.
_df4 = combined_reliability_df
_suspicious = (_df4["confidence"] >= 0.9) & (_df4["explanation_consistency"] <= _df4["explanation_consistency"].median())

fig, ax = plt.subplots(figsize=(7.5, 6.5))
ax.scatter(_df4.loc[_df4["correct"] & ~_suspicious, "mutual_information"],
           _df4.loc[_df4["correct"] & ~_suspicious, "explanation_consistency"],
           c="#4ac9c2", label="Correct", alpha=0.75, s=45)
ax.scatter(_df4.loc[~_df4["correct"] & ~_suspicious, "mutual_information"],
           _df4.loc[~_df4["correct"] & ~_suspicious, "explanation_consistency"],
           c="#e8664a", label="Incorrect", alpha=0.75, s=45)
ax.scatter(_df4.loc[_suspicious, "mutual_information"], _df4.loc[_suspicious, "explanation_consistency"],
           facecolors="none", edgecolors="black", linewidths=1.5, s=110,
           label="High-confidence & low-consistency (suspicious)")
ax.axvline(_df4["mutual_information"].median(), color="gray", linestyle="--", alpha=0.5, linewidth=1)
ax.axhline(_df4["explanation_consistency"].median(), color="gray", linestyle="--", alpha=0.5, linewidth=1)
ax.set_xlabel("Epistemic Uncertainty (Mutual Information)")
ax.set_ylabel("Explanation Consistency")
ax.set_title("Figure 4 — Uncertainty–Explanation Reliability Map\n"
              "(dashed lines = medians; consistency reflects XAI-method agreement, not ground-truth correctness)")
ax.legend(fontsize=8, loc="best"); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig4_uncertainty_explanation_reliability_map.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Figure 5 -- Confidence vs. Uncertainty (full test set): exposes whether
# high-confidence predictions can still carry high uncertainty (overconfidence).
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, metric, title in zip(axes, ["predictive_entropy", "mutual_information"],
                              ["Confidence vs. Predictive Entropy", "Confidence vs. Mutual Information"]):
    ax.scatter(multiscale_df.loc[multiscale_df["correct"], "confidence"],
               multiscale_df.loc[multiscale_df["correct"], metric],
               c="#4ac9c2", alpha=0.4, s=12, label="Correct")
    ax.scatter(multiscale_df.loc[~multiscale_df["correct"], "confidence"],
               multiscale_df.loc[~multiscale_df["correct"], metric],
               c="#e8664a", alpha=0.6, s=12, label="Incorrect")
    ax.set_xlabel("Confidence"); ax.set_ylabel(metric.replace("_", " ").title())
    ax.set_title(title); ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.suptitle("Figure 5 — Confidence vs. Uncertainty (full test set)")
plt.tight_layout()
plt.savefig(OUT_DIR / "fig5_confidence_vs_uncertainty.png", dpi=150, bbox_inches="tight")
plt.show()

_high_conf_high_mi = multiscale_df[(multiscale_df["confidence"] >= 0.9) &
                                    (multiscale_df["mutual_information"] >=
                                     multiscale_df["mutual_information"].quantile(0.75))]
print(f"[OVERCONFIDENCE CHECK] {len(_high_conf_high_mi)}/{len(multiscale_df)} test predictions "
      f"({len(_high_conf_high_mi) / len(multiscale_df) * 100:.1f}%) have confidence>=0.9 AND "
      f"mutual_information in the top quartile -- i.e. the model is highly confident despite "
      f"elevated epistemic uncertainty by this measure.")


### 9. High-Confidence Suspicious Predictions

Thresholds for "high uncertainty" and "low explanation consistency" are derived from the **validation split** (`full_result["val_samples"]`, never touched by test-set evaluation) — a lightweight MC-Dropout pass and a small stratified explanation subsample on validation data, both unsupervised with respect to the *test* set. This is the primary/methodological threshold. A test-set-percentile version is also computed purely as an *exploratory* comparison and is labeled as such throughout — it is not used for any conclusion below.

Four reliability categories are then assigned to every sample in `combined_reliability_df` (Part 6 of the task brief):

- **Reliable** — correct, low uncertainty, consistent explanations.
- **Uncertain but correct** — correct despite high uncertainty (ambiguous but not wrong).
- **Confident but suspicious** — high confidence combined with high uncertainty and/or poor explanation consistency (the overconfidence red flag).
- **Highly unreliable** — incorrect, high uncertainty, poor explanation consistency (candidates for human review).

The strict intersection of "high confidence AND high uncertainty AND poor explanation consistency AND incorrect" is reported separately as **Potentially Unreliable High-Confidence Predictions** (not asserted to be "clinically unsafe" — that claim isn't supported by this analysis alone).


In [ ]:
# --- Validation-derived thresholds (methodological / final; Part 13: no test-label leakage) ---
VAL_MC_T = min(MC_T, 20)          # slightly lighter than the test-set MC pass, for speed
VAL_EXPLAIN_N_PER_CLASS = 5

print(f"[THRESHOLD DERIVATION] running MC-Dropout (T={VAL_MC_T}) on the validation split "
      f"(n={len(full_result['val_samples'])}) -- used only to set thresholds, never to report accuracy.")
val_mc_df = mc_dropout_predict(full_result["model"], full_result["val_samples"],
                                T=VAL_MC_T, batch_size=BATCH_SIZE, seed=SEED)

val_explain_subset = select_stratified_subset(full_result["val_samples"], VAL_EXPLAIN_N_PER_CLASS)
print(f"[THRESHOLD DERIVATION] running explanation-consistency on {len(val_explain_subset)} "
      f"validation samples ({VAL_EXPLAIN_N_PER_CLASS}/class, faithfulness skipped for speed)")
val_explain_df = run_explainability_reliability_batch(
    full_result["model"], val_explain_subset, compute_faithfulness=False)

epistemic_threshold_val = float(val_mc_df["mutual_information"].median())
explanation_consistency_threshold_val = float(val_explain_df["explanation_consistency"].median())
CONFIDENCE_HIGH_THRESH = 0.9   # reuses the same high-confidence convention already used in error_analysis_report (§16)

print(f"\n[THRESHOLDS -- validation-derived, METHODOLOGICAL]")
print(f"  epistemic_threshold_val (median val MI)                  = {epistemic_threshold_val:.4f}")
print(f"  explanation_consistency_threshold_val (median val consistency) = {explanation_consistency_threshold_val:.4f}")
print(f"  confidence_high_thresh (fixed convention, §16)            = {CONFIDENCE_HIGH_THRESH:.4f}")

# --- Exploratory-only alternative: test-set percentiles (labeled, not used for conclusions) ---
epistemic_threshold_test_pctl = float(np.percentile(multiscale_df["mutual_information"], 75))
explanation_consistency_threshold_test_pctl = float(np.percentile(combined_reliability_df["explanation_consistency"], 25))
print(f"\n[THRESHOLDS -- test-set percentile, EXPLORATORY ONLY, not used below]")
print(f"  epistemic_threshold_test_pctl (75th pct test MI)           = {epistemic_threshold_test_pctl:.4f}")
print(f"  explanation_consistency_threshold_test_pctl (25th pct test consistency) = {explanation_consistency_threshold_test_pctl:.4f}")


In [ ]:
def _classify_reliability(row) -> str:
    high_unc = row["mutual_information"] >= epistemic_threshold_val
    low_expl = row["explanation_consistency"] <= explanation_consistency_threshold_val
    high_conf = row["confidence"] >= CONFIDENCE_HIGH_THRESH
    if row["correct"] and not high_unc and not low_expl:
        return "reliable"
    if row["correct"] and high_unc:
        return "uncertain_but_correct"
    if high_conf and (high_unc or low_expl):
        return "confident_but_suspicious"
    if (not row["correct"]) and high_unc and low_expl:
        return "highly_unreliable"
    return "other"


combined_reliability_df["reliability_category"] = combined_reliability_df.apply(_classify_reliability, axis=1)
_cat_counts = combined_reliability_df["reliability_category"].value_counts()
print("[RELIABILITY CATEGORIES] (validation-derived thresholds, test subsample, n="
      f"{len(combined_reliability_df)})")
print(_cat_counts.to_string())
combined_reliability_df.to_csv(OUT_DIR / "combined_reliability.csv", index=False)   # re-save with category column

fig, ax = plt.subplots(figsize=(7, 4.5))
_cat_order = ["reliable", "uncertain_but_correct", "confident_but_suspicious", "highly_unreliable", "other"]
_cat_counts_ordered = _cat_counts.reindex(_cat_order).fillna(0)
ax.bar(_cat_counts_ordered.index, _cat_counts_ordered.values,
       color=["#4ac9c2", "#e0a94a", "#8b7fd6", "#e8664a", "lightgray"])
ax.set_ylabel("Count"); ax.tick_params(axis="x", rotation=20)
ax.set_title("Reliability Category Breakdown (test subsample)")
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(OUT_DIR / "reliability_category_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
HIGH_RISK_TOP_N = 12

_strict_mask = ((combined_reliability_df["confidence"] >= CONFIDENCE_HIGH_THRESH) &
                (combined_reliability_df["mutual_information"] >= epistemic_threshold_val) &
                (combined_reliability_df["explanation_consistency"] <= explanation_consistency_threshold_val) &
                (~combined_reliability_df["correct"]))
high_risk_df = combined_reliability_df[_strict_mask].sort_values("confidence", ascending=False).head(HIGH_RISK_TOP_N)

_relaxed_note = ""
if len(high_risk_df) == 0:
    # Strict 4-way intersection on a small stratified subsample can easily be empty.
    # Fall back to a ranked composite score over the same criteria, clearly labeled
    # as relaxed rather than silently redefining "high-risk".
    _relaxed_note = (" (no sample satisfied the strict 4-way criterion in this subsample; "
                      "showing the closest matches by a composite suspicion score instead)")
    _susp = (_robust_zscore(combined_reliability_df["mutual_information"].to_numpy())
             - _robust_zscore(combined_reliability_df["explanation_consistency"].to_numpy())
             + _robust_zscore(combined_reliability_df["confidence"].to_numpy()))
    combined_reliability_df["_suspicion_score_fallback_only"] = _susp
    high_risk_df = combined_reliability_df[~combined_reliability_df["correct"]].assign(
        _suspicion_score_fallback_only=_susp[~combined_reliability_df["correct"].to_numpy()]
    ).sort_values("_suspicion_score_fallback_only", ascending=False).head(HIGH_RISK_TOP_N)

print(f"[POTENTIALLY UNRELIABLE HIGH-CONFIDENCE PREDICTIONS]{_relaxed_note}")
print(f"  n={len(high_risk_df)} of {len(combined_reliability_df)} subsample predictions")

# FEATURE-LEVEL UNCERTAINTY REMOVED FROM FINAL METHODOLOGY
# Retained only to preserve previously generated notebook outputs.
_high_risk_cols = ["path", "true_class", "pred_class", "confidence", "predictive_entropy",
                    "mutual_information", "prediction_disagreement",  # "feature_instability" removed
                    "explanation_consistency", "gradcampp_vs_scorecam_iou",
                    "gradcampp_vs_lime_iou", "scorecam_vs_lime_iou"]
high_risk_df[_high_risk_cols].to_csv(OUT_DIR / "potentially_unreliable_high_confidence_predictions.csv", index=False)
high_risk_df[_high_risk_cols]


In [ ]:
def plot_high_risk_case_grid(model: nn.Module, cases_df: pd.DataFrame, max_cases: int = 8,
                              save_path: Optional[str] = None):
    """Input / Grad-CAM++ / Score-CAM / LIME grid for the top-risk cases, in the same
    visual layout already used by explain_sample / run_xai_batch_validation (§14)."""
    cases = cases_df.head(max_cases)
    n = len(cases)
    if n == 0:
        print("[HIGH-RISK GRID] no cases to display.")
        return
    fig, axes = plt.subplots(n, 4, figsize=(14, 3.2 * n))
    if n == 1:
        axes = axes.reshape(1, -1)
    for row_i, (_, case) in enumerate(cases.iterrows()):
        input_tensor = load_input_tensor(case["path"])
        rgb01 = denormalize_to_rgb01(input_tensor[0])

        gcpp = GradCAMPlusPlus(model, model.attn)
        cam_gcpp, pred_idx = gcpp.generate(input_tensor)
        gcpp.remove()
        scam = ScoreCAM(model, model.attn)
        cam_score, _ = scam.generate(input_tensor, class_idx=pred_idx)
        scam.remove()
        lime_exp = LimeImageExplainer(model).explain(input_tensor, class_idx=pred_idx)

        axes[row_i, 0].imshow(rgb01)
        axes[row_i, 0].set_ylabel(f"true={case['true_class']}\npred={case['pred_class']}\n"
                                   f"conf={case['confidence']:.2f}",
                                   fontsize=8, rotation=0, labelpad=48, va="center")
        axes[row_i, 1].imshow(overlay_heatmap(rgb01, cam_gcpp))
        axes[row_i, 2].imshow(overlay_heatmap(rgb01, cam_score))
        axes[row_i, 3].imshow(LimeImageExplainer.visualize(rgb01, lime_exp))
        for col in range(4):
            axes[row_i, col].set_xticks([]); axes[row_i, col].set_yticks([])
    for col, name in enumerate(["Input", "Grad-CAM++", "Score-CAM", "LIME"]):
        axes[0, col].set_title(name)
    fig.suptitle("Potentially Unreliable High-Confidence Predictions")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


plot_high_risk_case_grid(full_result["model"], high_risk_df, max_cases=8,
                          save_path=str(OUT_DIR / "high_risk_case_grid.png"))


### 10. Selective Prediction with Uncertainty

Extends the existing confidence-based selective-prediction curve (`plot_coverage_accuracy_curve`, §24) by testing whether rejecting on an uncertainty measure — instead of, or in addition to, raw confidence — yields a better coverage/risk trade-off. The measures evaluated (predictive entropy, mutual information, and input-level perturbation sensitivity) are assessed on the full test set (`multiscale_df`), plus confidence itself as the existing baseline, so they're directly comparable on identical footing. (Feature instability was evaluated here in earlier iterations of this analysis; it has been removed from the final methodology — see §2/§7.)


In [ ]:
def selective_prediction_curve(df: pd.DataFrame, score_col: str,
                                higher_score_means_more_certain: bool,
                                n_thresholds: int = 30) -> pd.DataFrame:
    """Generalizes plot_coverage_accuracy_curve (§24, which is confidence-only) to any
    per-sample score column. Sweeps the fraction of samples REJECTED (highest
    uncertainty / lowest certainty first) from 0% to 95%, and reports the coverage,
    retained accuracy, and selective risk (1 - retained accuracy) at each point."""
    scores = df[score_col].to_numpy()
    correct = df["correct"].to_numpy()
    order = np.argsort(-scores) if higher_score_means_more_certain else np.argsort(scores)
    correct_sorted = correct[order]   # most-certain-first
    n = len(df)
    rows = []
    for frac_reject in np.linspace(0.0, 0.95, n_thresholds):
        n_keep = max(1, int(round(n * (1 - frac_reject))))
        kept = correct_sorted[:n_keep]
        acc = kept.mean()
        rows.append({"score_col": score_col, "rejection_fraction": frac_reject,
                     "coverage": n_keep / n, "accuracy": acc, "selective_risk": 1 - acc})
    return pd.DataFrame(rows)


# FEATURE-LEVEL UNCERTAINTY REMOVED FROM FINAL METHODOLOGY
# Retained only to preserve previously generated notebook outputs.
_selective_measures = [
    ("confidence", True),                    # higher confidence = more certain
    ("predictive_entropy", False),           # higher entropy = more uncertain
    ("mutual_information", False),           # higher MI = more uncertain (epistemic)
    ("input_perturbation_entropy", False),   # input-level sensitivity
    # ("feature_instability", False),        # feature-level sensitivity -- removed from final methodology
]
selective_prediction_dfs = {}
for score_col, higher_is_certain in _selective_measures:
    selective_prediction_dfs[score_col] = selective_prediction_curve(
        multiscale_df, score_col, higher_is_certain)

selective_prediction_all_df = pd.concat(selective_prediction_dfs.values(), ignore_index=True)
selective_prediction_all_df.to_csv(OUT_DIR / "selective_prediction_uncertainty_comparison.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 6))
_colors = {"confidence": "#333333", "predictive_entropy": "#4ac9c2", "mutual_information": "#e8664a",
           "input_perturbation_entropy": "#8b7fd6"}  # "feature_instability" color removed
_auc_scores = {}
for score_col, _ in _selective_measures:
    d = selective_prediction_dfs[score_col]
    ax.plot(d["coverage"], d["accuracy"], marker="o", markersize=3,
            color=_colors[score_col], label=score_col)
    _auc_scores[score_col] = float(np.trapz(d["accuracy"].to_numpy()[::-1], d["coverage"].to_numpy()[::-1]))
ax.set_xlabel("Coverage (fraction retained)"); ax.set_ylabel("Accuracy (retained subset)")
ax.set_title("Selective Prediction: Accuracy vs. Coverage\n(rejecting samples ranked by each measure)")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "selective_prediction_uncertainty_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

_ranking = sorted(_auc_scores.items(), key=lambda kv: kv[1], reverse=True)
print("[SELECTIVE PREDICTION] Area under the accuracy-vs-coverage curve, per rejection measure "
      "(higher = better at identifying errors to reject):")
for name, auc_val in _ranking:
    print(f"  {name:28s}  AUC={auc_val:.4f}")
print(f"\nMost effective measure for selective prediction in this run: '{_ranking[0][0]}'")


### 11. External Validation Uncertainty

Applies the exact same uncertainty methodology (§2–§6 above, unchanged) to each of the four external Mendeley datasets already defined in `EXTERNAL_DATASETS` (§13a). Sample lists aren't stored from the original `run_external_validation` calls (§13), so they're re-scanned here with the existing `scan_dataset_generic` (§8) — the same reuse pattern `plot_reliability_diagram`'s external-calibration comparison already uses in §20. The MC-Dropout pass count is reduced (`EXTERNAL_MC_T`) and the explanation subsample is smaller (`EXTERNAL_EXPLAIN_N_PER_CLASS`) purely for runtime, given four extra datasets are being swept — both are configurable. The methodology itself is identical across all five datasets (BRISC2025 test + 4 external), so the comparison is apples-to-apples.


In [ ]:
EXTERNAL_MC_T = 15
EXTERNAL_EXPLAIN_N_PER_CLASS = 3

external_uncertainty_rows = []
external_multiscale_dfs = {}

if "EXTERNAL_DATASETS" not in globals():
    print("[EXTERNAL VALIDATION UNCERTAINTY] EXTERNAL_DATASETS not in scope (§13a wasn't run "
          "in this session) -- skipping. Re-run §13a first if external uncertainty is needed.")
else:
    for ds in EXTERNAL_DATASETS:
        name = ds["name"]
        print(f"\n{'=' * 70}\n[EXTERNAL UNCERTAINTY: {name}]\n{'=' * 70}")
        ext_samples = scan_dataset_generic(ds["root"], ds["folder_to_class"], name)
        if len(ext_samples) == 0:
            print(f"  [SKIP] no images found for '{name}'")
            continue

        ext_pred_df = collect_test_predictions(full_result["model"], ext_samples)
        ext_if_df = compute_input_feature_uncertainty(full_result["model"], ext_samples,
                                                        batch_size=BATCH_SIZE)
        ext_mc_df = mc_dropout_predict(full_result["model"], ext_samples, T=EXTERNAL_MC_T,
                                        batch_size=BATCH_SIZE, seed=SEED)
        ext_multiscale_df = (ext_pred_df.merge(ext_if_df, on="path", how="inner")
                                          .merge(ext_mc_df, on="path", how="inner"))
        ext_multiscale_df.to_csv(OUT_DIR / f"multiscale_uncertainty_{name}.csv", index=False)
        external_multiscale_dfs[name] = ext_multiscale_df

        ext_explain_subset = select_stratified_subset(ext_samples, EXTERNAL_EXPLAIN_N_PER_CLASS)
        ext_explain_df = run_explainability_reliability_batch(
            full_result["model"], ext_explain_subset, compute_faithfulness=False)

# FEATURE-LEVEL UNCERTAINTY REMOVED FROM FINAL METHODOLOGY
# Retained only to preserve previously generated notebook outputs.
        external_uncertainty_rows.append({
            "dataset": name, "n": len(ext_samples),
            "accuracy": ext_multiscale_df["correct"].mean(),
            "mean_confidence": ext_multiscale_df["confidence"].mean(),
            "mean_predictive_entropy": ext_multiscale_df["predictive_entropy"].mean(),
            "mean_mutual_information": ext_multiscale_df["mutual_information"].mean(),
            "mean_input_perturbation_entropy": ext_multiscale_df["input_perturbation_entropy"].mean(),
            # "mean_feature_instability" removed from final methodology
            "mean_explanation_consistency": ext_explain_df["explanation_consistency"].mean(),
        })
        print(f"  n={len(ext_samples)}  acc={ext_multiscale_df['correct'].mean():.4f}  "
              f"mean_MI={ext_multiscale_df['mutual_information'].mean():.4f}")

    # Internal BRISC2025 test row, for direct comparison on the same columns.
    external_uncertainty_rows.insert(0, {
        "dataset": "BRISC2025_test (internal)", "n": len(multiscale_df),
        "accuracy": multiscale_df["correct"].mean(),
        "mean_confidence": multiscale_df["confidence"].mean(),
        "mean_predictive_entropy": multiscale_df["predictive_entropy"].mean(),
        "mean_mutual_information": multiscale_df["mutual_information"].mean(),
        "mean_input_perturbation_entropy": multiscale_df["input_perturbation_entropy"].mean(),
        # "mean_feature_instability" removed from final methodology
        "mean_explanation_consistency": explain_reliability_df["explanation_consistency"].mean(),
    })

    external_uncertainty_summary_df = pd.DataFrame(external_uncertainty_rows)
    external_uncertainty_summary_df.to_csv(OUT_DIR / "external_uncertainty_summary.csv", index=False)
    print("\n[EXTERNAL VALIDATION UNCERTAINTY SUMMARY]")
    print(external_uncertainty_summary_df.to_string(index=False))


In [ ]:
if "external_uncertainty_summary_df" in globals() and len(external_uncertainty_summary_df) > 1:
# FEATURE-LEVEL UNCERTAINTY REMOVED FROM FINAL METHODOLOGY
# Retained only to preserve previously generated notebook outputs.
    _ext_metrics = ["mean_mutual_information", "mean_input_perturbation_entropy",
                    "mean_explanation_consistency"]  # "mean_feature_instability" removed
    fig, axes = plt.subplots(1, len(_ext_metrics), figsize=(4.5 * len(_ext_metrics), 4.5))
    _ds_names = external_uncertainty_summary_df["dataset"]
    _bar_colors = ["#333333"] + ["#4ac9c2", "#e8664a", "#8b7fd6", "#e0a94a"][:len(_ds_names) - 1]
    for ax, metric in zip(axes, _ext_metrics):
        ax.bar(_ds_names, external_uncertainty_summary_df[metric], color=_bar_colors)
        ax.set_title(metric.replace("mean_", "").replace("_", " ").title())
        ax.tick_params(axis="x", rotation=45)
        ax.grid(alpha=0.3, axis="y")
    fig.suptitle("Internal Test vs. External Validation — Uncertainty & Explanation Consistency")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "external_uncertainty_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

    _internal_row = external_uncertainty_summary_df.iloc[0]
    _ext_rows = external_uncertainty_summary_df.iloc[1:]
    print("[DISTRIBUTION-SHIFT CHECK] external vs. internal (reporting actual results, not forcing a conclusion):")
    for _, r in _ext_rows.iterrows():
        mi_higher = r["mean_mutual_information"] > _internal_row["mean_mutual_information"]
        # fi_higher removed -- feature_instability no longer part of final methodology
        ec_lower = r["mean_explanation_consistency"] < _internal_row["mean_explanation_consistency"]
        print(f"  {r['dataset']:24s}  higher_MI={mi_higher}  "
              f"lower_explanation_consistency={ec_lower}")
else:
    print("[EXTERNAL VALIDATION UNCERTAINTY] nothing to plot (external summary unavailable or single-row).")


### 12. Computational Overhead

The existing efficiency analysis (§19, `compute_efficiency_metrics`) measures single deterministic inference already. This subsection adds the additional cost specific to this section: MC-Dropout inference overhead, and the full uncertainty + XAI pipeline cost per sample — reported separately so it's clear what the *baseline* deployment cost is vs. what *this analysis* adds on top of it.


In [ ]:
# Reuses efficiency_metrics["latency_ms_per_image"] (§19) as the single-deterministic-
# inference baseline, rather than re-measuring it from scratch.
_single_inference_ms = efficiency_metrics["latency_ms_per_image"]

# MC-Dropout overhead: derived directly from the timed run already performed (§4).
_mc_total_s = mc_dropout_df.attrs.get("mc_dropout_time_seconds", float("nan"))
_mc_T = mc_dropout_df.attrs.get("T", MC_T)
_mc_n = len(mc_dropout_df)
_mc_ms_per_sample = (_mc_total_s / _mc_n) * 1000 if _mc_n else float("nan")
_mc_ms_per_forward = (_mc_total_s / (_mc_n * _mc_T)) * 1000 if _mc_n and _mc_T else float("nan")
_mc_overhead_factor = _mc_ms_per_sample / _single_inference_ms if _single_inference_ms else float("nan")

# FEATURE-LEVEL UNCERTAINTY REMOVED FROM FINAL METHODOLOGY
# Retained only to preserve previously generated notebook outputs.
# Input-perturbation-sensitivity overhead: derived from the timed run already performed (§2-3).
_if_total_s = input_feature_df.attrs.get("elapsed_seconds", float("nan"))
_if_n_passes = input_feature_df.attrs.get("n_forward_passes", len(MC_UNCERTAINTY_PERTURBATIONS) + 1)
_if_ms_per_sample = (_if_total_s / len(input_feature_df)) * 1000 if len(input_feature_df) else float("nan")

# XAI overhead: derived from the timed explainability-subset run already performed (§6).
_xai_total_s = explain_reliability_df.attrs.get("elapsed_seconds", float("nan"))
_xai_ms_per_sample = (_xai_total_s / len(explain_reliability_df)) * 1000 if len(explain_reliability_df) else float("nan")

_full_pipeline_ms_per_sample = _if_ms_per_sample + _mc_ms_per_sample + _xai_ms_per_sample

overhead_rows = [
    {"stage": "Single deterministic inference (baseline, §19)",
     "ms_per_sample": _single_inference_ms, "notes": "reused from compute_efficiency_metrics"},
    {"stage": f"Input-level perturbation sensitivity ({_if_n_passes} passes: clean + "
              f"{len(MC_UNCERTAINTY_PERTURBATIONS)} mild perturbations)",
     "ms_per_sample": _if_ms_per_sample, "notes": f"{_if_n_passes}x deterministic forward passes, whole-set batched"},
    {"stage": f"MC-Dropout uncertainty (T={_mc_T})",
     "ms_per_sample": _mc_ms_per_sample,
     "notes": f"{_mc_ms_per_forward:.4f} ms/forward-pass amortized; overhead factor "
              f"{_mc_overhead_factor:.1f}x vs. single deterministic inference"},
    {"stage": "XAI (Grad-CAM++ + Score-CAM + LIME + agreement, subsample only)",
     "ms_per_sample": _xai_ms_per_sample,
     "notes": f"measured on the {EXPLAIN_N_PER_CLASS}/class stratified subsample -- LIME dominates this cost"},
    {"stage": "TOTAL: uncertainty-enabled inference (input-perturbation+MC, no XAI)",
     "ms_per_sample": _if_ms_per_sample + _mc_ms_per_sample, "notes": "what a deployed uncertainty-aware model would pay per sample"},
    {"stage": "TOTAL: full uncertainty + XAI reliability analysis",
     "ms_per_sample": _full_pipeline_ms_per_sample, "notes": "research/audit-time cost, not meant for per-sample deployment"},
]
uncertainty_overhead_df = pd.DataFrame(overhead_rows)
uncertainty_overhead_df.to_csv(OUT_DIR / "uncertainty_computational_overhead.csv", index=False)

print("[COMPUTATIONAL OVERHEAD]")
print(uncertainty_overhead_df.to_string(index=False))
print(f"\nDistinguishing the three deployment-relevant tiers:")
print(f"  single deterministic inference : {_single_inference_ms:.3f} ms/sample")
print(f"  uncertainty-enabled inference   : {_if_ms_per_sample + _mc_ms_per_sample:.3f} ms/sample "
      f"({(_if_ms_per_sample + _mc_ms_per_sample) / _single_inference_ms:.1f}x baseline)")
print(f"  full uncertainty + XAI analysis : {_full_pipeline_ms_per_sample:.3f} ms/sample "
      f"({_full_pipeline_ms_per_sample / _single_inference_ms:.1f}x baseline)")


### 14. Temperature Scaling Calibration (addendum)

The calibration work already in §20 (`plot_reliability_diagram`, `compare_calibration`, ECE, Brier) and the uncertainty-split comparison in §8 both evaluate calibration as it already exists — neither one *recalibrates* the model. This addendum adds that: standard single-parameter temperature scaling (Guo et al., 2017) — a scalar `T` fit by minimizing NLL on the **validation** logits only, then applied to test (and, for comparison, external) logits as `softmax(logits / T)`. `T` is a single learned scalar, applied post-hoc at inference time; it does not touch the trained weights, so it does not retrain or modify SMGF-Net itself, and it never sees test labels during fitting.


In [ ]:
@torch.no_grad()
def collect_logits(model: nn.Module, samples: List[Dict], batch_size: int = BATCH_SIZE
                    ) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """Raw pre-softmax logits (needed for temperature scaling -- softmax(logits/T)
    is not recoverable from already-softmaxed probabilities). Same deterministic
    forward-pass pattern as collect_test_predictions (§16), just returns logits
    instead of probabilities."""
    ds = BrainTumorDataset(samples, is_train=False)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    model.eval()
    logits_chunks, paths = [], []
    for batch in loader:
        img = batch["image"].to(DEVICE, non_blocking=True)
        out = model(img)
        logits_chunks.append(out["logits"].cpu().numpy())
        paths.extend(batch["path"])
    logits = np.concatenate(logits_chunks, axis=0)
    path_to_label = {s["path"]: s["label"] for s in samples}
    labels = np.array([path_to_label[p] for p in paths])
    return logits, labels, paths


def fit_temperature(val_logits: np.ndarray, val_labels: np.ndarray,
                     max_iter: int = 100, lr: float = 0.01) -> float:
    """Fits a single scalar T minimizing NLL on validation logits via LBFGS
    (Guo et al., 2017 "On Calibration of Modern Neural Networks"). Validation only
    -- test labels are never involved in fitting T."""
    logits_t = torch.tensor(val_logits, dtype=torch.float32, device=DEVICE)
    labels_t = torch.tensor(val_labels, dtype=torch.long, device=DEVICE)
    T = torch.nn.Parameter(torch.ones(1, device=DEVICE) * 1.5)
    optimizer = torch.optim.LBFGS([T], lr=lr, max_iter=max_iter)
    nll = torch.nn.CrossEntropyLoss()

    def closure():
        optimizer.zero_grad()
        loss = nll(logits_t / T.clamp(min=1e-2), labels_t)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(T.clamp(min=1e-2).item())


def softmax_with_temperature(logits: np.ndarray, T: float = 1.0, axis: int = 1) -> np.ndarray:
    z = logits / T
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

print("Temperature-scaling utilities ready.")


In [ ]:
# Fit T on the validation split only.
val_logits, val_labels, _ = collect_logits(full_result["model"], full_result["val_samples"])
T_optimal = fit_temperature(val_logits, val_labels)
print(f"[TEMPERATURE SCALING] fit on validation set (n={len(val_labels)}): T_optimal = {T_optimal:.4f}")
print(f"  (T > 1 means the model was overconfident pre-scaling; T < 1 means underconfident; "
      f"T = 1 means the model was already well-calibrated on this split)")

# Apply to the test set.
test_logits, test_labels, test_paths = collect_logits(full_result["model"], full_result["test_samples"])
assert test_paths == test_pred_df["path"].tolist(), "sample ordering mismatch vs test_pred_df"

probs_before = softmax_with_temperature(test_logits, T=1.0)      # == y_prob_arr, recomputed for consistency
probs_after = softmax_with_temperature(test_logits, T=T_optimal)

ece_before, _, _, _ = expected_calibration_error(probs_before, test_labels, n_bins=15)
ece_after, _, _, _ = expected_calibration_error(probs_after, test_labels, n_bins=15)
brier_before = brier_score_multiclass(probs_before, test_labels)
brier_after = brier_score_multiclass(probs_after, test_labels)
acc_before = (probs_before.argmax(axis=1) == test_labels).mean()   # temperature scaling is monotonic -> should match
acc_after = (probs_after.argmax(axis=1) == test_labels).mean()

temperature_scaling_df = pd.DataFrame([
    {"stage": "Before scaling (T=1.0)", "T": 1.0, "accuracy": acc_before, "ece": ece_before, "brier": brier_before},
    {"stage": f"After scaling (T={T_optimal:.4f})", "T": T_optimal, "accuracy": acc_after, "ece": ece_after, "brier": brier_after},
])
temperature_scaling_df.to_csv(OUT_DIR / "temperature_scaling_calibration.csv", index=False)
print("\n[TEMPERATURE SCALING] before vs. after (test set):")
print(temperature_scaling_df.to_string(index=False))
print(f"\n  ECE change:   {ece_before:.4f} -> {ece_after:.4f}  "
      f"({'improved' if ece_after < ece_before else 'no improvement'}, "
      f"Δ={ece_after - ece_before:+.4f})")
print(f"  Accuracy unchanged as expected (temperature scaling is monotonic, preserves argmax): "
      f"{np.isclose(acc_before, acc_after)}")


In [ ]:
# Visualization 1: reliability diagrams before vs. after, overlaid (reuses compare_calibration, §20).
temp_calib_compare_df = compare_calibration(
    {f"Before scaling (T=1.0, ECE={ece_before:.3f})": (probs_before, test_labels),
     f"After scaling (T={T_optimal:.3f}, ECE={ece_after:.3f})": (probs_after, test_labels)},
    save_path=str(OUT_DIR / "temperature_scaling_before_after.png"))

# Visualization 2 (new diagnostic): ECE vs. T sweep on the validation set, to show *why*
# T_optimal was chosen -- not just assert it. Reveals whether the ECE-minimizing T
# happens to coincide with the NLL-minimizing T_optimal (they need not be identical).
_T_sweep = np.linspace(0.3, 3.0, 40)
_val_probs_by_T = [softmax_with_temperature(val_logits, T=t) for t in _T_sweep]
_val_ece_by_T = [expected_calibration_error(p, val_labels, n_bins=15)[0] for p in _val_probs_by_T]
_ece_minimizing_T = float(_T_sweep[np.argmin(_val_ece_by_T)])

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(_T_sweep, _val_ece_by_T, color="#4ac9c2", linewidth=2)
ax.axvline(1.0, color="gray", linestyle=":", label="T=1.0 (no scaling)")
ax.axvline(T_optimal, color="#e8664a", linestyle="--", label=f"T_optimal (NLL-fit) = {T_optimal:.3f}")
ax.axvline(_ece_minimizing_T, color="#8b7fd6", linestyle="--", label=f"ECE-minimizing T = {_ece_minimizing_T:.3f}")
ax.set_xlabel("Temperature (T)"); ax.set_ylabel("Validation ECE")
ax.set_title("Validation ECE vs. Temperature\n(diagnostic: why T_optimal was chosen)")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "temperature_ece_sweep.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"NLL-fit T_optimal={T_optimal:.4f}  vs.  ECE-minimizing T={_ece_minimizing_T:.4f} "
      f"(these are fit to different objectives and need not match exactly).")


In [ ]:
# Apply the SAME validation-fit T to external datasets (never refit per-dataset --
# that would be a different, data-snooping procedure) to see whether temperature
# scaling generalizes under distribution shift or breaks down.
if "EXTERNAL_DATASETS" in globals():
    _ext_temp_calib_rows = []
    _ext_temp_calib_results = {f"BRISC2025 test (T={T_optimal:.3f})": (probs_after, test_labels)}
    for ds in EXTERNAL_DATASETS:
        name = ds["name"]
        ext_samples = scan_dataset_generic(ds["root"], ds["folder_to_class"], name)
        if len(ext_samples) == 0:
            continue
        ext_logits, ext_labels, _ = collect_logits(full_result["model"], ext_samples)
        ext_probs_before = softmax_with_temperature(ext_logits, T=1.0)
        ext_probs_after = softmax_with_temperature(ext_logits, T=T_optimal)
        ece_b, _, _, _ = expected_calibration_error(ext_probs_before, ext_labels, n_bins=15)
        ece_a, _, _, _ = expected_calibration_error(ext_probs_after, ext_labels, n_bins=15)
        _ext_temp_calib_rows.append({
            "dataset": name, "n": len(ext_labels),
            "ece_before_scaling": ece_b, "ece_after_scaling": ece_a,
            "improved": ece_a < ece_b,
        })
        _ext_temp_calib_results[f"{name} (T={T_optimal:.3f})"] = (ext_probs_after, ext_labels)

    external_temperature_scaling_df = pd.DataFrame(_ext_temp_calib_rows)
    external_temperature_scaling_df.to_csv(OUT_DIR / "external_temperature_scaling_calibration.csv", index=False)
    print("[TEMPERATURE SCALING UNDER DISTRIBUTION SHIFT] (T fit on internal validation only, applied as-is)")
    print(external_temperature_scaling_df.to_string(index=False))

    compare_calibration(_ext_temp_calib_results,
                         save_path=str(OUT_DIR / "temperature_scaling_external_comparison.png"))
else:
    print("[TEMPERATURE SCALING] EXTERNAL_DATASETS not in scope this session -- "
          "skipping external calibration-under-shift comparison (see §11 to enable it).")


### 15. Additional Visualization — Uncertainty Distribution Shapes

Figure 2 (§8) shows correct-vs-incorrect medians/spread as boxplots, which hides *where the mass* of each distribution actually sits and whether the two groups overlap heavily or separate cleanly. This adds histogram overlays for the same measures, which is what actually justifies (or doesn't) using these measures as a rejection rule in §10's selective prediction.


In [ ]:
# FEATURE-LEVEL UNCERTAINTY REMOVED FROM FINAL METHODOLOGY
# Retained only to preserve previously generated notebook outputs.
_hist_metrics = ["predictive_entropy", "mutual_information", "input_perturbation_entropy"]  # "feature_instability" removed
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes = axes.reshape(-1)
for ax, metric in zip(axes, _hist_metrics):
    correct_vals = multiscale_df.loc[multiscale_df["correct"], metric].to_numpy()
    incorrect_vals = multiscale_df.loc[~multiscale_df["correct"], metric].to_numpy()
    bins = np.linspace(multiscale_df[metric].min(), multiscale_df[metric].max(), 30)
    ax.hist(correct_vals, bins=bins, color="#4ac9c2", alpha=0.6, density=True, label="Correct")
    ax.hist(incorrect_vals, bins=bins, color="#e8664a", alpha=0.6, density=True, label="Incorrect")
    ax.axvline(np.median(correct_vals), color="#4ac9c2", linestyle="--", linewidth=1.5)
    ax.axvline(np.median(incorrect_vals), color="#e8664a", linestyle="--", linewidth=1.5)
    ax.set_xlabel(metric.replace("_", " ").title()); ax.set_ylabel("Density")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.suptitle("Uncertainty Distribution Shapes — Correct vs. Incorrect (full test set)\n"
             "(dashed lines = group medians; overlap here is exactly what limits selective-prediction performance in §10)")
plt.tight_layout()
plt.savefig(OUT_DIR / "uncertainty_distribution_histograms.png", dpi=150, bbox_inches="tight")
plt.show()


### 16. Summary

Verifies nothing above §25 (the original notebook) was disturbed by any part of this section (Part 17 of the task brief), then answers RQ1–RQ3 using the actual numbers computed in this run.


In [ ]:
# --- Part 17 verification: compare current results to the pre-section snapshot (§ "safeguard" cell) ---
_current_test_metrics = full_result["test_metrics"]
_verify_keys = [("test_accuracy", "acc"), ("test_balanced_accuracy", "balanced_acc"),
                ("test_f1_macro", "f1_mac"), ("test_auc", "auc"), ("test_mcc", "mcc"),
                ("test_cohen_kappa", "kappa"), ("test_ece", "ece")]
print("[PART 17 VERIFICATION] existing results before vs. after this section:")
_all_unchanged = True
for snap_key, metric_key in _verify_keys:
    before = BASELINE_RESULTS_SNAPSHOT[snap_key]
    after = _current_test_metrics[metric_key]
    unchanged = np.isclose(before, after, atol=1e-9)
    _all_unchanged &= unchanged
    print(f"  {snap_key:28s}  before={before:.6f}  after={after:.6f}  "
          f"{'OK (unchanged)' if unchanged else '*** CHANGED ***'}")
_cm_unchanged = np.array_equal(BASELINE_RESULTS_SNAPSHOT["test_confusion_matrix"], _current_test_metrics["cm"])
_all_unchanged &= _cm_unchanged
print(f"  {'confusion_matrix':28s}  {'OK (unchanged)' if _cm_unchanged else '*** CHANGED ***'}")
print(f"\n  Model was not retrained or modified by this section: {_all_unchanged}")

print(f"\n{'=' * 78}\nRESEARCH QUESTIONS\n{'=' * 78}")

# FEATURE-LEVEL UNCERTAINTY REMOVED FROM FINAL METHODOLOGY
# Retained only to preserve previously generated notebook outputs.
print("\n[RQ1] Can SMGF-Net quantify uncertainty at input and prediction levels?")
print(f"  Yes. On the {len(multiscale_df)}-sample BRISC2025 test set:")
print(f"    Input-level    -- mean prediction_disagreement={multiscale_df['prediction_disagreement'].mean():.4f}, "
      f"mean input_perturbation_entropy={multiscale_df['input_perturbation_entropy'].mean():.4f}")
# Feature-level line removed from final methodology (previously: mean feature_instability /
# mean cosine similarity to clean embedding).
print(f"    Prediction-level (MC-Dropout, T={MC_T}, {mc_dropout_df.attrs.get('n_dropout_layers', '?')} "
      f"dropout layer(s)) -- mean predictive_entropy={multiscale_df['predictive_entropy'].mean():.4f}, "
      f"mean mutual_information={multiscale_df['mutual_information'].mean():.4f}, "
      f"mean predictive_variance={multiscale_df['predictive_variance'].mean():.4f}")

print("\n[RQ2] Does uncertainty correlate with prediction errors and distribution shift?")
_mi_row = uncertainty_stats_df[uncertainty_stats_df["metric"] == "mutual_information"]
if len(_mi_row):
    r = _mi_row.iloc[0]
    print(f"  Errors: mutual_information is {'significantly' if r['significant_bh_0.05'] else 'not significantly'} "
          f"different between correct (median={r['median_correct']:.4f}) and incorrect "
          f"(median={r['median_incorrect']:.4f}) predictions (Mann-Whitney U, BH-corrected "
          f"p={r['p_value_bh_corrected']:.2e}, rank-biserial effect size={r['rank_biserial_effect_size']:.3f}).")
_best_selective, _best_auc = _ranking[0]
print(f"  Selective prediction: rejecting by '{_best_selective}' gave the best accuracy-vs-coverage "
      f"trade-off (AUC={_best_auc:.4f}) among the measures tested.")
if "external_uncertainty_summary_df" in globals() and len(external_uncertainty_summary_df) > 1:
    _n_shifted = sum(external_uncertainty_summary_df.iloc[1:]["mean_mutual_information"] >
                      external_uncertainty_summary_df.iloc[0]["mean_mutual_information"])
    print(f"  Distribution shift: {_n_shifted}/{len(external_uncertainty_summary_df) - 1} external datasets "
          f"showed higher mean mutual_information than the internal BRISC2025 test set "
          f"(see external_uncertainty_summary.csv for the full, non-cherry-picked breakdown).")
else:
    print("  Distribution shift: external validation uncertainty was not run in this session "
          "(§13a EXTERNAL_DATASETS not in scope) -- see §11 above to enable it.")

print("\n[RQ3] Can explanation consistency improve identification of unreliable high-confidence predictions?")
_ec_row = explanation_stats_df[explanation_stats_df["metric"] == "explanation_consistency"]
if len(_ec_row):
    r = _ec_row.iloc[0]
    print(f"  explanation_consistency is {'significantly' if r['significant_bh_0.05'] else 'not significantly'} "
          f"different between correct (median={r['median_correct']:.4f}) and incorrect "
          f"(median={r['median_incorrect']:.4f}) predictions in the test subsample "
          f"(p_bh={r['p_value_bh_corrected']:.2e}, effect size={r['rank_biserial_effect_size']:.3f}).")
print(f"  {len(high_risk_df)} case(s) flagged as Potentially Unreliable High-Confidence Predictions "
      f"out of {len(combined_reliability_df)} explained test samples "
      f"({len(high_risk_df) / len(combined_reliability_df) * 100:.1f}%).")
print(f"  Reliability category breakdown: {dict(_cat_counts)}")

print(f"\n{'=' * 78}\nOUTPUTS WRITTEN TO {OUT_DIR}\n{'=' * 78}")
_new_outputs = [
    "multiscale_uncertainty.csv", "input_feature_uncertainty_test.csv", "mc_dropout_uncertainty_test.csv",
    "explainability_reliability_test_subset.csv", "combined_reliability.csv",
    "uncertainty_correct_vs_incorrect_stats.csv", "explanation_correct_vs_incorrect_stats.csv",
    "uncertainty_explanation_correlation.csv", "calibration_by_uncertainty_group.csv",
    "potentially_unreliable_high_confidence_predictions.csv",
    "selective_prediction_uncertainty_comparison.csv", "external_uncertainty_summary.csv",
    "uncertainty_computational_overhead.csv",
    "fig1_multiscale_uncertainty_overview.png", "fig2_correct_vs_incorrect_uncertainty.png",
    "fig3_uncertainty_by_class.png", "fig4_uncertainty_explanation_reliability_map.png",
    "fig5_confidence_vs_uncertainty.png", "uncertainty_explanation_correlation_heatmap.png",
    "calibration_by_uncertainty_group.png", "reliability_category_breakdown.png",
    "high_risk_case_grid.png", "selective_prediction_uncertainty_comparison.png",
    "external_uncertainty_comparison.png",
    "temperature_scaling_calibration.csv", "temperature_scaling_before_after.png",
    "temperature_ece_sweep.png", "external_temperature_scaling_calibration.csv",
    "temperature_scaling_external_comparison.png", "uncertainty_distribution_histograms.png",
]
for f in _new_outputs:
    exists = (OUT_DIR / f).exists()
    print(f"  [{'x' if exists else ' '}] {f}")


## Noise-Induced Uncertainty Analysis

A dedicated, multi-level extension of the single-severity Gaussian-noise probe already present in this notebook. Nothing above is retrained, recomputed or overwritten: this section evaluates the *same* trained checkpoint (`full_result["model"]`) on the *same* BRISC2025 test split under progressively stronger controlled input noise, and asks whether the uncertainty measures already defined in §2–§5 respond to that perturbation.

**This is a robustness / uncertainty experiment. It is not differential privacy and provides no privacy guarantee of any kind.**

**Relation to what already exists.** §17 (`ROBUSTNESS_SEVERITIES` / `apply_perturbation`) sweeps `gaussian_noise` at σ ∈ {0.02, 0.05, 0.08, 0.12} but reports only accuracy/F1/AUC. §2 reuses the single mildest level (σ = 0.02) as one of five perturbation kinds folded into one aggregate sensitivity score. Neither one traces the *uncertainty* measures across noise strength, which is what this section adds. The σ levels used here (0.01 / 0.03 / 0.05) sit inside the range already validated as mild-to-moderate in §17, and are expressed in the same units: the noise is added to the robust-normalized `[0, 1]` grayscale image, exactly as `apply_perturbation(..., "gaussian_noise", σ)` does, then clipped back to `[0, 1]` before the ImageNet normalization in `to_3ch_tensor`.

**Why a local noise-injection helper instead of `preprocess_perturbed`.** `preprocess_perturbed` (§17) additionally applies `amef_preprocess`, while the model's actual inference path, `preprocess` (§3), does not. Routing this experiment through `preprocess_perturbed` would therefore make the σ = 0 row differ from the notebook's normal evaluation condition, and every reported change would confound noise with an AMEF pipeline switch. `preprocess_gaussian_noise` below is `preprocess` plus the noise step and nothing else, so **σ = 0 reproduces the standard clean evaluation exactly** (verified numerically at the end of this section).

**Reproducibility.** Noise for each image is drawn from a `np.random.RandomState` seeded deterministically from `(path, σ, SEED)`. This keeps the draws identical across reruns and across DataLoader worker counts, and — unlike the global-RNG draw inside `apply_perturbation` — leaves the notebook's existing seed strategy untouched.

**Metrics.** Accuracy and mean confidence come from one deterministic pass per level (same convention as `collect_test_predictions`, §16). Predictive entropy, mutual information and predictive variance come from MC-Dropout at each level, using the same estimator, the same `enable_mc_dropout`, the same `_entropy_np` and the same `T = MC_T` as §4. Disagreement is reported as the fraction of test images whose predicted class changes relative to the σ = 0 prediction — the set-level analogue of the per-sample `prediction_disagreement` in §2; it is 0 at σ = 0 by construction, not by measurement.

Runtime: 4 levels × (1 deterministic + `MC_T` stochastic) passes over the full 995-image test set.

In [ ]:
# --- Noise-Induced Uncertainty Analysis: helpers -----------------------------
# Two small helpers only. Everything else (enable_mc_dropout, _entropy_np,
# _PathTaggedDataset, _run_inference_pass, read_gray/robust_normalize/to_3ch_tensor)
# is reused unchanged from the sections above.

NOISE_LEVELS = [("Clean", 0.00), ("Mild", 0.01), ("Moderate", 0.03), ("Strong", 0.05)]
NOISE_MC_T = MC_T   # same number of stochastic passes as the main MC-Dropout run (§4)


def _noise_seed(path: str, sigma: float) -> int:
    """Deterministic per-(image, sigma) seed derived from the notebook's global SEED."""
    return int(hashlib.md5(f"{path}|{sigma}|{SEED}".encode()).hexdigest()[:8], 16)


def preprocess_gaussian_noise(path: str, sigma: float) -> torch.Tensor:
    """preprocess() (§3) with one extra step: additive Gaussian noise of std `sigma`
    on the robust-normalized [0,1] grayscale image, clipped back to [0,1] -- the same
    operation apply_perturbation(..., "gaussian_noise", sigma) (§17) performs, but
    drawn from a per-(path, sigma) RandomState so it is reproducible under
    multi-worker loading and does not disturb the global RNG state.

    sigma == 0 returns exactly preprocess(path); the clean row of this experiment is
    therefore the notebook's normal evaluation condition. (preprocess_perturbed is
    deliberately NOT used here -- it also applies AMEF, which preprocess does not.)"""
    gray = read_gray(path)
    gray = cv2.resize(gray, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    img01 = robust_normalize(gray)
    if sigma > 0:
        rs = np.random.RandomState(_noise_seed(path, sigma))
        noise = rs.normal(0.0, sigma, img01.shape).astype(np.float32)
        img01 = np.clip(img01 + noise, 0.0, 1.0).astype(np.float32)
    return to_3ch_tensor(img01)


@torch.no_grad()
def mc_dropout_probs(model: nn.Module, samples: List[Dict], preprocess_fn,
                     T: int = NOISE_MC_T, seed: int = SEED,
                     batch_size: int = BATCH_SIZE) -> Tuple[np.ndarray, List[str]]:
    """Identical MC-Dropout sampling to mc_dropout_predict (§4) -- same
    enable_mc_dropout (dropout train, BatchNorm and everything else eval), same T,
    same fixed seed -- but reads images through an arbitrary `preprocess_fn`, so the
    same estimator can be applied at each noise level. Returns the raw [T, N, C]
    probability stack plus the path order; the entropy/MI/variance formulas are
    applied by the caller using the existing _entropy_np."""
    ds = _PathTaggedDataset(samples, preprocess_fn)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
    model.eval()                     # BatchNorm + everything -> eval
    enable_mc_dropout(model)         # dropout only -> train
    torch.manual_seed(seed)          # same mask sequence at every noise level
    all_probs, paths_master = [], None
    for t in range(T):
        probs_t, paths_t = [], []
        for batch in loader:
            img = batch["image"].to(DEVICE, non_blocking=True)
            out = model(img)
            probs_t.append(torch.softmax(out["logits"], dim=1).cpu().numpy())
            paths_t.extend(batch["path"])
        all_probs.append(np.concatenate(probs_t, axis=0))
        if paths_master is None:
            paths_master = paths_t
    model.eval()                     # restore full eval mode (dropout off)
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return np.stack(all_probs, axis=0), paths_master


print("Noise-induced uncertainty helpers ready "
      f"(levels: {[(n, s) for n, s in NOISE_LEVELS]}, T={NOISE_MC_T}).")


In [ ]:
# --- Noise-Induced Uncertainty Analysis: run the sweep -----------------------
_noise_samples = full_result["test_samples"]
_noise_path_to_label = {s["path"]: s["label"] for s in _noise_samples}

noise_rows, noise_per_sample_frames = [], []
_clean_pred = None

for _level_name, _sigma in NOISE_LEVELS:
    _pf = (lambda p, s=_sigma: preprocess_gaussian_noise(p, s))
    print(f"\n[NOISE SWEEP] {_level_name}  sigma={_sigma}")
    _t0 = time.perf_counter()

    # (1) deterministic pass -> accuracy + confidence (same convention as §16)
    _probs_det, _, _paths = _run_inference_pass(full_result["model"], _noise_samples, _pf,
                                                batch_size=BATCH_SIZE)
    _y_true = np.array([_noise_path_to_label[p] for p in _paths])
    _pred = _probs_det.argmax(axis=1)
    _conf = _probs_det.max(axis=1)
    _acc = float((_pred == _y_true).mean())

    if _clean_pred is None:                       # sigma=0 is the reference prediction
        _clean_pred = _pred.copy()
    _disagreement = float((_pred != _clean_pred).mean())

    # (2) MC-Dropout pass -> predictive entropy / mutual information / variance
    _probs_arr, _paths_mc = mc_dropout_probs(full_result["model"], _noise_samples, _pf,
                                             T=NOISE_MC_T, seed=SEED, batch_size=BATCH_SIZE)
    assert _paths_mc == _paths, "sample ordering drifted between deterministic and MC passes"
    _mean_probs = _probs_arr.mean(axis=0)
    _pred_entropy = _entropy_np(_mean_probs, axis=1)
    _exp_entropy = _entropy_np(_probs_arr, axis=2).mean(axis=0)
    _mutual_info = _pred_entropy - _exp_entropy
    _pred_var = _probs_arr.var(axis=0).mean(axis=1)

    noise_rows.append({
        "noise_level": _level_name, "sigma": _sigma,
        "accuracy": _acc,
        "mean_confidence": float(_conf.mean()),
        "predictive_entropy": float(_pred_entropy.mean()),
        "mutual_information": float(_mutual_info.mean()),
        "probability_variance": float(_pred_var.mean()),
        "disagreement_vs_clean": _disagreement,
    })
    noise_per_sample_frames.append(pd.DataFrame({
        "path": _paths, "noise_level": _level_name, "sigma": _sigma,
        "true_idx": _y_true, "pred_idx": _pred, "correct": _pred == _y_true,
        "confidence": _conf, "predictive_entropy": _pred_entropy,
        "expected_entropy": _exp_entropy, "mutual_information": _mutual_info,
        "predictive_variance": _pred_var,
        "changed_vs_clean": _pred != _clean_pred,
    }))
    print(f"  done in {time.perf_counter() - _t0:.1f}s  "
          f"acc={_acc:.4f}  mean_conf={_conf.mean():.4f}  "
          f"PE={_pred_entropy.mean():.4f}  MI={_mutual_info.mean():.4f}")

noise_uncertainty_df = pd.DataFrame(noise_rows)
noise_uncertainty_per_sample_df = pd.concat(noise_per_sample_frames, ignore_index=True)
noise_uncertainty_df.to_csv(OUT_DIR / "noise_induced_uncertainty.csv", index=False)
noise_uncertainty_per_sample_df.to_csv(OUT_DIR / "noise_induced_uncertainty_per_sample.csv", index=False)

print("\n[NOISE-INDUCED UNCERTAINTY ANALYSIS]  "
      f"n={len(_noise_samples)} test images, MC-Dropout T={NOISE_MC_T}, seed={SEED}")
print(noise_uncertainty_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nDisagreement is measured against the sigma=0 prediction, so the clean row is "
      "0.0000 by construction rather than by measurement.")


In [ ]:
# --- Noise-Induced Uncertainty Analysis: plots + validation ------------------
# Three panels, all values read directly from noise_uncertainty_df (nothing entered by hand).
_x = noise_uncertainty_df["sigma"].to_numpy()
_panels = [("predictive_entropy", "Predictive Entropy", "#4ac9c2"),
           ("mutual_information", "Mutual Information", "#e8664a"),
           ("accuracy", "Accuracy", "#8b7fd6")]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (col, label, color) in zip(axes, _panels):
    ax.plot(_x, noise_uncertainty_df[col].to_numpy(), marker="o", color=color, linewidth=2)
    for _xi, _yi, _name in zip(_x, noise_uncertainty_df[col], noise_uncertainty_df["noise_level"]):
        ax.annotate(_name, (_xi, _yi), textcoords="offset points", xytext=(0, 7),
                    ha="center", fontsize=7, color="gray")
    ax.set_xlabel("Gaussian noise sigma (on the [0,1] normalized image)")
    ax.set_ylabel(label)
    ax.set_title(f"{label} vs. Noise Level")
    ax.grid(alpha=0.3)
axes[2].set_ylim(0, 1.02)
fig.suptitle("Noise-Induced Uncertainty Analysis — trained SMGF-Net + MS-CA-ECA, BRISC2025 test set")
plt.tight_layout()
plt.savefig(OUT_DIR / "noise_induced_uncertainty.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Validation: does sigma=0 reproduce the existing clean evaluation? -------
_clean_row = noise_uncertainty_df.iloc[0]
_ref_acc = float(test_pred_df["correct"].mean())
_ref_conf = float(test_pred_df["confidence"].mean())
_ref_pe = float(mc_dropout_df["predictive_entropy"].mean())
_ref_mi = float(mc_dropout_df["mutual_information"].mean())
print("[VALIDATION] sigma=0 vs. the notebook's existing clean evaluation "
      "(§16 predictions and §4 MC-Dropout, both unchanged):")
for _label, _new, _ref in [("accuracy", _clean_row["accuracy"], _ref_acc),
                           ("mean confidence", _clean_row["mean_confidence"], _ref_conf),
                           ("predictive entropy", _clean_row["predictive_entropy"], _ref_pe),
                           ("mutual information", _clean_row["mutual_information"], _ref_mi)]:
    _match = np.isclose(_new, _ref, atol=1e-4)
    print(f"  {_label:20s} noise-section={_new:.6f}  existing={_ref:.6f}  "
          f"{'MATCH' if _match else '*** MISMATCH ***'}")

# --- Observed trends: reported as measured, monotonic or not ------------------
print("\n[OBSERVED TRENDS] across sigma = "
      f"{list(noise_uncertainty_df['sigma'])} (direction is measured, not assumed):")
for _col in ["accuracy", "mean_confidence", "predictive_entropy", "mutual_information",
             "probability_variance", "disagreement_vs_clean"]:
    _v = noise_uncertainty_df[_col].to_numpy()
    _d = np.diff(_v)
    if np.all(_d > 0):
        _trend = "monotonically increasing"
    elif np.all(_d < 0):
        _trend = "monotonically decreasing"
    elif np.all(_d >= 0):
        _trend = "non-decreasing (with a flat step)"
    elif np.all(_d <= 0):
        _trend = "non-increasing (with a flat step)"
    else:
        _trend = "NOT monotonic"
    print(f"  {_col:24s} {_v[0]:.4f} -> {_v[-1]:.4f}  (delta={_v[-1] - _v[0]:+.4f})  {_trend}")

print("\nInterpretation is limited to what these numbers show: how this trained model's "
      "uncertainty measures and accuracy move as controlled Gaussian input noise increases. "
      "An increase in entropy or mutual information under stronger perturbation indicates "
      "greater predictive instability, not that the uncertainty estimates are calibrated; "
      "no robustness or privacy claim follows from this experiment.")


## §23 · Final Results Summary

In [ ]:
print("=" * 70)
print("FINAL MODEL: SMGF-Net + MS-CA-ECA")
print("=" * 70)
_m = full_result["test_metrics"]
print(f"  Parameters:     {full_result['total_params']:,}  "
      f"(budget {PARAM_BUDGET:,}, diff vs. previous model {full_result['total_params'] - REFERENCE_SMGFNET_PARAMS:+,})")
print(f"  Test Accuracy:  {_m['acc']:.4f}")
print(f"  Macro-F1:       {_m['f1_mac']:.4f}")
print(f"  Weighted-F1:    {_m['f1_weighted']:.4f}")
print(f"  AUC:            {_m['auc']:.4f}")
print(f"  MCC:            {_m['mcc']:.4f}")

print("\n" + "=" * 70)
print("EXTERNAL VALIDATION  (dataset-wise, vs. internal BRISC2025 test)")
print("=" * 70)
print(f"  Internal (BRISC2025 held-out test): Accuracy={_m['acc']:.4f}  AUC={_m['auc']:.4f}")
print(ext_summary_df.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))

print("\n" + "=" * 70)
print("XAI")
print("=" * 70)
for method, s in xai_batch_results["summary"].items():
    print(f"  {method:10s}  deletion_AUC={s['deletion_auc_mean']:.4f}\u00b1{s['deletion_auc_std']:.4f}  "
          f"insertion_AUC={s['insertion_auc_mean']:.4f}\u00b1{s['insertion_auc_std']:.4f}")
print("\n  Cross-method agreement (mean over sampled test images):")
for pair in ["gradcampp_vs_scorecam", "gradcampp_vs_lime", "scorecam_vs_lime"]:
    print(f"    {pair:24s}  IoU={agreement_df[f'{pair}_iou'].mean():.3f}  "
          f"Spearman \u03c1={agreement_df[f'{pair}_rho'].mean():.3f}")
_final_iou = sanity["cascade"][-1]["iou_top_k"]
_final_rho = sanity["cascade"][-1]["spearman_rho"]
if _final_iou < 0.5:
    _sanity_verdict = "explanation is weight-dependent, as expected"
else:
    _sanity_verdict = "agreement stayed high -- Grad-CAM++ may be acting like an edge detector on this input rather than explaining the trained model"
print(f"\n  Randomization sanity check (Grad-CAM++ vs. cascading weight randomization, "
      f"final stage): IoU={_final_iou:.3f}  Spearman \u03c1={_final_rho:.3f}  ({_sanity_verdict})")

